In [ ]:
import sys, subprocess, importlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

def _ensure(module_name, pip_name=None):
    try:
        importlib.import_module(module_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module_name], check=True)

_ensure("torch_geometric")
_ensure("optuna")
print("Setup complete. Colab:", IN_COLAB)

In [ ]:
DATA_DIR    = "/content/drive/MyDrive/Colab Notebooks/732 sampleGAT-GRU"
OUTPUT_ROOT = "/content/drive/MyDrive/Colab Notebooks/732 sampleGAT-GRU"
RUN_NAME = None
OVERWRITE_POLICY = "version"

SILOS = ["Silo1", "Silo2", "Silo3"]
RAW_FILES = {"Silo1": "GAT_IP_Silo1.csv", "Silo2": "GAT_IP_Silo2.csv", "Silo3": "GAT_IP_Silo3.csv"}
OLD_LABEL_FILES = {"Silo1": "Label_Silo1.csv", "Silo2": "Label_Silo2.csv", "Silo3": "Label_Silo3.csv"}
EDGE_FILE_CANDIDATES = ["EDGE_INDEX.csv", "edge_index.csv", "EDGE_INDEX.npy", "edge_index.npy",
                        "EDGE_INDEX.pt", "edge_index.pt", "adjacency.csv", "adjacency_matrix.csv",
                        "adjacency.npy", "ADJ.csv"]
ADJACENCY_FILE = None
ADJ_NODE_ORDER = "auto"

ADJ_NODE_ORDER_FALLBACK = "layer_major"

DATE_COL = None
AMB_T_COL = None
AMB_RH_COL = None
DATE_FORMAT = None
SENSOR_NAME_REGEX = None
SENSOR_ORDER_FALLBACK = None
MISSING_SENTINELS = []
DUPLICATE_POLICY = "error"

IMPUTED_POSITIONS = "auto"
RESOLUTION_STEP_C = 0.5
OFFGRID_THRESHOLD = 0.5
EXCLUDE_IMPUTED = True

N_CABLES, N_LAYERS = 19, 14
N_SENSORS, N_NODES = 266, 268
IDX_AMB_T, IDX_AMB_RH = 266, 267
N_CLASSES = 4
STAGE_THRESHOLDS = (37.0, 42.0, 45.0)
STAGE_NAMES = {0: "Stage-0 (Safe)", 1: "Stage-1 (Initial)", 2: "Stage-2 (Active)", 3: "Stage-3 (Severe)"}
MISSING_LABEL = -1

GRAIN_T_PLAUSIBLE = (-10.0, 80.0)
AMB_T_PLAUSIBLE = (-15.0, 55.0)
RH_PLAUSIBLE = (0.0, 100.0)
JUMP_THRESHOLD_C = 10.0
ROBUST_Z_THRESHOLD = 6.0

INPUT_DAYS, HORIZONS = 15, 3
MAX_SAMPLES_PER_SILO = 244
TRAIN_FRAC = 0.80
GAP_WINDOWS = 11
SAMPLES_PER_SILO = MAX_SAMPLES_PER_SILO
N_TRAIN_PER_SILO = int(round(TRAIN_FRAC * SAMPLES_PER_SILO))
N_TEST_PER_SILO = SAMPLES_PER_SILO - N_TRAIN_PER_SILO
GAP_POLICY = f"{GAP_WINDOWS}_window_gap"
SELECTION_STRATEGY = "max_class_coverage"
MAX_INPUT_MISSING_FRAC = 0.20
VAL_FRAC = 0.20

ARCHS = ["GAT_GRU"]
DATASETS = ["Silo1", "Silo2", "Silo3", "Combined"]
NODE_EMB_DIM = 8
NUM_SPATIAL_LAYERS = 2
LOSS_WEIGHTING = "none"
GRAD_CLIP = 1.0
MAX_EPOCHS, PATIENCE = 1000, 25
STRICT_EPOCHS = True

FINAL_FIT_MODE = "refit_full"

EVAL_BATCH_SIZE = 16
SEEDS = {"Trial-1": 42, "Trial-2": 123, "Trial-3": 2026}

SEARCH_SPACE = {
    "lr": [1e-4, 5e-4, 1e-3],
    "batch_size": [8, 16, 32],
    "hidden": [32, 64, 128],
    "heads": [2, 4, 8],
    "gru_layers": [1, 2],
    "dropout": [0.1, 0.2, 0.3],
    "weight_decay": [0.0, 1e-4, 1e-3],
}
TUNE_DATASET = "Combined"
N_TUNING_TRIALS = 20
TUNE_MAX_EPOCHS, TUNE_PATIENCE = 60, 10
TUNING_SEED = 42

SMOKE_EPOCHS = 10
SMOKE_HP = dict(lr=1e-3, batch_size=16, hidden=32, heads=2, gru_layers=1, dropout=0.1, weight_decay=0.0)

assert N_TRAIN_PER_SILO + N_TEST_PER_SILO == SAMPLES_PER_SILO
assert N_CABLES * N_LAYERS == N_SENSORS and N_SENSORS + 2 == N_NODES
assert GAP_WINDOWS >= 0
print("Configuration loaded. Windows excluded between the last training window and the first testing window:", GAP_WINDOWS)

In [ ]:
import os, re, json, time, math, random, shutil, platform, datetime, warnings, copy
from collections import OrderedDict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)

class DataContractError(RuntimeError):
    pass

assert os.path.isdir(DATA_DIR), f"DATA_DIR not found: {DATA_DIR}. Mount Google Drive first (Cell 1)."
OUTPUT_ROOT = os.path.abspath(OUTPUT_ROOT)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
if RUN_NAME is None:
    RUN_NAME = "run_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(OUTPUT_ROOT, RUN_NAME)
RESUMING = os.path.isdir(RUN_DIR)
_DIR_NAMES = ["01_preprocessing", "02_graph", "03_samples", "04_tensors", "05_models", "06_tuning",
              "07_smoke_test", "08_trials", "09_metrics", "10_tables", "11_figures", "12_logs", "13_environment"]
DIRS = {n.split("_", 1)[1]: os.path.join(RUN_DIR, n) for n in _DIR_NAMES}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
LOG_FILE = os.path.join(DIRS["logs"], "run_log.txt")

def _inside_output(path):
    if not os.path.abspath(path).startswith(OUTPUT_ROOT + os.sep):
        raise PermissionError(f"Refusing to write outside OUTPUT_ROOT: {path}")

def log(msg, level="INFO"):
    line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] [{level}] {msg}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def versioned_path(path):
    _inside_output(path)
    if not os.path.exists(path):
        return path
    if OVERWRITE_POLICY == "error":
        raise FileExistsError(f"{path} exists and OVERWRITE_POLICY='error'.")
    base, ext = os.path.splitext(path)
    k = 2
    while os.path.exists(f"{base}__v{k}{ext}"):
        k += 1
    return f"{base}__v{k}{ext}"

class NpEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.bool_,)): return bool(o)
        if isinstance(o, np.ndarray): return o.tolist()
        if isinstance(o, (pd.Timestamp, datetime.date, datetime.datetime)): return str(o)
        return str(o)

def save_df(df, key, name, index=False):
    p = versioned_path(os.path.join(DIRS[key], name))
    df.to_csv(p, index=index)
    log(f"saved {os.path.relpath(p, RUN_DIR)}")
    return p

def save_json(obj, key, name):
    p = versioned_path(os.path.join(DIRS[key], name))
    with open(p, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, cls=NpEncoder)
    log(f"saved {os.path.relpath(p, RUN_DIR)}")
    return p

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        p = versioned_path(os.path.join(DIRS["figures"], f"{name}.{ext}"))
        fig.savefig(p, dpi=300, bbox_inches="tight")
    log(f"saved figure {name}.png/.pdf")

def show_table(df, title):
    print("\n" + "=" * 100 + f"\n{title}\n" + "=" * 100)
    display(df)

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log(f"RUN_DIR = {RUN_DIR}  (resuming={RESUMING})  device={DEVICE}")
if DEVICE.type == "cuda":
    log(f"GPU: {torch.cuda.get_device_name(0)}")
save_json({k: v for k, v in globals().items()
           if k.isupper() and isinstance(v, (int, float, str, list, dict, tuple, type(None)))},
          "logs", "config_snapshot.json")

In [ ]:
_NON_SENSOR = re.compile(r"^(unnamed.*|s\.?\s*no\.?|sl\.?\s*no\.?|sr\.?\s*no\.?|index|id|serial.*)$", re.I)

def read_csv_clean(path):
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    return df

def detect_columns(df, silo):
    cols = list(df.columns)
    date_col = DATE_COL
    if date_col is None:
        hits = [c for c in cols if re.search(r"date|time|day", c, re.I)]
        if len(hits) != 1:
            raise DataContractError(f"[{silo}] date column not unique: {hits}. Set DATE_COL. Columns: {cols[:8]}...")
        date_col = hits[0]
    rh = AMB_RH_COL
    if rh is None:
        hits = [c for c in cols if c != date_col and re.search(r"humid|rh", c, re.I)]
        if len(hits) != 1:
            raise DataContractError(f"[{silo}] ambient RH column not unique: {hits}. Set AMB_RH_COL.")
        rh = hits[0]
    amb = AMB_T_COL
    if amb is None:
        hits = [c for c in cols if c not in (date_col, rh)
                and re.search(r"amb|outside|outdoor|atmos|environment|external", c, re.I)]
        if len(hits) != 1:
            raise DataContractError(f"[{silo}] ambient temperature column not unique: {hits}. Set AMB_T_COL.")
        amb = hits[0]
    for c in (date_col, amb, rh):
        if c not in cols:
            raise DataContractError(f"[{silo}] configured column '{c}' not found.")
    ignored = [c for c in cols if c not in (date_col, amb, rh) and _NON_SENSOR.match(c)]
    sensors = [c for c in cols if c not in (date_col, amb, rh) and c not in ignored]
    if len(sensors) != N_SENSORS:
        raise DataContractError(
            f"[{silo}] expected {N_SENSORS} internal sensor columns, found {len(sensors)}. "
            f"First extras/candidates: {sensors[:5]} ... {sensors[-5:]}. Ignored: {ignored}.")
    return dict(date_col=date_col, amb_t=amb, amb_rh=rh, sensor_cols=sensors, ignored_cols=ignored)

def parse_sensor_grid(sensor_cols):
    parsed = []
    for c in sensor_cols:
        if SENSOR_NAME_REGEX:
            m = re.search(SENSOR_NAME_REGEX, c)
            if not m:
                parsed = None; break
            parsed.append((int(m.group("cable")), int(m.group("layer"))))
        else:
            ints = re.findall(r"\d+", c)
            if len(ints) < 2:
                parsed = None; break
            parsed.append((int(ints[-2]), int(ints[-1])))
    if parsed is not None:
        cabs = sorted({p[0] for p in parsed}); lays = sorted({p[1] for p in parsed})
        if len(set(parsed)) == N_SENSORS and len(cabs) == N_CABLES and len(lays) == N_LAYERS:
            ci = np.array([cabs.index(p[0]) for p in parsed]); li = np.array([lays.index(p[1]) for p in parsed])
            return ci, li, "parsed_from_column_names"
    idx = np.arange(N_SENSORS)
    if SENSOR_ORDER_FALLBACK == "csv_order_cable_major":
        return idx // N_LAYERS, idx % N_LAYERS, "ASSUMPTION_csv_order_cable_major"
    if SENSOR_ORDER_FALLBACK == "csv_order_layer_major":
        return idx % N_CABLES, idx // N_CABLES, "ASSUMPTION_csv_order_layer_major"
    return None, None, "unresolved"

RAW, META, inspect_rows = {}, {}, []
for silo in SILOS:
    path = os.path.join(DATA_DIR, RAW_FILES[silo])
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing raw file: {path}")
    df = read_csv_clean(path)
    meta = detect_columns(df, silo)
    ci, li, src = parse_sensor_grid(meta["sensor_cols"])
    meta.update(cable_idx=ci, layer_idx=li, grid_source=src)
    is_cable_major = (ci is not None) and bool(np.all(np.lexsort((li, ci)) == np.arange(N_SENSORS)))
    RAW[silo], META[silo] = df, meta
    inspect_rows.append(dict(
        silo=silo, file=RAW_FILES[silo], rows=len(df), columns=df.shape[1], date_col=meta["date_col"],
        ambient_T_col=meta["amb_t"], ambient_RH_col=meta["amb_rh"], n_sensor_cols=len(meta["sensor_cols"]),
        first_sensor=meta["sensor_cols"][0], last_sensor=meta["sensor_cols"][-1],
        ignored_cols=";".join(meta["ignored_cols"]), cable_layer_source=src,
        csv_order_is_cable_major=is_cable_major))
    log(f"[{silo}] {len(df)} rows; sensors={len(meta['sensor_cols'])}; grid={src}")

inspection_df = pd.DataFrame(inspect_rows)
show_table(inspection_df, "Dataset inspection")
save_df(inspection_df, "preprocessing", "dataset_inspection.csv")

ref = META[SILOS[0]]
for silo in SILOS[1:]:
    m = META[silo]
    same_names = m["sensor_cols"] == ref["sensor_cols"]
    same_grid = (m["cable_idx"] is not None and ref["cable_idx"] is not None
                 and np.array_equal(m["cable_idx"], ref["cable_idx"]) and np.array_equal(m["layer_idx"], ref["layer_idx"]))
    if not (same_names or same_grid):
        raise DataContractError(f"{silo} sensor columns/order differ from {SILOS[0]}; node mapping would be inconsistent.")
    log(f"[{silo}] structural consistency with {SILOS[0]}: identical names={same_names}, identical grid={same_grid}")
print("\nCell 4 OK — all three silos share one 266-sensor layout.")

In [ ]:
def parse_dates(raw, silo):
    if DATE_FORMAT:
        d = pd.to_datetime(raw, format=DATE_FORMAT, errors="coerce"); how = f"format={DATE_FORMAT}"
    else:
        opts = {}
        for dayfirst in (True, False):
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                dd = pd.to_datetime(raw, dayfirst=dayfirst, errors="coerce")
            dn = dd.dropna().dt.normalize().drop_duplicates().sort_values()
            frac1 = float((dn.diff().dt.days == 1).mean()) if len(dn) > 1 else 0.0
            opts[dayfirst] = (int(dd.isna().sum()), -frac1, dd)
        best = min(opts, key=lambda k: opts[k][:2])
        d = opts[best][2]; how = f"auto(dayfirst={best})"
        if opts[True][:2] == opts[False][:2] and not opts[True][2].equals(opts[False][2]):
            log(f"[{silo}] day/month order is ambiguous; using dayfirst={best}. Set DATE_FORMAT to be explicit.", "WARN")
    n_bad = int(d.isna().sum())
    if n_bad:
        ex = raw[d.isna()].astype(str).head(5).tolist()
        raise DataContractError(f"[{silo}] {n_bad} unparseable dates, e.g. {ex}. Set DATE_FORMAT.")
    return d.dt.normalize(), how

def clean_silo(df, meta, silo):
    rep = OrderedDict(silo=silo)
    dates, rep["date_parsing"] = parse_dates(df[meta["date_col"]], silo)
    rep["input_chronologically_sorted"] = bool(dates.is_monotonic_increasing)
    num_cols = meta["sensor_cols"] + [meta["amb_t"], meta["amb_rh"]]
    cols_out = {"Date": dates.values}
    nonnum, sentinel = 0, 0
    for c in num_cols:
        orig = df[c]
        conv = pd.to_numeric(orig, errors="coerce")
        bad = orig.notna() & conv.isna() & (orig.astype(str).str.strip() != "")
        nonnum += int(bad.sum())
        if MISSING_SENTINELS:
            s = conv.isin(MISSING_SENTINELS); sentinel += int(s.sum()); conv = conv.mask(s)
        cols_out[c] = conv.astype(float).values
    out = pd.DataFrame(cols_out)
    rep["non_numeric_values_set_missing"] = nonnum
    rep["sentinel_values_set_missing"] = sentinel
    out = out.sort_values("Date", kind="stable").reset_index(drop=True)
    dup = out["Date"].duplicated(keep=False)
    rep["rows_with_duplicate_dates"] = int(dup.sum())
    if dup.any():
        conflicting = [d for d, g in out[dup].groupby("Date") if g[num_cols].drop_duplicates().shape[0] > 1]
        rep["conflicting_duplicate_dates"] = len(conflicting)
        if conflicting and DUPLICATE_POLICY == "error":
            raise DataContractError(f"[{silo}] {len(conflicting)} dates have different readings "
                                    f"(e.g. {conflicting[:3]}). Set DUPLICATE_POLICY to 'first' or 'mean'.")
        if conflicting and DUPLICATE_POLICY == "mean":
            out = out.groupby("Date", as_index=False)[num_cols].mean()
        else:
            out = out.drop_duplicates(subset="Date", keep="first").reset_index(drop=True)
        log(f"[{silo}] duplicate dates resolved (policy={DUPLICATE_POLICY}); rows now {len(out)}", "WARN")
    else:
        rep["conflicting_duplicate_dates"] = 0
    diffs = out["Date"].diff().dt.days
    gaps = out.loc[diffs > 1, "Date"]
    rep["n_days"] = len(out)
    rep["first_date"] = str(out["Date"].iloc[0].date()); rep["last_date"] = str(out["Date"].iloc[-1].date())
    rep["calendar_span_days"] = int((out["Date"].iloc[-1] - out["Date"].iloc[0]).days) + 1
    rep["n_date_gaps"] = int(len(gaps)); rep["missing_calendar_days"] = int((diffs[diffs > 1] - 1).sum())
    gap_df = pd.DataFrame({"gap_ends_on": gaps.dt.date.astype(str), "missing_days": (diffs[diffs > 1] - 1).astype(int)})
    T = out[meta["sensor_cols"]].to_numpy(float)
    rep["sensor_values_total"] = T.size
    rep["sensor_values_missing"] = int(np.isnan(T).sum())
    rep["ambient_T_missing"] = int(out[meta["amb_t"]].isna().sum())
    rep["ambient_RH_missing"] = int(out[meta["amb_rh"]].isna().sum())
    miss_col = out[num_cols].isna().sum().rename("missing").reset_index().rename(columns={"index": "column"})

    with np.errstate(invalid="ignore"):
        rep["grain_T_below_plausible"] = int((T < GRAIN_T_PLAUSIBLE[0]).sum())
        rep["grain_T_above_plausible"] = int((T > GRAIN_T_PLAUSIBLE[1]).sum())
        consec = (diffs.values[1:] == 1)
        jumps = np.abs(np.diff(T, axis=0))[consec]
        rep["day_to_day_jumps_ge_threshold"] = int((jumps >= JUMP_THRESHOLD_C).sum())
        med = np.nanmedian(T, axis=0); mad = np.nanmedian(np.abs(T - med), axis=0) * 1.4826
        rz = np.abs(T - med) / np.where(mad > 0, mad, np.nan)
        rep["robust_z_ge_threshold"] = int(np.nansum(rz >= ROBUST_Z_THRESHOLD))
        a = out[meta["amb_t"]].to_numpy(float); h = out[meta["amb_rh"]].to_numpy(float)
        rep["ambient_T_outside_plausible"] = int(((a < AMB_T_PLAUSIBLE[0]) | (a > AMB_T_PLAUSIBLE[1])).sum())
        rep["ambient_RH_outside_0_100"] = int(((h < RH_PLAUSIBLE[0]) | (h > RH_PLAUSIBLE[1])).sum())
    rep["grain_T_min"] = float(np.nanmin(T)); rep["grain_T_max"] = float(np.nanmax(T))
    rep["note"] = "Outliers are reported only; extreme valid temperatures are retained (possible Stage-3)."
    flagged = np.argwhere((rz >= ROBUST_Z_THRESHOLD) | (T < GRAIN_T_PLAUSIBLE[0]) | (T > GRAIN_T_PLAUSIBLE[1]))
    flag_df = pd.DataFrame({"date": out["Date"].dt.date.astype(str).values[flagged[:, 0]] if len(flagged) else [],
                            "sensor": np.array(meta["sensor_cols"])[flagged[:, 1]] if len(flagged) else [],
                            "temperature": T[flagged[:, 0], flagged[:, 1]] if len(flagged) else []})
    return out, rep, gap_df, miss_col, flag_df

CLEAN, prep_rows = {}, []
for silo in SILOS:
    out, rep, gap_df, miss_col, flag_df = clean_silo(RAW[silo], META[silo], silo)
    CLEAN[silo] = out
    prep_rows.append(rep)
    save_df(out, "preprocessing", f"Clean_Temperature_{silo}.csv")
    save_df(gap_df, "preprocessing", f"date_gaps_{silo}.csv")
    save_df(miss_col, "preprocessing", f"missing_by_column_{silo}.csv")
    save_df(flag_df, "preprocessing", f"outlier_inspection_{silo}.csv")
    assert out["Date"].is_monotonic_increasing and not out["Date"].duplicated().any()
prep_df = pd.DataFrame(prep_rows)
show_table(prep_df.T, "Preprocessing report (per silo)")
save_df(prep_df, "preprocessing", "preprocessing_report.csv")

In [ ]:
def label_temperature(T):
    T = np.asarray(T, dtype=float)
    lab = np.full(T.shape, MISSING_LABEL, dtype=np.int64)
    v = np.isfinite(T)
    t1, t2, t3 = STAGE_THRESHOLDS
    lab[v & (T < t1)] = 0
    lab[v & (T >= t1) & (T < t2)] = 1
    lab[v & (T >= t2) & (T < t3)] = 2
    lab[v & (T >= t3)] = 3
    return lab

def label_reference(T):
    T = np.asarray(T, dtype=float)
    ref = np.digitize(T, STAGE_THRESHOLDS, right=False).astype(np.int64)
    ref[~np.isfinite(T)] = MISSING_LABEL
    return ref

_bt = np.array([36.99, 37.0, 41.99, 42.0, 44.99, 45.0, 60.0, -5.0, np.nan])
_be = np.array([0, 1, 1, 2, 2, 3, 3, 0, -1])
assert np.array_equal(label_temperature(_bt), _be), label_temperature(_bt)
assert np.array_equal(label_reference(_bt), _be)
log("Boundary test passed: 37->Stage-1, 42->Stage-2, 45->Stage-3, NaN->missing(-1)")

LABELS, t1_rows, audit_rows = {}, [], []
for silo in SILOS:
    m = META[silo]
    T = CLEAN[silo][m["sensor_cols"]].to_numpy(float)
    L = label_temperature(T)
    R = label_reference(T)
    valid = np.isfinite(T)
    mism = int((L[valid] != R[valid]).sum())
    miss_ok = bool(np.all(L[~valid] == MISSING_LABEL))
    assert mism == 0 and miss_ok, f"[{silo}] label verification failed: mismatches={mism}, missing_ok={miss_ok}"
    assert L.shape == (len(CLEAN[silo]), N_SENSORS)
    lab_df = pd.DataFrame(L, columns=m["sensor_cols"])
    lab_df.insert(0, "Date", CLEAN[silo]["Date"].dt.date.astype(str).values)
    LABELS[silo] = lab_df
    save_df(lab_df, "preprocessing", f"Fresh_Label_{silo}.csv")
    counts = {s: int((L == s).sum()) for s in range(N_CLASSES)}
    n_valid = int(valid.sum())
    row = OrderedDict(Silo=silo, Days=len(lab_df), Sensors=N_SENSORS, Total_observations=L.size,
                      Valid_observations=n_valid, Missing_observations=int((~valid).sum()))
    for s in range(N_CLASSES):
        row[f"{STAGE_NAMES[s]}"] = counts[s]
        row[f"Stage-{s} %"] = round(100 * counts[s] / max(n_valid, 1), 4)
    row.update({f"T == {t}": int((T == t).sum()) for t in STAGE_THRESHOLDS})
    row["Threshold-label mismatches"] = mism
    t1_rows.append(row)

TABLE1 = pd.DataFrame(t1_rows)
tot = TABLE1.select_dtypes("number").sum(); tot["Silo"] = "All"
TABLE1 = pd.concat([TABLE1, tot.to_frame().T], ignore_index=True)
for s in range(N_CLASSES):
    TABLE1.loc[TABLE1.Silo == "All", f"Stage-{s} %"] = round(
        100 * TABLE1.loc[TABLE1.Silo == "All", STAGE_NAMES[s]].iloc[0] / TABLE1.loc[TABLE1.Silo == "All", "Valid_observations"].iloc[0], 4)
show_table(TABLE1.T, "TABLE 1 — Fresh threshold-based label verification (data verification only)")
save_df(TABLE1, "tables", "Table01_label_verification.csv")
save_json({"label_rule": "Stage-0 T<37; Stage-1 37<=T<42; Stage-2 42<=T<45; Stage-3 T>=45",
           "missing_label_value": MISSING_LABEL, "boundary_test": "passed",
           "verification": "two independent implementations agree on every valid observation"},
          "preprocessing", "label_verification_report.json")

def normalize_old_labels(s):
    num = pd.to_numeric(s, errors="coerce")
    if num.notna().mean() > 0.5:
        return num
    st = s.astype(str).str.lower()
    out = pd.to_numeric(st.str.extract(r"(\d)")[0], errors="coerce")
    for w, v in {"safe": 0, "initial": 1, "active": 2, "severe": 3}.items():
        out[st.str.contains(w) & out.isna()] = v
    return out

for silo in SILOS:
    p = os.path.join(DATA_DIR, OLD_LABEL_FILES[silo])
    rec = OrderedDict(silo=silo, file=OLD_LABEL_FILES[silo])
    try:
        if not os.path.exists(p):
            raise DataContractError("file not found")
        old = read_csv_clean(p)
        dc = [c for c in old.columns if re.search(r"date|time|day", c, re.I)]
        if not dc:
            raise DataContractError("no date column")
        old_dates, _ = parse_dates(old[dc[0]], silo + "-old")
        sc = META[silo]["sensor_cols"]
        mapping = {}
        for c in sc:
            for cand in (c, f"{c}_label", f"{c}_Label", f"Label_{c}", f"{c}_stage", f"{c}_Stage"):
                if cand in old.columns:
                    mapping[c] = cand; break
        if not mapping:
            raise DataContractError("no sensor label columns matched the raw sensor names")
        o = pd.DataFrame({c: normalize_old_labels(old[mc]).values for c, mc in mapping.items()})
        o["Date"] = old_dates.dt.date.astype(str).values
        o = o.drop_duplicates("Date")
        fresh = LABELS[silo][["Date"] + list(mapping)]
        mg = fresh.merge(o, on="Date", suffixes=("_fresh", "_old"))
        fr = np.concatenate([mg[f"{c}_fresh"].to_numpy() for c in mapping])
        od = np.concatenate([mg[f"{c}_old"].to_numpy() for c in mapping])
        ok = (fr >= 0) & np.isfinite(od)
        ct = pd.crosstab(pd.Series(od[ok].astype(int), name="old_label"), pd.Series(fr[ok], name="fresh_label"))
        save_df(ct, "preprocessing", f"old_label_audit_crosstab_{silo}.csv", index=True)
        rec.update(matched_sensor_columns=len(mapping), matched_dates=len(mg), compared=int(ok.sum()),
                   agree=int((od[ok] == fr[ok]).sum()), disagree=int((od[ok] != fr[ok]).sum()),
                   disagreement_pct=round(100 * float((od[ok] != fr[ok]).mean()), 4) if ok.any() else np.nan,
                   status="audited")
    except Exception as e:
        rec.update(status=f"audit skipped: {e}")
    audit_rows.append(rec)
audit_df = pd.DataFrame(audit_rows)
show_table(audit_df, "Old-label discrepancy audit (old labels are NOT used anywhere in modelling)")
save_df(audit_df, "preprocessing", "old_label_discrepancy_audit.csv")

def offgrid_fraction(T, step=RESOLUTION_STEP_C):
    fin = np.isfinite(T)
    with np.errstate(invalid="ignore"):
        r = np.abs(T / step - np.round(T / step))
    return ((r > 1e-6) & fin).sum(0) / np.maximum(fin.sum(0), 1)

IMPUTED_MASK, OFFGRID, imp_rows, imp_list = {}, {}, [], []
for silo in SILOS:
    sc = META[silo]["sensor_cols"]
    frac = offgrid_fraction(CLEAN[silo][sc].to_numpy(float)); OFFGRID[silo] = frac
    if IMPUTED_POSITIONS == "auto":
        m = frac > OFFGRID_THRESHOLD
    elif IMPUTED_POSITIONS == "none":
        m = np.zeros(N_SENSORS, bool)
    else:
        unknown = set(IMPUTED_POSITIONS) - set(sc)
        assert not unknown, f"IMPUTED_POSITIONS contains unknown columns: {unknown}"
        m = np.isin(sc, list(IMPUTED_POSITIONS))
    IMPUTED_MASK[silo] = m
    phys, imp = frac[~m], frac[m]
    imp_rows.append(dict(silo=silo, imputed_positions=int(m.sum()), evaluated_positions=int((~m).sum()),
                         physical_offgrid_max=float(phys.max()) if len(phys) else np.nan,
                         imputed_offgrid_min=float(imp.min()) if len(imp) else np.nan,
                         cleanly_separated=bool(len(imp) == 0 or len(phys) == 0 or phys.max() < imp.min())))
    ci, li = META[silo]["cable_idx"], META[silo]["layer_idx"]
    imp_list.append(pd.DataFrame({"silo": silo, "column": sc, "offgrid_fraction": frac, "imputed": m,
                                  "cable": (ci + 1) if ci is not None else np.nan,
                                  "layer": (li + 1) if li is not None else np.nan}))
imp_df = pd.DataFrame(imp_rows)
show_table(imp_df, "Imputed-position identification (0.5 degC resolution signature)")
save_df(imp_df, "preprocessing", "imputed_positions_summary.csv")
save_df(pd.concat(imp_list, ignore_index=True), "preprocessing", "imputed_positions_by_sensor.csv")
if len({tuple(np.where(v)[0]) for v in IMPUTED_MASK.values()}) > 1:
    log("Imputed positions differ between silos; a per-silo mask is used.", "WARN")
log(f"EXCLUDE_IMPUTED={EXCLUDE_IMPUTED}: imputed positions are graph input nodes"
    + (" but are masked (-1) from loss and evaluation." if EXCLUDE_IMPUTED else " and ARE evaluated."))

In [ ]:
def find_supplied_graph():
    if ADJACENCY_FILE:
        p = os.path.join(DATA_DIR, ADJACENCY_FILE)
        if not os.path.exists(p):
            raise FileNotFoundError(f"ADJACENCY_FILE not found: {p}")
        return p
    for name in EDGE_FILE_CANDIDATES:
        p = os.path.join(DATA_DIR, name)
        if os.path.exists(p):
            return p
    known = set(RAW_FILES.values()) | set(OLD_LABEL_FILES.values())
    hits = [f for f in sorted(os.listdir(DATA_DIR)) if f not in known and re.search(r"adj|edge", f, re.I)
            and f.lower().endswith((".csv", ".npy", ".pt", ".txt"))]
    if len(hits) > 1:
        raise DataContractError(f"Several possible graph files in DATA_DIR: {hits}. Set ADJACENCY_FILE in Cell 2.")
    return os.path.join(DATA_DIR, hits[0]) if hits else None

def ambient_edges():
    src, dst = [], []
    for a in (IDX_AMB_T, IDX_AMB_RH):
        for i in range(N_SENSORS):
            src += [a, i]; dst += [i, a]
    src += [IDX_AMB_T, IDX_AMB_RH]; dst += [IDX_AMB_RH, IDX_AMB_T]
    return np.array([src, dst], dtype=np.int64)

def file_index_to_node(order, meta):
    if order == "csv":
        return np.arange(N_SENSORS)
    ci, li = meta["cable_idx"], meta["layer_idx"]
    if ci is None:
        raise DataContractError("cable/layer could not be parsed from column names; cannot map a layer-/cable-major file.")
    pos = {(int(c), int(l)): i for i, (c, l) in enumerate(zip(ci, li))}
    k = np.arange(N_SENSORS)
    if order == "layer_major":
        return np.array([pos[(kk % N_CABLES, kk // N_CABLES)] for kk in k])
    if order == "cable_major":
        return np.array([pos[(kk // N_LAYERS, kk % N_LAYERS)] for kk in k])
    raise ValueError(f"unknown node order {order}")

def edge_geometry(ei, meta):
    ci, li = meta["cable_idx"], meta["layer_idx"]
    if ci is None or ei.size == 0:
        return {}
    dl = np.abs(li[ei[0]] - li[ei[1]]); dc = np.abs(ci[ei[0]] - ci[ei[1]])
    return dict(mean_abs_dlayer=round(float(dl.mean()), 3), share_dlayer_le1=round(float((dl <= 1).mean()), 3),
                mean_abs_dcable=round(float(dc.mean()), 3), share_dcable_le1=round(float((dc <= 1).mean()), 3))

def load_supplied_graph(p, meta):
    names = None
    if p.endswith(".npy"):
        arr = np.load(p)
    elif p.endswith(".pt"):
        arr = torch.load(p, map_location="cpu")
        arr = arr.numpy() if torch.is_tensor(arr) else np.asarray(arr)
    else:
        raw = pd.read_csv(p, header=None, dtype=str, sep=None, engine="python")
        sc = set(meta["sensor_cols"])
        r0 = [str(x).strip() for x in raw.iloc[0, :]]; c0 = [str(x).strip() for x in raw.iloc[:, 0]]
        if sum(x in sc for x in r0) == N_SENSORS:
            names = [x for x in r0 if x in sc]
        elif sum(x in sc for x in c0) == N_SENSORS:
            names = [x for x in c0 if x in sc]
        arr = raw.apply(pd.to_numeric, errors="coerce").dropna(how="all").dropna(axis=1, how="all").to_numpy()
    arr = np.asarray(arr)
    n_file = None
    if arr.ndim == 2 and arr.shape[0] == arr.shape[1] and arr.shape[0] in (N_SENSORS, N_NODES):
        ei = np.vstack(np.nonzero(arr)).astype(np.int64); n_file = arr.shape[0]; kind = f"adjacency_matrix_{n_file}x{n_file}"
    elif arr.ndim == 2 and arr.shape[0] == 2 and arr.shape[1] != 2:
        ei = arr.astype(np.int64); kind = "edge_index_2xE"
    elif arr.ndim == 2 and arr.shape[1] == 2:
        ei = arr.T.astype(np.int64); kind = "edge_list_Ex2"
    else:
        raise DataContractError(f"Unrecognised graph file shape {arr.shape} in {p}")
    if n_file is None:
        if ei.min() == 1 and ei.max() in (N_SENSORS, N_NODES):
            ei = ei - 1; kind += " (1-based, shifted)"
        n_file = N_NODES if ei.max() >= N_SENSORS else N_SENSORS
    info = OrderedDict(file=os.path.basename(p), kind=kind, file_nodes=n_file)
    if n_file == N_SENSORS:
        if names is not None and ADJ_NODE_ORDER == "auto":
            idx = {c: i for i, c in enumerate(meta["sensor_cols"])}
            f2n = np.array([idx[n] for n in names]); order = "sensor names inside the file"
        else:
            order = ADJ_NODE_ORDER_FALLBACK if ADJ_NODE_ORDER == "auto" else ADJ_NODE_ORDER
            f2n = file_index_to_node(order, meta)
            if ADJ_NODE_ORDER == "auto":
                log(f"Adjacency file has no sensor names; node order assumed '{order}' (ADJ_NODE_ORDER_FALLBACK). "
                    "Check the edge-geometry table below.", "WARN")
        geo = {}
        for o in ("layer_major", "cable_major", "csv"):
            try:
                geo[o] = edge_geometry(file_index_to_node(o, meta)[ei], meta)
            except DataContractError:
                pass
        ei_s = f2n[ei]
        geo["USED: " + order] = edge_geometry(ei_s, meta)
        info["node_order"] = order
        info["edge_geometry_by_order"] = geo
        ei = np.hstack([ei_s, ambient_edges()])
        info["ambient_edges"] = "added (modelling assumption): ambient T and RH nodes <-> all 266 sensors, and T <-> RH"
    else:
        info["node_order"] = "file indices used directly (0..265 sensors in CSV order, 266 ambient T, 267 ambient RH)"
    return ei, info

def build_documented_graph(meta):
    ci, li = meta["cable_idx"], meta["layer_idx"]
    if ci is None:
        raise DataContractError(
            "No EDGE_INDEX/adjacency file was found in DATA_DIR and the sensor column names do not encode "
            "cable/layer. Missing information: physical sensor-to-cable mapping. Options: (1) upload the "
            "original EDGE_INDEX/adjacency file to DATA_DIR; (2) set SENSOR_NAME_REGEX; (3) explicitly set "
            "SENSOR_ORDER_FALLBACK if the CSV column order is known to be cable-major or layer-major.")
    src, dst, etype = [], [], []
    for i in range(N_SENSORS):
        for j in range(i + 1, N_SENSORS):
            if ci[i] == ci[j] and abs(int(li[i]) - int(li[j])) == 1:
                src += [i, j]; dst += [j, i]; etype += ["within_cable_adjacent_layer"] * 2
    for a in (IDX_AMB_T, IDX_AMB_RH):
        for i in range(N_SENSORS):
            src += [a, i]; dst += [i, a]; etype += ["ambient_to_sensor_assumption"] * 2
    src += [IDX_AMB_T, IDX_AMB_RH]; dst += [IDX_AMB_RH, IDX_AMB_T]; etype += ["ambient_T_RH_assumption"] * 2
    return np.array([src, dst], dtype=np.int64), np.array(etype)

def verify_graph(ei):
    rep = OrderedDict()
    assert ei.ndim == 2 and ei.shape[0] == 2, ei.shape
    assert ei.min() >= 0 and ei.max() < N_NODES, (ei.min(), ei.max())
    self_loops = ei[0] == ei[1]
    rep["self_loops_removed"] = int(self_loops.sum())
    ei = ei[:, ~self_loops]
    uniq = np.unique(ei, axis=1)
    rep["duplicate_edges_removed"] = int(ei.shape[1] - uniq.shape[1]); ei = uniq
    A = np.zeros((N_NODES, N_NODES), dtype=np.int8); A[ei[0], ei[1]] = 1
    rep["symmetric"] = bool((A == A.T).all())
    deg = A.sum(1) + A.sum(0)
    rep["isolated_nodes"] = int((deg == 0).sum())
    parent = list(range(N_NODES))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for s, d in ei.T:
        rs, rd = find(int(s)), find(int(d))
        if rs != rd: parent[rs] = rd
    rep["connected_components"] = len({find(i) for i in range(N_NODES)})
    rep["num_nodes"] = N_NODES; rep["num_directed_edges"] = int(ei.shape[1])
    rep["in_degree_min"] = int(A.sum(0).min()); rep["in_degree_max"] = int(A.sum(0).max())
    rep["in_degree_mean"] = float(A.sum(0).mean())
    return ei, A, rep

graph_file = find_supplied_graph()
if graph_file:
    EDGE_INDEX_NP, GRAPH_INFO = load_supplied_graph(graph_file, META[SILOS[0]])
    edge_types = np.array(["supplied"] * EDGE_INDEX_NP.shape[1])
    graph_source = f"SUPPLIED: {GRAPH_INFO['file']} ({GRAPH_INFO['kind']}; node order: {GRAPH_INFO['node_order']})"
    if "edge_geometry_by_order" in GRAPH_INFO:
        show_table(pd.DataFrame(GRAPH_INFO["edge_geometry_by_order"]).T,
                   "Edge geometry under each possible node order (the correct order gives short |dlayer|/|dcable|)")
else:
    GRAPH_INFO = {}
    EDGE_INDEX_NP, edge_types = build_documented_graph(META[SILOS[0]])
    graph_source = ("CONSTRUCTED (no EDGE_INDEX/adjacency file found in DATA_DIR). Physical: same-cable adjacent-layer "
                    f"edges from column names ({META[SILOS[0]]['grid_source']}). Modelling assumptions: ambient "
                    "T/RH nodes linked to all 266 sensors and to each other. No inter-cable edges, because "
                    "cable positions were not supplied.")
log("Graph source: " + graph_source, "WARN" if not graph_file else "INFO")
EDGE_INDEX_NP, ADJ, graph_rep = verify_graph(EDGE_INDEX_NP)
graph_rep["source"] = graph_source
assert ADJ.shape == (N_NODES, N_NODES)
if graph_rep["isolated_nodes"]:
    log(f"{graph_rep['isolated_nodes']} isolated nodes: they receive no messages (only self-loops in GAT).", "WARN")

m0 = META[SILOS[0]]
node_map = pd.DataFrame({
    "node_id": np.arange(N_NODES),
    "node_type": ["grain_sensor"] * N_SENSORS + ["ambient_temperature", "ambient_RH"],
    "column": m0["sensor_cols"] + [m0["amb_t"], m0["amb_rh"]],
    "cable_index": list(m0["cable_idx"]) + [np.nan, np.nan] if m0["cable_idx"] is not None else [np.nan] * N_NODES,
    "layer_index": list(m0["layer_idx"]) + [np.nan, np.nan] if m0["layer_idx"] is not None else [np.nan] * N_NODES,
    "is_classification_target": [True] * N_SENSORS + [False, False]})
assert node_map["column"].is_unique, "Duplicated node columns."
dup_pairs = []
for silo in SILOS:
    T = CLEAN[silo][META[silo]["sensor_cols"]]
    h = T.T.apply(lambda r: hash(tuple(np.round(r.fillna(-9999).values, 6))), axis=1)
    for _, grp in h.groupby(h):
        if len(grp) > 1:
            dup_pairs.append({"silo": silo, "identical_columns": ";".join(grp.index)})
graph_rep["identical_sensor_series_groups"] = len(dup_pairs)
save_df(node_map, "graph", "node_mapping.csv")
save_df(pd.DataFrame(EDGE_INDEX_NP.T, columns=["source", "target"]), "graph", "edge_index.csv")
np.save(versioned_path(os.path.join(DIRS["graph"], "adjacency_268x268.npy")), ADJ)
save_df(pd.DataFrame(dup_pairs) if dup_pairs else pd.DataFrame(columns=["silo", "identical_columns"]),
        "graph", "identical_sensor_series_check.csv")
save_json(graph_rep, "graph", "graph_verification.json")
show_table(pd.Series(graph_rep).to_frame("value"), "Graph topology verification")

In [ ]:
WIN_LEN = INPUT_DAYS + HORIZONS
SILO_ARR = {}
for silo in SILOS:
    m, c = META[silo], CLEAN[silo]
    dates = pd.DatetimeIndex(c["Date"])
    Xraw = c[m["sensor_cols"] + [m["amb_t"], m["amb_rh"]]].to_numpy(float)
    Y_all = LABELS[silo][m["sensor_cols"]].to_numpy(np.int64)
    eval_mask = ~IMPUTED_MASK[silo] if EXCLUDE_IMPUTED else np.ones(N_SENSORS, bool)
    Y = Y_all.copy(); Y[:, ~eval_mask] = MISSING_LABEL
    Xff = pd.DataFrame(Xraw).ffill().to_numpy()
    SILO_ARR[silo] = dict(dates=dates, day=np.asarray((dates - dates[0]).days), Xraw=Xraw, Xff=Xff,
                          Y=Y, Y_all=Y_all, eval_mask=eval_mask)

def enumerate_windows(silo):
    A = SILO_ARR[silo]; D = len(A["dates"]); rows = []
    for s in range(D - WIN_LEN + 1):
        consecutive = A["day"][s + WIN_LEN - 1] - A["day"][s] == WIN_LEN - 1
        in_miss = float(np.isnan(A["Xraw"][s:s + INPUT_DAYS]).mean())
        tgt = A["Y"][s + INPUT_DAYS:s + WIN_LEN]
        tgt_ok = bool(all((tgt[h] >= 0).any() for h in range(HORIZONS)))
        reason = [] if consecutive else ["non-consecutive dates"]
        if in_miss > MAX_INPUT_MISSING_FRAC: reason.append("input missing fraction too high")
        if not tgt_ok: reason.append("a target day has no valid label")
        counts = np.bincount(tgt[tgt >= 0].ravel(), minlength=N_CLASSES)
        rows.append(dict(silo=silo, start_row=s, start_day=int(A["day"][s]),
                         input_start=str(A["dates"][s].date()), input_end=str(A["dates"][s + INPUT_DAYS - 1].date()),
                         target_day16=str(A["dates"][s + INPUT_DAYS].date()),
                         target_day18=str(A["dates"][s + WIN_LEN - 1].date()),
                         input_missing_frac=in_miss, target_masked=int((tgt < 0).sum()),
                         c0=counts[0], c1=counts[1], c2=counts[2], c3=counts[3],
                         valid=bool(not reason), reason="; ".join(reason)))
    return pd.DataFrame(rows)

def feasible_blocks(V, n_tr, n_te, gap_w):
    return [(a, a + n_tr + gap_w) for a in range(len(V) - n_tr - gap_w - n_te + 1)]

def max_feasible_total(V, gap_w):
    for tot in range(MAX_SAMPLES_PER_SILO, 0, -1):
        n_tr = int(round(TRAIN_FRAC * tot)); n_te = tot - n_tr
        if n_tr > 0 and n_te > 0 and feasible_blocks(V, n_tr, n_te, gap_w):
            return tot, n_tr, n_te
    return 0, 0, 0

WINDOWS, VALID, SELECTION, feas_rows, problems = {}, {}, {}, [], []
for silo in SILOS:
    W = enumerate_windows(silo); WINDOWS[silo] = W
    save_df(W, "samples", f"all_candidate_windows_{silo}.csv")
    V = W[W.valid].reset_index(drop=True); VALID[silo] = V
    tot, ntr, nte = max_feasible_total(V, GAP_WINDOWS)
    feas_rows.append(dict(silo=silo, days=len(SILO_ARR[silo]["dates"]), candidate_windows=len(W),
                          valid_windows=len(V), requested_samples=MAX_SAMPLES_PER_SILO,
                          gap_windows=GAP_WINDOWS,
                          valid_windows_needed_for_requested=int(round(TRAIN_FRAC * MAX_SAMPLES_PER_SILO)) + GAP_WINDOWS
                          + MAX_SAMPLES_PER_SILO - int(round(TRAIN_FRAC * MAX_SAMPLES_PER_SILO)),
                          max_feasible_samples=tot, max_feasible_train=ntr, max_feasible_test=nte))
    if tot == 0:
        problems.append(f"{silo}: {len(V)} valid windows; no {int(TRAIN_FRAC * 100)}/{100 - int(TRAIN_FRAC * 100)} "
                        f"split fits with a {GAP_WINDOWS}-window gap.")

feas_df = pd.DataFrame(feas_rows)
show_table(feas_df, "Window feasibility check")
save_df(feas_df, "samples", "window_feasibility.csv")
if problems:
    raise DataContractError("SAMPLE SELECTION STOPPED.\n  " + "\n  ".join(problems))

SAMPLES_PER_SILO = int(feas_df.max_feasible_samples.min())
N_TRAIN_PER_SILO = int(round(TRAIN_FRAC * SAMPLES_PER_SILO))
N_TEST_PER_SILO = SAMPLES_PER_SILO - N_TRAIN_PER_SILO
N_TRAIN_TOTAL, N_TEST_TOTAL = 3 * N_TRAIN_PER_SILO, 3 * N_TEST_PER_SILO
N_TOTAL_SAMPLES = N_TRAIN_TOTAL + N_TEST_TOTAL
if SAMPLES_PER_SILO < MAX_SAMPLES_PER_SILO:
    log(f"With a {GAP_WINDOWS}-window gap, {MAX_SAMPLES_PER_SILO} samples per silo do not fit; using "
        f"{SAMPLES_PER_SILO} per silo ({N_TRAIN_PER_SILO} train / {N_TEST_PER_SILO} test), "
        f"{N_TOTAL_SAMPLES} in total ({N_TRAIN_TOTAL}/{N_TEST_TOTAL}).", "WARN")
else:
    log(f"Samples per silo: {SAMPLES_PER_SILO} ({N_TRAIN_PER_SILO} train / {N_TEST_PER_SILO} test), "
        f"{N_TOTAL_SAMPLES} in total ({N_TRAIN_TOTAL}/{N_TEST_TOTAL}); gap = {GAP_WINDOWS} windows.")

GAP_COUNT = {}
for silo in SILOS:
    V = VALID[silo]
    blocks = feasible_blocks(V, N_TRAIN_PER_SILO, N_TEST_PER_SILO, GAP_WINDOWS)
    cnt = V[["c0", "c1", "c2", "c3"]].to_numpy()
    cs = np.vstack([np.zeros((1, N_CLASSES), int), np.cumsum(cnt, axis=0)])
    def score(ab):
        a, b = ab
        tr = cs[a + N_TRAIN_PER_SILO] - cs[a]; te = cs[b + N_TEST_PER_SILO] - cs[b]
        both = int(((tr > 0) & (te > 0)).sum())
        return (both, int((tr > 0).sum()), int((te > 0).sum()), a)
    if SELECTION_STRATEGY == "max_class_coverage":
        a, b = max(blocks, key=score)
    elif SELECTION_STRATEGY == "latest":
        a, b = blocks[-1]
    else:
        a, b = blocks[0]
    tr = V.iloc[a:a + N_TRAIN_PER_SILO].copy(); tr["split"] = "train"
    te = V.iloc[b:b + N_TEST_PER_SILO].copy(); te["split"] = "test"
    gapw = V.iloc[a + N_TRAIN_PER_SILO:b].copy()
    GAP_COUNT[silo] = len(gapw)
    n_val = int(round(VAL_FRAC * N_TRAIN_PER_SILO))
    val = tr.iloc[-n_val:]
    inner = tr.iloc[:max(0, N_TRAIN_PER_SILO - n_val - GAP_WINDOWS)]
    tr["tuning_role"] = np.where(tr.index.isin(val.index), "validation",
                                 np.where(tr.index.isin(inner.index), "inner_train", "purged_gap"))
    te["tuning_role"] = "test"
    sel = pd.concat([tr, te]).reset_index(drop=True)
    sel.insert(0, "sample_in_silo", np.arange(len(sel)))
    SELECTION[silo] = sel
    log(f"[{silo}] selected block a={a}, b={b}: train {tr.input_start.iloc[0]}..{tr.target_day18.iloc[-1]}, "
        f"test {te.input_start.iloc[0]}..{te.target_day18.iloc[-1]}; {len(gapw)} gap windows excluded; "
        f"inner-train={len(inner)}, val={len(val)}")
    save_df(gapw, "samples", f"excluded_gap_windows_{silo}.csv")

for silo, sel in SELECTION.items():
    tr, te = sel[sel.split == "train"], sel[sel.split == "test"]
    assert len(tr) == N_TRAIN_PER_SILO and len(te) == N_TEST_PER_SILO
    assert tr.start_row.is_unique and te.start_row.is_unique and not set(tr.start_row) & set(te.start_row)
    assert GAP_COUNT[silo] == GAP_WINDOWS
    assert te.start_row.min() >= tr.start_row.max() + GAP_WINDOWS + 1
    assert pd.to_datetime(tr.target_day18).max() < pd.to_datetime(te.target_day16).min()
ALL_SEL = pd.concat([SELECTION[s] for s in SILOS], ignore_index=True)
save_df(ALL_SEL, "samples", f"selected_samples_{N_TOTAL_SAMPLES}.csv")
assert len(ALL_SEL) == 3 * SAMPLES_PER_SILO == N_TOTAL_SAMPLES

t2 = []
for silo in SILOS:
    s = SELECTION[silo]; tr, te = s[s.split == "train"], s[s.split == "test"]
    t2.append(dict(Dataset=silo, Days=len(SILO_ARR[silo]["dates"]), Valid_windows=int(WINDOWS[silo].valid.sum()),
                   Training=len(tr), Testing=len(te), Total=len(s),
                   Train_period=f"{tr.input_start.iloc[0]} to {tr.target_day18.iloc[-1]}",
                   Test_period=f"{te.input_start.iloc[0]} to {te.target_day18.iloc[-1]}",
                   Inner_train=int((s.tuning_role == "inner_train").sum()),
                   Validation=int((s.tuning_role == "validation").sum())))
t2.append(dict(Dataset="Combined", Days=np.nan, Valid_windows=np.nan, Training=3 * N_TRAIN_PER_SILO,
               Testing=3 * N_TEST_PER_SILO, Total=N_TOTAL_SAMPLES, Train_period="per silo", Test_period="per silo",
               Inner_train=sum(r["Inner_train"] for r in t2), Validation=sum(r["Validation"] for r in t2)))
TABLE2 = pd.DataFrame(t2)
show_table(TABLE2, "TABLE 2 — Silo-wise dataset and sample distribution")
save_df(TABLE2, "tables", "Table02_sample_distribution.csv")

t3, coverage_notes = [], []
for ds in SILOS + ["Combined"]:
    sel = ALL_SEL if ds == "Combined" else SELECTION[ds]
    for split in ("train", "test"):
        x = sel[sel.split == split]
        row = dict(Dataset=ds, Split=split, Samples=len(x), Label_instances=len(x) * HORIZONS * N_SENSORS,
                   Masked_targets_missing_or_imputed=int(x.target_masked.sum()))
        for k in range(N_CLASSES):
            row[STAGE_NAMES[k]] = int(x[f"c{k}"].sum())
        t3.append(row)
        absent = [STAGE_NAMES[k] for k in range(N_CLASSES) if row[STAGE_NAMES[k]] == 0]
        if absent:
            coverage_notes.append(f"{ds}/{split}: absent {absent}")
TABLE3 = pd.DataFrame(t3)
show_table(TABLE3, "TABLE 3 — Training and testing target-stage distribution")
save_df(TABLE3, "tables", "Table03_target_stage_distribution.csv")
if coverage_notes:
    log("CLASS-COVERAGE LIMITATIONS (no duplication/synthesis applied):\n  " + "\n  ".join(coverage_notes), "WARN")
    log("With all four labels included, micro P/R/F1 still equal accuracy; absent classes simply contribute no "
        "true instances.", "WARN")
save_json({"strategy": SELECTION_STRATEGY, "gap_policy": GAP_POLICY, "gap_windows": GAP_WINDOWS,
           "samples_per_silo": SAMPLES_PER_SILO, "train_per_silo": N_TRAIN_PER_SILO, "test_per_silo": N_TEST_PER_SILO,
           "total_samples": N_TOTAL_SAMPLES, "coverage_limitations": coverage_notes,
           "note": "Period selection used target-label coverage only (no model output); all training windows precede all testing windows."},
          "samples", "selection_protocol.json")

In [ ]:
NODE_GROUP = np.array([0] * N_SENSORS + [1, 2])

def windows_of(ds, role):
    sel = ALL_SEL if ds == "Combined" else SELECTION[ds]
    if role == "train": x = sel[sel.split == "train"]
    elif role == "test": x = sel[sel.split == "test"]
    else: x = sel[sel.tuning_role == role]
    return list(zip(x.silo, x.start_row))

def fit_norm_stats(train_windows):
    vals = {g: [] for g in range(3)}
    days = {(s, r) for s, st in train_windows for r in range(st, st + INPUT_DAYS)}
    for s, r in sorted(days):
        x = SILO_ARR[s]["Xraw"][r]
        for g in range(3):
            vals[g].append(x[NODE_GROUP == g])
    stats = {}
    for g in range(3):
        v = np.concatenate(vals[g]); v = v[np.isfinite(v)]
        stats[g] = (float(v.mean()), float(v.std()) if v.std() > 0 else 1.0)
    return stats

def stack_windows(wins, stats):
    mean = np.array([stats[g][0] for g in NODE_GROUP]); std = np.array([stats[g][1] for g in NODE_GROUP])
    X = np.stack([SILO_ARR[s]["Xff"][st:st + INPUT_DAYS] for s, st in wins])
    Y = np.stack([SILO_ARR[s]["Y"][st + INPUT_DAYS:st + WIN_LEN] for s, st in wins])
    X = (X - mean) / std
    X = np.nan_to_num(X, nan=0.0)
    return X[..., None].astype(np.float32), Y.astype(np.int64)

def verify_alignment(wins, Y):
    for k, (s, st) in enumerate(wins):
        A = SILO_ARR[s]
        assert A["day"][st + WIN_LEN - 1] - A["day"][st] == WIN_LEN - 1, "non-consecutive window"
        assert A["dates"][st + INPUT_DAYS - 1] < A["dates"][st + INPUT_DAYS], "input overlaps target"
        T = A["Xraw"][st + INPUT_DAYS:st + WIN_LEN, :N_SENSORS]
        exp = label_temperature(T); exp[:, ~A["eval_mask"]] = MISSING_LABEL
        assert np.array_equal(exp, Y[k]), "target labels do not match raw temperatures"

_ARRAY_CACHE = {}
def get_arrays(ds, phase):
    key = (ds, phase)
    if key not in _ARRAY_CACHE:
        tr_role, ev_role = ("train", "test") if phase == "final" else ("inner_train", "validation")
        wtr, wev = windows_of(ds, tr_role), windows_of(ds, ev_role)
        stats = fit_norm_stats(wtr)
        Xtr, Ytr = stack_windows(wtr, stats); Xev, Yev = stack_windows(wev, stats)
        _ARRAY_CACHE[key] = dict(X_tr=Xtr, Y_tr=Ytr, X_ev=Xev, Y_ev=Yev, w_tr=wtr, w_ev=wev, stats=stats)
    return _ARRAY_CACHE[key]

t4 = []
for ds in DATASETS:
    d = get_arrays(ds, "final")
    verify_alignment(d["w_tr"], d["Y_tr"]); verify_alignment(d["w_ev"], d["Y_ev"])
    n_tr = 3 * N_TRAIN_PER_SILO if ds == "Combined" else N_TRAIN_PER_SILO
    n_te = 3 * N_TEST_PER_SILO if ds == "Combined" else N_TEST_PER_SILO
    exp = {"X_train": (n_tr, INPUT_DAYS, N_NODES, 1), "Y_train": (n_tr, HORIZONS, N_SENSORS),
           "X_test": (n_te, INPUT_DAYS, N_NODES, 1), "Y_test": (n_te, HORIZONS, N_SENSORS)}
    got = {"X_train": d["X_tr"].shape, "Y_train": d["Y_tr"].shape, "X_test": d["X_ev"].shape, "Y_test": d["Y_ev"].shape}
    for k in exp:
        assert got[k] == exp[k], f"{ds} {k}: got {got[k]}, expected {exp[k]}"
    assert np.isfinite(d["X_tr"]).all() and np.isfinite(d["X_ev"]).all()
    t4.append(dict(Dataset=ds, X_train=str(got["X_train"]), Y_train=str(got["Y_train"]),
                   X_test=str(got["X_test"]), Y_test=str(got["Y_test"]),
                   Train_logits=str((n_tr, HORIZONS, N_SENSORS, N_CLASSES)),
                   Test_logits=str((n_te, HORIZONS, N_SENSORS, N_CLASSES)),
                   Test_pred_labels=str((n_te, HORIZONS, N_SENSORS)),
                   Adjacency=str(ADJ.shape), Edge_index=str(EDGE_INDEX_NP.shape),
                   Norm_stats_grainT_ambT_RH=str({g: tuple(round(v, 3) for v in d["stats"][g]) for g in range(3)})))
    np.savez_compressed(versioned_path(os.path.join(DIRS["tensors"], f"{ds}_final_tensors.npz")),
                        X_train=d["X_tr"], Y_train=d["Y_tr"], X_test=d["X_ev"], Y_test=d["Y_ev"],
                        train_windows=np.array([f"{s}:{r}" for s, r in d["w_tr"]]),
                        test_windows=np.array([f"{s}:{r}" for s, r in d["w_ev"]]))
TABLE4 = pd.DataFrame(t4)
show_table(TABLE4, "TABLE 4 — Input and output tensor structure (layout: batch, time, node, feature)")
save_df(TABLE4, "tables", "Table04_tensor_structure.csv")
print("Feature dimension is kept explicitly: X = (B, 15, 268, 1). Targets: -1 marks a missing (masked) label.")

In [ ]:
from torch_geometric.nn import GATConv

ARCH_DESCRIPTION = {
    "GAT_GRU": "Proposed: 2x GATConv (multi-head attention) per day -> per-node GRU over 15 days -> shared head -> 3x4 logits",
}
assert ARCHS == ["GAT_GRU"], "Set ARCHS = ['GAT_GRU'] in Cell 2 for the GAT-GRU-only run."

class GATGRUStageModel(nn.Module):
    def __init__(self, edge_index, hidden=64, heads=4, gru_layers=1, dropout=0.2,
                 node_emb_dim=NODE_EMB_DIM, num_spatial_layers=NUM_SPATIAL_LAYERS,
                 num_nodes=N_NODES, n_targets=N_SENSORS, in_feats=1,
                 horizons=HORIZONS, n_classes=N_CLASSES):
        super().__init__()
        assert hidden % heads == 0, f"hidden ({hidden}) must be divisible by heads ({heads})"
        self.arch = "GAT_GRU"
        self.hidden, self.horizons, self.n_classes = hidden, horizons, n_classes
        self.num_nodes, self.n_targets = num_nodes, n_targets
        self.register_buffer("edge_index", edge_index.long(), persistent=False)
        self._ei_cache = {}
        self.node_emb = nn.Embedding(num_nodes, node_emb_dim) if node_emb_dim > 0 else None
        d_in = in_feats + node_emb_dim
        self.spatial, self.norms = nn.ModuleList(), nn.ModuleList()
        for l in range(num_spatial_layers):
            di = d_in if l == 0 else hidden
            self.spatial.append(GATConv(di, hidden // heads, heads=heads, concat=True,
                                        dropout=dropout, add_self_loops=True))
            self.norms.append(nn.LayerNorm(hidden))
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, hidden, num_layers=gru_layers, batch_first=True,
                          dropout=dropout if gru_layers > 1 else 0.0)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
                                  nn.Linear(hidden, horizons * n_classes))

    def batched_edge_index(self, num_graphs):
        key = (num_graphs, self.edge_index.device)
        if key not in self._ei_cache:
            E = self.edge_index.shape[1]
            off = torch.arange(num_graphs, device=self.edge_index.device).repeat_interleave(E) * self.num_nodes
            self._ei_cache[key] = self.edge_index.repeat(1, num_graphs) + off
        return self._ei_cache[key]

    def forward(self, x, trace=None):
        B, T, N, Fin = x.shape
        if trace is not None: trace["01 input (B,T,N,F)"] = tuple(x.shape)
        h = x
        if self.node_emb is not None:
            e = self.node_emb.weight[None, None].expand(B, T, N, -1)
            h = torch.cat([h, e], dim=-1)
            if trace is not None: trace["02 + node embedding (B,T,N,F+E)"] = tuple(h.shape)
        h = h.reshape(B * T * N, -1)
        if trace is not None: trace["03 flattened node features (B*T*N,F')"] = tuple(h.shape)
        ei = self.batched_edge_index(B * T)
        if trace is not None: trace["04 batched edge_index (2, B*T*E)"] = tuple(ei.shape)
        for l, (gat, norm) in enumerate(zip(self.spatial, self.norms)):
            z = self.drop(norm(F.elu(gat(h, ei))))
            h = z + h if z.shape == h.shape else z
            if trace is not None: trace[f"05.{l+1} GAT layer {l+1} (B*T*N,H)"] = tuple(h.shape)
        h = h.reshape(B, T, N, self.hidden)
        if trace is not None: trace["06 spatial embeddings (B,T,N,H)"] = tuple(h.shape)
        seq = h.permute(0, 2, 1, 3).reshape(B * N, T, self.hidden)
        if trace is not None: trace["07 GRU input sequences (B*N,T,H)"] = tuple(seq.shape)
        out, h_n = self.gru(seq)
        if trace is not None:
            trace["08 GRU outputs (B*N,T,H)"] = tuple(out.shape)
            trace["09 GRU final hidden (layers,B*N,H)"] = tuple(h_n.shape)
        z = h_n[-1]
        if trace is not None: trace["10 temporal representation (B*N,H)"] = tuple(z.shape)
        logits = self.head(z).reshape(B, N, self.horizons, self.n_classes)
        if trace is not None: trace["11 head output, all nodes (B,N,3,4)"] = tuple(logits.shape)
        logits = logits[:, :self.n_targets].permute(0, 2, 1, 3).contiguous()
        if trace is not None: trace["12 logits (B,horizon,sensor,class)"] = tuple(logits.shape)
        return logits

EDGE_INDEX_T = torch.as_tensor(EDGE_INDEX_NP, dtype=torch.long)

def build_model(arch, hp):
    assert arch == "GAT_GRU", f"This notebook builds GAT-GRU only (got {arch})."
    return GATGRUStageModel(EDGE_INDEX_T, hidden=hp["hidden"], heads=hp.get("heads", 1),
                            gru_layers=hp.get("gru_layers", 1), dropout=hp["dropout"])

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Architecture defined:", list(ARCH_DESCRIPTION))

In [ ]:
assert "GATGRUStageModel" in globals(), "Run Cell 10 (GAT-GRU-only version) first."
assert ARCHS == ["GAT_GRU"], "Set ARCHS = ['GAT_GRU'] in Cell 2."

set_seed(0)
B_TEST = 2
x_syn = torch.randn(B_TEST, INPUT_DAYS, N_NODES, 1, device=DEVICE)
y_syn = torch.randint(0, N_CLASSES, (B_TEST, HORIZONS, N_SENSORS), device=DEVICE)
y_syn[0, 0, :5] = MISSING_LABEL

model = build_model("GAT_GRU", SMOKE_HP).to(DEVICE)
trace = OrderedDict()
logits = model(x_syn, trace=trace)
print(f"--- GAT_GRU: {ARCH_DESCRIPTION['GAT_GRU']}")
for k, v in trace.items():
    print(f"   {k:<45s} {v}")
assert logits.shape == (B_TEST, HORIZONS, N_SENSORS, N_CLASSES)

probs = logits.softmax(-1)
assert torch.allclose(probs.sum(-1), torch.ones_like(probs.sum(-1)), atol=1e-5)
preds = probs.argmax(-1)
assert preds.shape == (B_TEST, HORIZONS, N_SENSORS)
print(f"   softmax probabilities {tuple(probs.shape)} sum to 1; predicted stages {tuple(preds.shape)}")

crit = nn.CrossEntropyLoss(ignore_index=MISSING_LABEL)
loss = crit(logits.reshape(-1, N_CLASSES), y_syn.reshape(-1))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
before = [p.detach().clone() for p in model.parameters()]
loss.backward()
grads_ok = all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters() if p.requires_grad)
opt.step()
changed = sum(not torch.equal(b, p) for b, p in zip(before, model.parameters()))
assert grads_ok and changed > 0, "gradients missing/non-finite or no parameter was updated"
print(f"   loss={loss.item():.4f}; finite gradients on all parameters; {changed} tensors updated by the optimizer")

model.eval()
with torch.no_grad():
    o1 = model(x_syn)
    x2 = x_syn.clone(); x2[:, :, 0, 0] += 5.0
    o2 = model(x2)
others_changed = not torch.allclose(o1[:, :, 1:], o2[:, :, 1:], atol=1e-6)
assert others_changed, "GAT-GRU should propagate information across the graph"
print(f"   perturbing node 0 changes other nodes' outputs: {others_changed} (expected True)")

with torch.no_grad():
    h0 = torch.cat([x_syn[:1, :1].reshape(N_NODES, 1), model.node_emb.weight], -1)
    _, (ei_sl, alpha) = model.spatial[0](h0, model.edge_index, return_attention_weights=True)
print(f"   attention (one graph): edge_index incl. self-loops {tuple(ei_sl.shape)}, alpha (edges, heads) {tuple(alpha.shape)}")

n_params = count_params(model)
print(f"   trainable parameters (SMOKE_HP): {n_params:,}")
save_df(pd.DataFrame([dict(arch="GAT_GRU", params=n_params, loss=float(loss), output=str(tuple(logits.shape)),
                           attention_shape=str(tuple(alpha.shape)), message_passing_ok=others_changed)]),
        "models", "unit_test_forward_pass.csv")
del model, opt

dC = get_arrays("Combined", "final")
m_chk = build_model("GAT_GRU", SMOKE_HP).to(DEVICE).eval()
with torch.no_grad():
    lt = torch.cat([m_chk(torch.from_numpy(dC["X_tr"][i:i + 16]).to(DEVICE)).cpu() for i in range(0, len(dC["X_tr"]), 16)])
    le = torch.cat([m_chk(torch.from_numpy(dC["X_ev"][i:i + 16]).to(DEVICE)).cpu() for i in range(0, len(dC["X_ev"]), 16)])
assert lt.shape == (3 * N_TRAIN_PER_SILO, HORIZONS, N_SENSORS, N_CLASSES), lt.shape
assert le.shape == (3 * N_TEST_PER_SILO, HORIZONS, N_SENSORS, N_CLASSES), le.shape
assert le.argmax(-1).shape == (3 * N_TEST_PER_SILO, HORIZONS, N_SENSORS)
print(f"\nFull-size check: training logits {tuple(lt.shape)}, testing logits {tuple(le.shape)}, "
      f"predicted labels {tuple(le.argmax(-1).shape)}, adjacency {ADJ.shape}")
del m_chk, lt, le
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
print("\nCELL 11 PASSED — GAT-GRU forward pass, loss, backprop, message passing and output shapes verified.")

In [ ]:
def class_weights(Y_np):
    if LOSS_WEIGHTING == "none":
        return None
    c = np.bincount(Y_np[Y_np >= 0].ravel(), minlength=N_CLASSES).astype(float)
    w = np.where(c > 0, 1.0 / np.sqrt(np.maximum(c, 1)), 0.0)
    w = w / (w * c).sum() * c.sum()
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

def compute_overall_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).ravel(); y_pred = np.asarray(y_pred).ravel()
    mask = y_true >= 0
    yt, yp = y_true[mask], y_pred[mask]
    acc = accuracy_score(yt, yp)
    p, r, f1, _ = precision_recall_fscore_support(yt, yp, labels=list(range(N_CLASSES)), average="micro", zero_division=0)
    assert abs(p - acc) < 1e-9 and abs(r - acc) < 1e-9 and abs(f1 - acc) < 1e-9, "micro P/R/F1 must equal accuracy"
    mse = float(np.mean((yt.astype(float) - yp.astype(float)) ** 2))
    return OrderedDict(Accuracy=float(acc), Micro_Precision=float(p), Micro_Recall=float(r), Micro_F1=float(f1),
                       MSE=mse, RMSE=float(math.sqrt(mse)), n_evaluated=int(mask.sum()), n_masked=int((~mask).sum()))

METRIC_COLS = ["Accuracy", "Micro_Precision", "Micro_Recall", "Micro_F1", "MSE", "RMSE"]

def train_one_epoch(model, X, Y, crit, opt, batch_size, gen):
    model.train()
    perm = torch.randperm(X.shape[0], generator=gen).to(X.device)
    tot, cnt = 0.0, 0
    for i in range(0, X.shape[0], batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = X[idx], Y[idx]
        nv = int((yb >= 0).sum())
        if nv == 0:
            continue
        opt.zero_grad(set_to_none=True)
        loss = crit(model(xb).reshape(-1, N_CLASSES), yb.reshape(-1))
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite training loss ({loss.item()}) — stopping this run.")
        loss.backward()
        if GRAD_CLIP:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        opt.step()
        tot += loss.item() * nv; cnt += nv
    return tot / max(cnt, 1)

@torch.no_grad()
def predict_logits(model, X, batch_size=EVAL_BATCH_SIZE):
    model.eval()
    return torch.cat([model(X[i:i + batch_size]).float() for i in range(0, X.shape[0], batch_size)])

def evaluate(model, X, Y, crit):
    logits = predict_logits(model, X)
    loss = float(crit(logits.reshape(-1, N_CLASSES), Y.reshape(-1)))
    probs = logits.softmax(-1); preds = probs.argmax(-1)
    m = compute_overall_metrics(Y.cpu().numpy(), preds.cpu().numpy())
    return dict(loss=loss, metrics=m, preds=preds.cpu().numpy().astype(np.int8), probs=probs.cpu().numpy().astype(np.float16))

def to_dev(arr, dtype):
    return torch.as_tensor(arr, dtype=dtype, device=DEVICE)

def fit_model(arch, hp, seed, X_tr, Y_tr, X_ev=None, Y_ev=None, max_epochs=MAX_EPOCHS, patience=PATIENCE,
              fixed_epochs=None, optuna_trial=None, tag="", monitor=None):
    set_seed(seed)
    model = build_model(arch, hp).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS, eta_min=hp["lr"] * 0.01)
    crit = nn.CrossEntropyLoss(weight=class_weights(Y_tr.cpu().numpy()), ignore_index=MISSING_LABEL)
    gen = torch.Generator(); gen.manual_seed(seed)
    n_epochs = fixed_epochs or max_epochs
    best = dict(loss=float("inf"), epoch=0, state=None, acc=float("nan")); bad = 0; hist = []
    for ep in range(1, n_epochs + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, X_tr, Y_tr, crit, opt, hp["batch_size"], gen)
        row = dict(epoch=ep, train_loss=tr_loss, lr=opt.param_groups[0]["lr"])
        sched.step()
        if X_ev is not None:
            ev = evaluate(model, X_ev, Y_ev, crit)
            row.update(val_loss=ev["loss"], val_accuracy=ev["metrics"]["Accuracy"])
            if ev["loss"] < best["loss"] - 1e-6:
                best.update(loss=ev["loss"], epoch=ep, acc=ev["metrics"]["Accuracy"],
                            state={k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
                bad = 0
            else:
                bad += 1
            if optuna_trial is not None:
                optuna_trial.report(ev["loss"], ep)
                if optuna_trial.should_prune():
                    import optuna
                    raise optuna.TrialPruned()
        if monitor is not None:
            em = evaluate(model, monitor[0], monitor[1], crit)
            row.update(test_loss=em["loss"], test_accuracy=em["metrics"]["Accuracy"])
        row["epoch_time_s"] = time.time() - t0
        hist.append(row)
        if ep == 1 or ep % 50 == 0 or ep == n_epochs:
            log(f"{tag} ep {ep:4d}/{n_epochs} train_loss={tr_loss:.4f}" +
                (f" val_loss={row['val_loss']:.4f} val_acc={row['val_accuracy']:.4f}" if X_ev is not None else "") +
                (f" test_loss={row['test_loss']:.4f} test_acc={row['test_accuracy']:.4f}" if monitor is not None else "") +
                f" ({row['epoch_time_s']:.1f}s/epoch)")
        if X_ev is not None and fixed_epochs is None and bad >= patience:
            log(f"{tag} early stop at epoch {ep} (best epoch {best['epoch']})")
            break
    if X_ev is not None and best["state"] is not None:
        model.load_state_dict(best["state"])
    return model, pd.DataFrame(hist), best, crit

print("Engine ready. Metrics:", METRIC_COLS)

In [ ]:
dT = get_arrays("Combined", "tune")
Xtr, Ytr = to_dev(dT["X_tr"], torch.float32), to_dev(dT["Y_tr"], torch.long)
Xva, Yva = to_dev(dT["X_ev"], torch.float32), to_dev(dT["Y_ev"], torch.long)
smoke_rows = []
for arch in ARCHS:
    t0 = time.time()
    model, hist, best, crit = fit_model(arch, SMOKE_HP, 42, Xtr, Ytr, Xva, Yva, fixed_epochs=SMOKE_EPOCHS, tag=f"[smoke {arch}]")
    ck = versioned_path(os.path.join(DIRS["smoke_test"], f"{arch}_smoke.pt"))
    torch.save({"state_dict": model.state_dict(), "hp": SMOKE_HP, "arch": arch}, ck)
    reloaded = build_model(arch, SMOKE_HP).to(DEVICE)
    reloaded.load_state_dict(torch.load(ck, map_location=DEVICE)["state_dict"])

    sd_saved = torch.load(ck, map_location="cpu")["state_dict"]
    sd_model = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    same = sd_saved.keys() == sd_model.keys() and all(torch.equal(sd_saved[k], sd_model[k]) for k in sd_model)
    assert same, "checkpoint weights differ from the trained model"

    lo1, lo2 = predict_logits(model, Xva), predict_logits(reloaded, Xva)
    max_diff = (lo1 - lo2).abs().max().item()
    label_agree = (lo1.argmax(-1) == lo2.argmax(-1)).float().mean().item()
    log(f"[smoke {arch}] reload: weights identical={same}; max|logit diff|={max_diff:.2e}; label agreement={label_agree:.6f}")
    assert max_diff < 1e-3, f"reloaded model outputs differ too much (max diff {max_diff:.2e})"
    hist.to_csv(versioned_path(os.path.join(DIRS["smoke_test"], f"{arch}_smoke_history.csv")), index=False)
    smoke_rows.append(dict(arch=arch, epochs=len(hist), first_train_loss=hist.train_loss.iloc[0],
                           last_train_loss=hist.train_loss.iloc[-1], last_val_loss=hist.val_loss.iloc[-1],
                           checkpoint_reload_identical=same, reload_max_logit_diff=max_diff, reload_label_agreement=label_agree, seconds=round(time.time() - t0, 1)))
    del model, reloaded
smoke_df = pd.DataFrame(smoke_rows)
show_table(smoke_df, "SMOKE TEST (validation split only — implementation check, not scientific results)")
save_df(smoke_df, "smoke_test", "smoke_test_summary.csv")
SMOKE_TEST_PASSED = bool(np.isfinite(smoke_df[["last_train_loss", "last_val_loss"]].to_numpy()).all())
assert SMOKE_TEST_PASSED
log("SMOKE TEST PASSED: loading, labels, windows, split, tensors, graph, forward, loss, backprop, evaluation, saving.")
del Xtr, Ytr, Xva, Yva
if DEVICE.type == "cuda": torch.cuda.empty_cache()

In [ ]:
assert globals().get("SMOKE_TEST_PASSED"), "Run the smoke test (Cell 13) first."
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def suggest_hp(trial, arch):
    hp = {k: trial.suggest_categorical(k, SEARCH_SPACE[k]) for k in ("lr", "batch_size", "hidden", "dropout", "weight_decay")}
    hp["heads"] = trial.suggest_categorical("heads", SEARCH_SPACE["heads"]) if arch in ("GAT_GRU", "GAT_only") else 1
    hp["gru_layers"] = trial.suggest_categorical("gru_layers", SEARCH_SPACE["gru_layers"]) if arch != "GAT_only" else 1
    return hp

def tune_architecture(arch):
    d = get_arrays(TUNE_DATASET, "tune")
    Xtr, Ytr = to_dev(d["X_tr"], torch.float32), to_dev(d["Y_tr"], torch.long)
    Xva, Yva = to_dev(d["X_ev"], torch.float32), to_dev(d["Y_ev"], torch.long)
    drive_db = os.path.join(DIRS["tuning"], f"optuna_{arch}.db")
    local_db = f"/tmp/optuna_{RUN_NAME}_{arch}.db"
    if os.path.exists(drive_db) and not os.path.exists(local_db):
        shutil.copy(drive_db, local_db)
    study = optuna.create_study(study_name=arch, storage=f"sqlite:///{local_db}", load_if_exists=True,
                                direction="minimize", sampler=optuna.samplers.TPESampler(seed=TUNING_SEED),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10))
    def objective(trial):
        hp = suggest_hp(trial, arch)
        t0 = time.time()
        try:
            model, hist, best, _ = fit_model(arch, hp, TUNING_SEED, Xtr, Ytr, Xva, Yva, max_epochs=TUNE_MAX_EPOCHS,
                                             patience=TUNE_PATIENCE, optuna_trial=trial, tag=f"[tune {arch} #{trial.number}]")
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); log(f"[tune {arch} #{trial.number}] CUDA OOM -> pruned", "WARN")
            raise optuna.TrialPruned()
        trial.set_user_attr("best_epoch", best["epoch"]); trial.set_user_attr("best_val_accuracy", best["acc"])
        trial.set_user_attr("params", count_params(model)); trial.set_user_attr("seconds", time.time() - t0)
        hist.to_csv(versioned_path(os.path.join(DIRS["tuning"], f"{arch}_trial{trial.number:03d}_history.csv")), index=False)
        del model
        return best["loss"]
    def backup(study, trial):
        shutil.copy(local_db, drive_db)
    remaining = N_TUNING_TRIALS - len([t for t in study.trials if t.state.is_finished()])
    if remaining > 0:
        study.optimize(objective, n_trials=remaining, callbacks=[backup])
    shutil.copy(local_db, drive_db)
    tdf = study.trials_dataframe()
    best_hp = suggest_hp(optuna.trial.FixedTrial(study.best_params), arch)
    return study, tdf, best_hp

BEST_HP, t6 = {}, []
for arch in ARCHS:
    log(f"Tuning {arch}: {N_TUNING_TRIALS} trials, <= {TUNE_MAX_EPOCHS} epochs each, dataset={TUNE_DATASET} (inner-train vs validation)")
    study, tdf, best_hp = tune_architecture(arch)
    BEST_HP[arch] = best_hp
    save_df(tdf, "tuning", f"{arch}_tuning_trials.csv")
    bt = study.best_trial
    t6.append(dict(Architecture=arch, Trials_completed=int((tdf.state == "COMPLETE").sum()),
                   Trials_pruned=int((tdf.state == "PRUNED").sum()), Best_val_loss=bt.value,
                   Best_val_accuracy=bt.user_attrs.get("best_val_accuracy"), Best_epoch=bt.user_attrs.get("best_epoch"),
                   **{f"best_{k}": v for k, v in best_hp.items()}))
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
save_json(BEST_HP, "tuning", "best_hyperparameters.json")
TABLE6 = pd.DataFrame(t6)
show_table(TABLE6, "TABLE 6 — Hyperparameter tuning results (selection on validation loss; test data never used)")
save_df(TABLE6, "tables", "Table06_hyperparameter_tuning.csv")

In [ ]:
assert globals().get("SMOKE_TEST_PASSED"), "Run the smoke test first."
if "BEST_HP" not in globals() or not BEST_HP:
    with open(os.path.join(DIRS["tuning"], "best_hyperparameters.json")) as f:
        BEST_HP = json.load(f)

def run_trial(ds, arch, trial_name, seed):
    rdir = os.path.join(DIRS["trials"], ds, arch, f"{trial_name}_seed{seed}")
    done = os.path.join(rdir, "metrics.json")
    if os.path.exists(done):
        with open(done) as f:
            log(f"[{ds}/{arch}/{trial_name}] already complete — loaded (resume)")
            return json.load(f)
    if os.path.isdir(rdir):
        shutil.move(rdir, rdir + f"_incomplete_{datetime.datetime.now():%Y%m%d_%H%M%S}")
    os.makedirs(rdir)
    hp = BEST_HP[arch]
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t_start = time.time()
    dF = get_arrays(ds, "final")
    Xte, Yte = to_dev(dF["X_ev"], torch.float32), to_dev(dF["Y_ev"], torch.long)
    bestA, dA = dict(epoch=np.nan, loss=np.nan), None
    if STRICT_EPOCHS:

        model, hist, _, crit = fit_model(arch, hp, seed, to_dev(dF["X_tr"], torch.float32), to_dev(dF["Y_tr"], torch.long),
                                         fixed_epochs=MAX_EPOCHS, monitor=(Xte, Yte), tag=f"[{ds}/{arch}/{trial_name}]")
        hist.to_csv(os.path.join(rdir, f"history_{MAX_EPOCHS}_epochs.csv"), index=False)
        epochs_final = len(hist)
        assert epochs_final == MAX_EPOCHS, f"trained {epochs_final} epochs, expected {MAX_EPOCHS}"
    else:

        dA = get_arrays(ds, "tune")
        mA, hA, bestA, _ = fit_model(arch, hp, seed, to_dev(dA["X_tr"], torch.float32), to_dev(dA["Y_tr"], torch.long),
                                     to_dev(dA["X_ev"], torch.float32), to_dev(dA["Y_ev"], torch.long),
                                     tag=f"[{ds}/{arch}/{trial_name} A]")
        hA.to_csv(os.path.join(rdir, "history_phaseA_validation.csv"), index=False)
        if FINAL_FIT_MODE == "refit_full":
            model, hB, _, crit = fit_model(arch, hp, seed, to_dev(dF["X_tr"], torch.float32), to_dev(dF["Y_tr"], torch.long),
                                           fixed_epochs=max(bestA["epoch"], 1), tag=f"[{ds}/{arch}/{trial_name} B]")
            hB.to_csv(os.path.join(rdir, "history_phaseB_refit.csv"), index=False)
            epochs_final = len(hB)
            del mA
        else:
            model, crit = mA, nn.CrossEntropyLoss(ignore_index=MISSING_LABEL)
            epochs_final = bestA["epoch"]
            Xa, _ = stack_windows(dF["w_ev"], dA["stats"])
            Xte = to_dev(Xa, torch.float32)
    train_seconds = time.time() - t_start
    t_inf = time.time()
    ev = evaluate(model, Xte, Yte, crit)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    inf_seconds = time.time() - t_inf

    if STRICT_EPOCHS or FINAL_FIT_MODE == "refit_full":
        Xtr_e = dF["X_tr"]
    else:
        Xtr_e, _ = stack_windows(dF["w_tr"], dA["stats"])
    ev_tr = evaluate(model, to_dev(Xtr_e, torch.float32), to_dev(dF["Y_tr"], torch.long), crit)
    def horizon_acc(pred, true):
        out = {}
        for h in range(HORIZONS):
            mk = true[:, h] >= 0
            out[f"Day{INPUT_DAYS + 1 + h}"] = float((pred[:, h][mk] == true[:, h][mk]).mean())
        return out
    hz_te, hz_tr = horizon_acc(ev["preds"], dF["Y_ev"]), horizon_acc(ev_tr["preds"], dF["Y_tr"])
    torch.save({"state_dict": model.state_dict(), "arch": arch, "hp": hp, "seed": seed, "dataset": ds,
                "norm_stats": dF["stats"], "epochs_final": epochs_final, "edge_index": EDGE_INDEX_T},
               os.path.join(rdir, "model.pt"))
    np.savez_compressed(os.path.join(rdir, "test_predictions.npz"), pred_stage=ev["preds"], true_stage=dF["Y_ev"].astype(np.int8),
                        probabilities=ev["probs"], silo=np.array([s for s, _ in dF["w_ev"]]),
                        start_row=np.array([r for _, r in dF["w_ev"]]))
    res = OrderedDict(Dataset=ds, Architecture=arch, Trial=trial_name, Seed=seed, **ev["metrics"],
                      Best_val_epoch=bestA["epoch"], Best_val_loss=bestA["loss"], Epochs_final=epochs_final,
                      Train_seconds=train_seconds, Inference_seconds=inf_seconds,
                      Peak_GPU_MB=(torch.cuda.max_memory_allocated() / 2**20) if DEVICE.type == "cuda" else np.nan,
                      Params=count_params(model), Device=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
                      Final_fit_mode=(f"strict_{MAX_EPOCHS}_epochs" if STRICT_EPOCHS else FINAL_FIT_MODE), hp=json.dumps(hp))
    res.update({f"Train_{k}": ev_tr["metrics"][k] for k in METRIC_COLS})
    res.update({f"Test_Acc_{k}": v for k, v in hz_te.items()})
    res.update({f"Train_Acc_{k}": v for k, v in hz_tr.items()})
    with open(os.path.join(rdir, "config.json"), "w") as f:
        json.dump(dict(hp=hp, seed=seed, arch=arch, dataset=ds, gap_policy=GAP_POLICY, loss_weighting=LOSS_WEIGHTING), f, indent=2)
    with open(done, "w") as f:
        json.dump(res, f, indent=2, cls=NpEncoder)
    log(f"[{ds}/{arch}/{trial_name}] TEST acc={res['Accuracy']:.4f} MSE={res['MSE']:.4f} RMSE={res['RMSE']:.4f} "
        f"(n={res['n_evaluated']}, {train_seconds/60:.1f} min)")
    del model
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return res

RESULTS = []
for ds in DATASETS:
    for arch in ARCHS:
        for trial_name, seed in SEEDS.items():
            RESULTS.append(run_trial(ds, arch, trial_name, seed))
RES_DF = pd.DataFrame(RESULTS)
save_df(RES_DF, "metrics", "all_trial_metrics.csv")
assert len(RES_DF) == len(DATASETS) * len(ARCHS) * len(SEEDS)

In [ ]:
if "RES_DF" not in globals():
    RES_DF = pd.read_csv(os.path.join(DIRS["metrics"], "all_trial_metrics.csv"))
TRIALS = list(SEEDS)

t5 = []
for arch in ARCHS:
    m = build_model(arch, BEST_HP[arch])
    t5.append(dict(Architecture=arch, Description=ARCH_DESCRIPTION[arch], Params=count_params(m),
                   Spatial=("GATConv" if "GAT" in arch else "GraphConv" if arch == "GraphConv_GRU" else "per-node Linear (no message passing)"),
                   Temporal=("GRU" if arch != "GAT_only" else "concatenation + MLP (non-recurrent)"),
                   **{k: v for k, v in BEST_HP[arch].items()}))
TABLE5 = pd.DataFrame(t5); show_table(TABLE5, "TABLE 5 — Model architecture and parameter counts")
save_df(TABLE5, "tables", "Table05_architectures_params.csv")

def stats_table(df):
    rows = []
    for (ds, arch), g in df.groupby(["Dataset", "Architecture"], sort=False):
        g = g.set_index("Trial").loc[TRIALS]
        for met in METRIC_COLS:
            v = g[met].to_numpy(float)
            rows.append(dict(Dataset=ds, Architecture=arch, Metric=met, **{t: v[i] for i, t in enumerate(TRIALS)},
                             Mean=v.mean(), SD_ddof1=v.std(ddof=1)))
    return pd.DataFrame(rows)
STATS = stats_table(RES_DF)
save_df(STATS, "metrics", "statistical_summary_mean_sd_ddof1.csv")

def per_dataset_table(ds):
    d = RES_DF[RES_DF.Dataset == ds][["Architecture", "Trial", "Seed"] + METRIC_COLS + ["n_evaluated"]].copy()
    agg = []
    for arch, g in d.groupby("Architecture", sort=False):
        agg.append(dict(Architecture=arch, Trial="Mean", **{m: g[m].mean() for m in METRIC_COLS}))
        agg.append(dict(Architecture=arch, Trial="SD (ddof=1)", **{m: g[m].std(ddof=1) for m in METRIC_COLS}))
    return pd.concat([d, pd.DataFrame(agg)], ignore_index=True)

for num, ds in zip((7, 8, 9, 10), DATASETS):
    t = per_dataset_table(ds)
    show_table(t.round(6), f"TABLE {num} — {ds} three-trial overall evaluation")
    save_df(t, "tables", f"Table{num:02d}_{ds}_three_trials.csv")

def fmt(m, s): return f"{m:.4f} ± {s:.4f}"
piv = STATS.pivot_table(index=["Dataset", "Architecture"], columns="Metric", values=["Mean", "SD_ddof1"], sort=False)
t11 = []
for ds in DATASETS:
    ref_acc = piv.loc[(ds, "GAT_GRU"), ("Mean", "Accuracy")]; ref_rmse = piv.loc[(ds, "GAT_GRU"), ("Mean", "RMSE")]
    for arch in ARCHS:
        r = piv.loc[(ds, arch)]
        t11.append(dict(Dataset=ds, Architecture=arch, Removed_or_replaced={"GAT_GRU": "— (full model)", "GAT_only": "GRU removed",
                        "GRU_only": "graph attention removed", "GraphConv_GRU": "attention -> GraphConv"}[arch],
                        Accuracy=fmt(r[("Mean", "Accuracy")], r[("SD_ddof1", "Accuracy")]),
                        Micro_F1=fmt(r[("Mean", "Micro_F1")], r[("SD_ddof1", "Micro_F1")]),
                        RMSE=fmt(r[("Mean", "RMSE")], r[("SD_ddof1", "RMSE")]),
                        Delta_mean_accuracy_vs_GAT_GRU=r[("Mean", "Accuracy")] - ref_acc,
                        Delta_mean_RMSE_vs_GAT_GRU=r[("Mean", "RMSE")] - ref_rmse))
TABLE11 = pd.DataFrame(t11); show_table(TABLE11, "TABLE 11 — Architecture-wise ablation comparison")
save_df(TABLE11, "tables", "Table11_ablation.csv")

TABLE12 = pd.DataFrame([dict(Dataset=ds, Architecture=arch,
                             **{m: fmt(piv.loc[(ds, arch), ("Mean", m)], piv.loc[(ds, arch), ("SD_ddof1", m)]) for m in METRIC_COLS})
                        for ds in DATASETS for arch in ARCHS])
show_table(TABLE12, "TABLE 12 — Mean ± sample SD (ddof=1) over three trials")
save_df(TABLE12, "tables", "Table12_mean_sd_summary.csv")

TABLE13 = RES_DF.pivot_table(index=["Dataset", "Architecture"], columns="Trial", values=["MSE", "RMSE"], sort=False)
TABLE13.columns = [f"{a}_{b}" for a, b in TABLE13.columns]
for m in ("MSE", "RMSE"):
    TABLE13[f"{m}_mean"] = TABLE13[[f"{m}_{t}" for t in TRIALS]].mean(1)
    TABLE13[f"{m}_SD"] = TABLE13[[f"{m}_{t}" for t in TRIALS]].std(1, ddof=1)
TABLE13 = TABLE13.reset_index()
show_table(TABLE13.round(6), "TABLE 13 — MSE and RMSE (ordinal stage-label errors, stage units)")
save_df(TABLE13, "tables", "Table13_mse_rmse.csv")

TABLE14 = RES_DF.groupby(["Dataset", "Architecture"], sort=False).agg(
    Params=("Params", "first"), Mean_train_minutes=("Train_seconds", lambda s: s.mean() / 60),
    SD_train_minutes=("Train_seconds", lambda s: s.std(ddof=1) / 60), Mean_final_epochs=("Epochs_final", "mean"),
    Mean_inference_ms_per_sample=("Inference_seconds", "mean"), Peak_GPU_MB=("Peak_GPU_MB", "max"),
    Device=("Device", "first")).reset_index()
n_test = TABLE14.Dataset.map(lambda d: 3 * N_TEST_PER_SILO if d == "Combined" else N_TEST_PER_SILO)
TABLE14["Mean_inference_ms_per_sample"] = TABLE14["Mean_inference_ms_per_sample"] * 1000 / n_test
show_table(TABLE14.round(3), "TABLE 14 — Training time and computational performance (train time includes phase A + B)")
save_df(TABLE14, "tables", "Table14_compute.csv")

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
UNITS = {"Accuracy": "Accuracy (fraction)", "Micro_F1": "Micro F1-score (fraction)",
         "MSE": "MSE (stage units²)", "RMSE": "RMSE (stage units)"}

def grouped_bar(metric, fname, title):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    w = 0.8 / len(ARCHS); x = np.arange(len(DATASETS))
    for i, arch in enumerate(ARCHS):
        s = STATS[(STATS.Metric == metric) & (STATS.Architecture == arch)].set_index("Dataset").loc[DATASETS]
        ax.bar(x + i * w - 0.4 + w / 2, s.Mean, w, yerr=s.SD_ddof1, capsize=3, label=arch)
    ax.set_xticks(x); ax.set_xticklabels(DATASETS); ax.set_xlabel("Dataset"); ax.set_ylabel(UNITS[metric])
    ax.set_title(title + "\n(bars: mean of 3 trials; error bars: sample SD, ddof=1)")
    ax.legend(title="Architecture", fontsize=8); ax.grid(axis="y", alpha=.3)
    save_fig(fig, fname); plt.show()

grouped_bar("Accuracy", "Fig01_accuracy_by_architecture", "Overall accuracy by architecture")
grouped_bar("Micro_F1", "Fig02_microF1_by_architecture", "Micro F1-score by architecture (equals accuracy for single-label multiclass)")
grouped_bar("MSE", "Fig03_MSE_by_architecture", "Stage-label MSE by architecture")
grouped_bar("RMSE", "Fig04_RMSE_by_architecture", "Stage-label RMSE by architecture")

fig, axes = plt.subplots(1, len(DATASETS), figsize=(16, 4), sharey=True)
for ax, ds in zip(axes, DATASETS):
    for arch in ARCHS:
        g = RES_DF[(RES_DF.Dataset == ds) & (RES_DF.Architecture == arch)].set_index("Trial").loc[TRIALS]
        ax.plot(TRIALS, g.Accuracy, marker="o", label=arch)
    ax.set_title(ds); ax.set_xlabel("Trial (seed 42 / 123 / 2026)"); ax.grid(alpha=.3)
axes[0].set_ylabel(UNITS["Accuracy"]); axes[-1].legend(fontsize=8)
fig.suptitle("Trial-wise accuracy variation"); save_fig(fig, "Fig05_trialwise_variation"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, met in zip(axes, ("Accuracy", "RMSE")):
    for arch in ARCHS:
        s = STATS[(STATS.Metric == met) & (STATS.Architecture == arch)].set_index("Dataset").loc[SILOS]
        ax.errorbar(SILOS, s.Mean, yerr=s.SD_ddof1, marker="o", capsize=3, label=arch)
    ax.set_xlabel("Silo (separately trained models)"); ax.set_ylabel(UNITS[met]); ax.grid(alpha=.3)
axes[0].legend(fontsize=8); fig.suptitle("Silo-wise overall performance (mean ± SD)")
save_fig(fig, "Fig06_silowise_comparison"); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, met in zip(axes, ("Accuracy", "MSE", "RMSE")):
    s = STATS[(STATS.Metric == met) & (STATS.Dataset == "Combined")].set_index("Architecture").loc[ARCHS]
    ax.bar(ARCHS, s.Mean, yerr=s.SD_ddof1, capsize=4, color=[f"C{i}" for i in range(len(ARCHS))])
    ax.set_ylabel(UNITS[met]); ax.set_xlabel("Architecture"); ax.tick_params(axis="x", rotation=20); ax.grid(axis="y", alpha=.3)
fig.suptitle(f"Combined dataset ({3 * N_TRAIN_PER_SILO} train / {3 * N_TEST_PER_SILO} test): mean ± SD over three trials")
save_fig(fig, "Fig07_combined_comparison"); plt.show()

fig, axes = plt.subplots(len(DATASETS), len(ARCHS), figsize=(4 * len(ARCHS) + 2, 12), sharex=True, squeeze=False)
for i, ds in enumerate(DATASETS):
    for j, arch in enumerate(ARCHS):
        ax = axes[i, j]
        p = os.path.join(DIRS["trials"], ds, arch, f"Trial-1_seed{SEEDS['Trial-1']}", f"history_{MAX_EPOCHS}_epochs.csv")
        if os.path.exists(p):
            h = pd.read_csv(p)
            ax.plot(h.epoch, h.train_loss, label="training"); ax.plot(h.epoch, h.test_loss, label="testing (monitor only)")
        ax.set_title(f"{ds} / {arch}", fontsize=9); ax.grid(alpha=.3)
        if j == 0: ax.set_ylabel("Cross-entropy loss")
        if i == len(DATASETS) - 1: ax.set_xlabel("Epoch")
axes[0, 0].legend(fontsize=8)
fig.suptitle(f"Training and testing loss (Trial-1, {MAX_EPOCHS} epochs; testing loss logged for monitoring only)")
save_fig(fig, "Fig08_loss_curves"); plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
for arch in ARCHS:
    p = os.path.join(DIRS["tuning"], f"{arch}_tuning_trials.csv")
    if os.path.exists(p):
        t = pd.read_csv(p); t = t[t.state == "COMPLETE"].sort_values("number")
        ax.plot(t.number, t.value, "o", alpha=.35); ax.plot(t.number, t.value.cummin(), "-", label=f"{arch} (best so far)")
ax.set_xlabel("Optuna trial number"); ax.set_ylabel("Validation cross-entropy loss"); ax.legend(fontsize=8); ax.grid(alpha=.3)
ax.set_title("Hyperparameter tuning progress (points: completed trials; lines: running minimum)")
save_fig(fig, "Fig09_tuning_progress"); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, met in zip(axes.ravel(), ("Accuracy", "Micro_F1", "MSE", "RMSE")):
    x = np.arange(len(ARCHS))
    for k, ds in enumerate(DATASETS):
        s = STATS[(STATS.Metric == met) & (STATS.Dataset == ds)].set_index("Architecture").loc[ARCHS]
        ax.errorbar(x + (k - 1.5) * 0.08, s.Mean, yerr=s.SD_ddof1, fmt="o", capsize=3, label=ds)
    ax.set_xticks(x); ax.set_xticklabels(ARCHS, rotation=15); ax.set_ylabel(UNITS[met]); ax.grid(alpha=.3)
axes[0, 0].legend(fontsize=8); fig.suptitle("Mean performance with sample-SD error bars (3 trials)")
save_fig(fig, "Fig10_mean_sd_errorbars"); plt.show()

In [ ]:
RV_DIR = os.path.join(RUN_DIR, "14_reviewer5_results")
os.makedirs(RV_DIR, exist_ok=True)
DIRS["reviewer"] = RV_DIR

def check(name, ok, detail):
    tag = "PASS" if ok else "WARN"
    log(f"  [{tag}] {name}: {detail}", "INFO" if ok else "WARN")

def describe_file(path, sensor_names):
    if not os.path.exists(path):
        return None
    hdr = [str(c).strip() for c in pd.read_csv(path, nrows=0).columns]
    date = next((c for c in hdr if re.search(r"date|time|day", c, re.I)), None)
    ctx = {}
    for c in hdr:
        if c in sensor_names or c == date:
            continue
        if re.search(r"humid|rh", c, re.I):
            ctx["relative_humidity"] = c
        elif re.search(r"amb|outside|outdoor|atmos|environment|external", c, re.I):
            ctx["ambient_temperature"] = c
    return date, sum(c in sensor_names for c in hdr), ctx

SENSOR_NAMES = set(META[SILOS[0]]["sensor_cols"])
print("File / column detection")
for silo in SILOS:
    for fname in (RAW_FILES[silo], OLD_LABEL_FILES[silo]):
        r = describe_file(os.path.join(DATA_DIR, fname), SENSOR_NAMES)
        if r is None:
            print(f"  {fname:>20s}: not present (old label files are optional, audit only)")
            continue
        print(f"  {fname:>20s}: date={r[0]!r} | sensors={r[1]} | context={r[2] if r[2] else '-'}")

m0 = META[SILOS[0]]
n_imp = int(IMPUTED_MASK[SILOS[0]].sum())
n_eval = N_SENSORS - n_imp if EXCLUDE_IMPUTED else N_SENSORS
print(f"\nNumber of sensor positions (classification targets) = {N_SENSORS}  (graph positions: {N_CABLES} cables x {N_LAYERS} layers)")
print(f"Number of cables = {N_CABLES}")
print(f"Number of layers = {N_LAYERS}")
print(f"Number of environmental/context variables = 2 [{m0['amb_t']!r}, {m0['amb_rh']!r}]")
print(f"Total graph nodes in this model = {N_NODES}  (266 sensor positions + ambient-temperature node + ambient-RH node)")
print("Context variables ARE graph nodes in this design (research specification); they are inputs only and never classified.")
print(f"Positions identified as imputed (no physical sensor) = {n_imp} -> kept as graph input nodes; "
      + ("excluded from loss and evaluation" if EXCLUDE_IMPUTED else "INCLUDED in evaluation") + f"; evaluated positions = {n_eval}")
if os.path.exists(os.path.join(DATA_DIR, OLD_LABEL_FILES[SILOS[0]])):
    print("NOTE: context-variable columns inside the old label files are ignored (context variables have no stage).")
ok266 = all(len(META[s]["sensor_cols"]) == N_SENSORS for s in SILOS)
check("266 sensor detection", ok266 and m0["grid_source"] == "parsed_from_column_names",
      f"{N_CABLES} x {N_LAYERS} = {N_SENSORS} sensor columns in every raw file (grid: {m0['grid_source']}); "
      f"context = [{m0['amb_t']!r}, {m0['amb_rh']!r}]")

In [ ]:
OLD_ALIGNED, r2_rows, r2_mism = {}, [], []
for silo in SILOS:
    p = os.path.join(DATA_DIR, OLD_LABEL_FILES[silo])
    if not os.path.exists(p):
        print(f"{silo}: old label file not present — skipped"); continue
    old = read_csv_clean(p)
    dcol = next((c for c in old.columns if re.search(r"date|time|day", c, re.I)), None)
    sc = META[silo]["sensor_cols"]
    if dcol is None or any(c not in old.columns for c in sc):
        print(f"{silo}: old label file has no date column or different sensor names — skipped"); continue
    od, _ = parse_dates(old[dcol], silo + "-old-labels")
    O = pd.DataFrame({c: normalize_old_labels(old[c]).values for c in sc})
    O["Date"] = od.values
    O = O.drop_duplicates("Date").set_index("Date")
    C = CLEAN[silo].set_index("Date")
    common = C.index.intersection(O.index)
    T = C.loc[common, sc].to_numpy(float); S = O.loc[common, sc].to_numpy(float)
    Fr = label_temperature(T)
    tp, sp = np.isfinite(T), np.isfinite(S)
    both = tp & sp
    mm = both & (S != Fr)
    OLD_ALIGNED[silo] = S
    r2_rows.append(OrderedDict([("Silo", silo), ("Dates aligned", f"{len(common)}/{len(C)}"),
                                ("Checked", int(both.sum())), ("Matching", int((both & (S == Fr)).sum())),
                                ("Mismatching", int(mm.sum())), ("Mismatch %", round(100 * mm.sum() / max(both.sum(), 1), 4)),
                                ("Label values", sorted(np.unique(S[sp]).tolist())),
                                ("Label missing where temperature present", int((tp & ~sp).sum())),
                                ("Label present where temperature missing", int((~tp & sp).sum()))]))
    r, c = np.where(mm)
    ci, li = META[silo]["cable_idx"], META[silo]["layer_idx"]
    r2_mism.append(pd.DataFrame({"silo": silo, "date": common[r].strftime("%Y-%m-%d"), "sensor": np.array(sc)[c],
                                 "cable": (ci[c] + 1) if ci is not None else np.nan,
                                 "layer": (li[c] + 1) if li is not None else np.nan,
                                 "raw_temperature": T[r, c], "supplied_class": S[r, c].astype(int),
                                 "threshold_class": Fr[r, c]}))
if r2_rows:
    R2 = pd.DataFrame(r2_rows)
    check("Original-label alignment", all(r["Dates aligned"].split("/")[0] == r["Dates aligned"].split("/")[1] for r in r2_rows),
          "input and label rows aligned by date in every silo")
    show_table(R2, "Old (supplied) labels vs fresh threshold labels")
    save_df(R2, "reviewer", "R2_old_label_alignment.csv")
    M = pd.concat(r2_mism, ignore_index=True)
    save_df(M, "reviewer", "R2_all_mismatches.csv")
    print(f"\nSample mismatches (first 20 of {len(M)}):")
    display(M.head(20))
    if len(M):
        pat = M.groupby(["raw_temperature", "supplied_class", "threshold_class"]).size().rename("count").reset_index()
        print("\nMismatch pattern (temperature, supplied, threshold):"); display(pat)
        save_df(pat, "reviewer", "R2_mismatch_pattern.csv")
        if M.raw_temperature.isin(STAGE_THRESHOLDS).all() and (M.supplied_class == M.threshold_class - 1).all():
            diag = ("every mismatch is at a threshold value exactly (37, 42 or 45 degC) and the supplied label is one stage "
                    "lower -> the old label files treat each upper boundary as INCLUSIVE (e.g. 45.0 -> Stage 2), whereas the "
                    "stated rule makes each lower boundary inclusive (45.0 -> Stage 3). Fresh labels follow the stated rule.")
        else:
            diag = "mismatches are not limited to boundary values; inspect R2_all_mismatches.csv."
        print("DIAGNOSIS:", diag)
        save_json({"diagnosis": diag, "n_mismatches": len(M)}, "reviewer", "R2_diagnosis.json")
    allvals = set(v for r in r2_rows for v in r["Label values"])
    check("Label domain", allvals <= {0.0, 1.0, 2.0, 3.0}, f"all supplied labels in {sorted(allvals)}")
    check("Label/threshold agreement", len(M) == 0,
          f"{len(M)} mismatches; fresh threshold labels are used for all modelling (old labels: audit only)")

In [ ]:
pr = prep_df.set_index("silo")
dq = {}
for silo in SILOS:
    c = CLEAN[silo]; m = META[silo]
    T = c[m["sensor_cols"]].to_numpy(float); ctx = c[[m["amb_t"], m["amb_rh"]]].to_numpy(float)
    d = c["Date"].reset_index(drop=True); steps = d.diff().dt.days.iloc[1:].to_numpy()
    has = np.isfinite(T).any(1)
    run = 1 if has[0] else 0
    for k in range(1, len(d)):
        if run and steps[k - 1] == 1 and has[k]:
            run += 1
        else:
            break
    Tf = T[np.isfinite(T)]
    dq[silo] = OrderedDict([
        ("Days in file (raw rows)", len(RAW[silo])), ("Days after cleaning", len(c)),
        ("First date", str(d.iloc[0].date())), ("Last date", str(d.iloc[-1].date())),
        ("Dates increasing (raw file order)", bool(pr.loc[silo, "input_chronologically_sorted"])),
        ("Duplicate dates (raw rows)", int(pr.loc[silo, "rows_with_duplicate_dates"])),
        ("Gaps (step != 1 day)", int((steps != 1).sum())),
        ("Missing sensor cells", int(np.isnan(T).sum())), ("Missing %", round(100 * float(np.isnan(T).mean()), 4)),
        ("Rows with all sensors missing", int((~has).sum())),
        ("Last day with sensor data", str(d[has].iloc[-1].date()) if has.any() else "-"),
        ("Usable consecutive days from start", run),
        ("Missing context cells", int(np.isnan(ctx).sum())),
        ("Inf values", int(np.isinf(T).sum() + np.isinf(ctx).sum())),
        ("Sensor min (degC)", float(Tf.min())), ("Sensor max (degC)", float(Tf.max()))])
R3 = pd.DataFrame(dq)
show_table(R3, "Data-quality summary per silo")
save_df(R3, "reviewer", "R3_data_quality.csv", index=True)
check("Date ordering", all(CLEAN[s]["Date"].is_monotonic_increasing for s in SILOS), "dates strictly increasing after sorting")
check("Duplicate dates", int(R3.loc["Duplicate dates (raw rows)"].sum()) == 0, f"{int(R3.loc['Duplicate dates (raw rows)'].sum())} duplicate rows in raw files")
check("Inf values", int(R3.loc["Inf values"].sum()) == 0, f"{int(R3.loc['Inf values'].sum())} Inf cells")
check("Missing values", int(R3.loc["Missing sensor cells"].sum() + R3.loc["Missing context cells"].sum()) == 0,
      f"{int(R3.loc['Missing sensor cells'].sum())} missing sensor cells, {int(R3.loc['Missing context cells'].sum())} missing context cells")

res_audit = pd.DataFrame({"imputed": np.concatenate([IMPUTED_MASK[s] for s in SILOS]),
                          "offgrid_fraction": np.concatenate([OFFGRID[s] for s in SILOS])})
ra = res_audit.groupby("imputed")["offgrid_fraction"].agg(["count", "mean", "min", "max"]).round(3)
print(f"\nResolution-signature audit (fraction of values NOT on the {RESOLUTION_STEP_C} degC grid; positions x silos):")
display(ra)
save_df(ra, "reviewer", "R3_resolution_signature.csv", index=True)
if True in ra.index and False in ra.index:
    check("Imputed-position identification", ra.loc[False, "max"] < ra.loc[True, "min"],
          "imputed positions are cleanly separated from physical sensors by the 0.5 degC signature"
          if ra.loc[False, "max"] < ra.loc[True, "min"] else "signatures overlap; review imputed_positions_by_sensor.csv")

In [ ]:
def class_row(source, scope, silo, L, cols):
    v = L[:, cols]; v = v[np.isfinite(v)] if v.dtype.kind == "f" else v[v >= 0]
    v = v.astype(int); cnt = np.bincount(v, minlength=N_CLASSES)[:N_CLASSES]; n = int(cnt.sum())
    row = OrderedDict(Source=source, Scope=scope, Silo=silo, Labelled_cells=n)
    for k in range(N_CLASSES): row[f"Class {k}"] = int(cnt[k])
    for k in range(N_CLASSES): row[f"Class {k} %"] = round(100 * cnt[k] / max(n, 1), 4)
    return row

sources = [("threshold-derived (fresh)", {s: LABELS[s][META[s]["sensor_cols"]].to_numpy() for s in SILOS})]
if len(OLD_ALIGNED) == len(SILOS):
    sources.insert(0, ("supplied labels (old)", OLD_ALIGNED))
r4 = []
for src, Ls in sources:
    for scope in ("all", "eval"):
        rows = []
        for s in SILOS:
            cols = np.ones(N_SENSORS, bool) if scope == "all" else ~IMPUTED_MASK[s]
            lab = f"all {N_SENSORS} positions" if scope == "all" else f"evaluated positions ({int(cols.sum())})"
            rows.append(class_row(src, lab, s, Ls[s], cols))
        comb = OrderedDict(Source=src, Scope=rows[0]["Scope"], Silo="Combined",
                           Labelled_cells=sum(r["Labelled_cells"] for r in rows))
        for k in range(N_CLASSES): comb[f"Class {k}"] = sum(r[f"Class {k}"] for r in rows)
        for k in range(N_CLASSES): comb[f"Class {k} %"] = round(100 * comb[f"Class {k}"] / max(comb["Labelled_cells"], 1), 4)
        r4 += rows + [comb]
R4 = pd.DataFrame(r4)
show_table(R4, "Class distribution (data verification only — not a performance result)")
save_df(R4, "reviewer", "R4_class_distribution.csv")
ev = R4[R4.Scope.str.startswith("evaluated")]
check("Class availability", (ev[[f"Class {k}" for k in range(N_CLASSES)]] > 0).all().all(),
      "all four stages present in every silo (evaluated positions)")

fig, axes = plt.subplots(1, len(sources), figsize=(9 * len(sources), 4.6), squeeze=False)
for ax, (src, _) in zip(axes[0], sources):
    sub = ev[ev.Source == src]
    x = np.arange(N_CLASSES); w = 0.2
    for i, s in enumerate(list(SILOS) + ["Combined"]):
        vals = sub[sub.Silo == s][[f"Class {k}" for k in range(N_CLASSES)]].to_numpy().ravel()
        bars = ax.bar(x + (i - 1.5) * w, vals, w, label=s)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, max(v, 1), f"{int(v)}", ha="center", va="bottom", fontsize=7, rotation=90)
    ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(["Stage 0 Safe", "Stage 1 Initial", "Stage 2 Active", "Stage 3 Severe"])
    ax.set_ylabel("Labelled cells (log scale)"); ax.set_xlabel("Stage")
    ax.set_title(f"Class distribution, {src}, {sub.Scope.iloc[0]}"); ax.legend(fontsize=8)
fig.tight_layout(); save_fig(fig, "R5_class_distribution_log"); plt.show()

In [ ]:
As = ADJ[:N_SENSORS, :N_SENSORS].astype(int)
Asym = np.maximum(As, As.T)
deg = Asym.sum(1)
parent = list(range(N_SENSORS))
def _f(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
for a, b in zip(*np.nonzero(Asym)):
    ra_, rb_ = _f(int(a)), _f(int(b))
    if ra_ != rb_: parent[ra_] = rb_
R5 = OrderedDict([
    ("source", graph_rep["source"]),
    ("sensor nodes", N_SENSORS),
    ("sensor-sensor directed_edges", int(As.sum())),
    ("sensor-sensor undirected_edges", int(np.triu(Asym, 1).sum())),
    ("sensor edge_index_shape", [2, int(As.sum())]),
    ("degree_min", int(deg.min())), ("degree_mean", round(float(deg.mean()), 3)),
    ("degree_median", float(np.median(deg))), ("degree_max", int(deg.max())),
    ("isolated_nodes (sensor subgraph)", int((deg == 0).sum())),
    ("connected_components (sensor subgraph)", len({_f(i) for i in range(N_SENSORS)})),
    ("symmetric", bool((As == As.T).all())),
    ("edge_index_valid", bool(EDGE_INDEX_NP.min() >= 0 and EDGE_INDEX_NP.max() < N_NODES)),
    ("model graph nodes (incl. 2 ambient)", N_NODES), ("model graph directed edges", int(EDGE_INDEX_NP.shape[1])),
    ("model graph connected_components", graph_rep["connected_components"])])
for k, v in R5.items():
    print(f"{k:>42s}: {v}")
save_json(R5, "reviewer", "R5_graph_report.json")
check("Graph edge validity", R5["edge_index_valid"], f"{R5['sensor-sensor undirected_edges']} undirected sensor edges, symmetric={R5['symmetric']}")
check("Isolated nodes", R5["isolated_nodes (sensor subgraph)"] == 0,
      f"{R5['isolated_nodes (sensor subgraph)']} isolated; {R5['connected_components (sensor subgraph)']} connected component(s) in the sensor subgraph")

ci, li = m0["cable_idx"], m0["layer_idx"]
if ci is not None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
    grid = np.full((N_LAYERS, N_CABLES), np.nan); grid[li, ci] = deg
    im = axes[0].imshow(grid, origin="lower", aspect="auto", cmap="viridis",
                        extent=(0.5, N_CABLES + 0.5, 0.5, N_LAYERS + 0.5))
    imp = IMPUTED_MASK[SILOS[0]]
    axes[0].scatter(ci[imp] + 1, li[imp] + 1, marker="x", color="white", s=18)
    axes[0].set_xticks(range(1, N_CABLES + 1)); axes[0].set_yticks(range(1, N_LAYERS + 1))
    axes[0].tick_params(labelsize=7); axes[0].set_xlabel("Cable"); axes[0].set_ylabel("Layer")
    axes[0].set_title("Node degree by cable and layer (x = imputed position, not evaluated)")
    fig.colorbar(im, ax=axes[0], label="degree (sensor-sensor edges)")
    order = np.lexsort((ci, li))
    axes[1].spy(Asym[np.ix_(order, order)], markersize=0.6, color="black")
    axes[1].set_title(f"Adjacency matrix ({N_SENSORS} x {N_SENSORS}, layer-major node order)")
    axes[1].set_xlabel("node"); axes[1].set_ylabel("node"); axes[1].xaxis.set_ticks_position("bottom")
    fig.suptitle(f"Graph: {graph_rep['source'][:90]}", fontsize=9)
    fig.tight_layout(); save_fig(fig, "R7_graph_degree_and_adjacency"); plt.show()

In [ ]:
GG = "GAT_GRU"
TRIAL_NAMES = list(SEEDS)
HZ = [f"Day{INPUT_DAYS + 1 + h}" for h in range(HORIZONS)]
PCT = ["Accuracy", "Micro_Precision", "Micro_Recall", "Micro_F1"]

def trial_metrics(ds, trial, seed):
    rdir = os.path.join(DIRS["trials"], ds, GG, f"{trial}_seed{seed}")
    p = os.path.join(rdir, "metrics.json")
    if not os.path.exists(p):
        return None
    with open(p) as f:
        r = json.load(f)
    if "Train_Accuracy" not in r:
        ck = torch.load(os.path.join(rdir, "model.pt"), map_location=DEVICE)
        mdl = build_model(GG, ck["hp"]).to(DEVICE); mdl.load_state_dict(ck["state_dict"])
        dF = get_arrays(ds, "final"); crit = nn.CrossEntropyLoss(ignore_index=MISSING_LABEL)
        for split, X, Y in (("Train", dF["X_tr"], dF["Y_tr"]), ("Test", dF["X_ev"], dF["Y_ev"])):
            e = evaluate(mdl, to_dev(X, torch.float32), to_dev(Y, torch.long), crit)
            if split == "Train":
                r.update({f"Train_{k}": e["metrics"][k] for k in METRIC_COLS})
            for h in range(HORIZONS):
                mk = Y[:, h] >= 0
                r[f"{split}_Acc_{HZ[h]}"] = float((e["preds"][:, h][mk] == Y[:, h][mk]).mean())
        del mdl
    return r

rv_rows, missing = [], []
for ds in DATASETS:
    for trial, seed in SEEDS.items():
        r = trial_metrics(ds, trial, seed)
        if r is None:
            missing.append(f"{ds}/{trial}"); continue
        for split, pre in (("Training", "Train_"), ("Testing", "")):
            row = OrderedDict(Dataset=ds, Trial=trial, Seed=seed, Split=split)
            row.update({k: r[pre + k] for k in METRIC_COLS})
            hpre = "Train_Acc_" if split == "Training" else "Test_Acc_"
            row.update({f"Acc_{h}": r.get(hpre + h, np.nan) for h in HZ})
            rv_rows.append(row)
if missing:
    raise RuntimeError(f"GAT-GRU trial outputs missing for {missing}. Run Cell 15 first (ARCHS must include 'GAT_GRU').")
RV = pd.DataFrame(rv_rows)
save_df(RV, "reviewer", "R6_gatgru_trial_metrics_train_test.csv")

DS_COLORS = {"Silo1": "#1f77b4", "Silo2": "#ff7f0e", "Silo3": "#2ca02c", "Combined": "#d62728"}
TRIAL_ALPHA = [0.55, 0.78, 1.0]
TRIAL_HATCH = ["", "//", ".."]

summ = []
for (ds, split), g in RV.groupby(["Dataset", "Split"], sort=False):
    row = OrderedDict(Dataset=ds, Split=split)
    for k in PCT + ["MSE", "RMSE"] + [f"Acc_{h}" for h in HZ]:
        sc_ = 100 if (k in PCT or k.startswith("Acc_")) else 1
        row[k] = f"{sc_ * g[k].mean():.2f} ± {sc_ * g[k].std(ddof=1):.2f}"
    summ.append(row)
RV_SUMMARY = pd.DataFrame(summ)
show_table(RV_SUMMARY, "GAT-GRU: mean ± SD (ddof=1) over 3 trials, % for classification metrics")
save_df(RV_SUMMARY, "reviewer", "R6_gatgru_summary_mean_sd.csv")

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
LABEL = {"Accuracy": "Accuracy", "Micro_F1": "Micro F1", "Micro_Precision": "Micro precision", "Micro_Recall": "Micro recall"}

def trial_bars(ax, metric, split):
    x = np.arange(len(DATASETS)); w = 0.22
    sub = RV[RV.Split == split]
    allv = []
    for i, ds in enumerate(DATASETS):
        for t, trial in enumerate(TRIAL_NAMES):
            v = 100 * sub[(sub.Dataset == ds) & (sub.Trial == trial)][metric].iloc[0]
            allv.append(v)
            ax.bar(x[i] + (t - 1) * w, v, w, color=DS_COLORS[ds], alpha=TRIAL_ALPHA[t],
                   hatch=TRIAL_HATCH[t], edgecolor="black", linewidth=0.5)
            ax.text(x[i] + (t - 1) * w, v, f"{v:.2f}", ha="center", va="bottom", fontsize=6.5, rotation=90)
        g = 100 * sub[sub.Dataset == ds][metric]
        ax.errorbar(x[i] + 2 * w, g.mean(), yerr=g.std(ddof=1), fmt="D", color="black", capsize=4)
        allv += [g.mean() - g.std(ddof=1), g.mean() + g.std(ddof=1)]
    lo = min(allv) - 1.0; hi = min(max(allv) + 1.5, 101.5)
    ax.set_ylim(lo, hi); ax.set_xticks(x); ax.set_xticklabels(DATASETS)
    ax.set_xlabel("Dataset"); ax.set_ylabel(f"{LABEL[metric]} (%)"); ax.grid(axis="y", alpha=0.3)
    ax.set_title(f"{LABEL[metric]} — {split} (pooled Day16–18)")

def legend_below(fig, y=-0.02):
    h = [Patch(facecolor=DS_COLORS[d], edgecolor="black", label=d) for d in DATASETS]
    h += [Patch(facecolor="lightgrey", alpha=TRIAL_ALPHA[t], hatch=TRIAL_HATCH[t], edgecolor="black",
                label=TRIAL_NAMES[t].replace("-", " ")) for t in range(len(TRIAL_NAMES))]
    h += [Line2D([0], [0], marker="D", color="black", linestyle="none", label="Mean ± SD")]
    fig.legend(handles=h, loc="lower center", ncol=len(h), fontsize=9, bbox_to_anchor=(0.5, y))

for metric, fname in (("Accuracy", "R8_accuracy_train_test"), ("Micro_F1", "R9_microF1_train_test")):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    trial_bars(axes[0], metric, "Training"); trial_bars(axes[1], metric, "Testing")
    fig.suptitle(f"GAT-GRU: silo-wise and combined {LABEL[metric].lower()} across three trials (measured)")
    fig.tight_layout(rect=(0, 0.06, 1, 1)); legend_below(fig); save_fig(fig, fname); plt.show()

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for i, metric in enumerate(("Micro_Precision", "Micro_Recall")):
    trial_bars(axes[i, 0], metric, "Training"); trial_bars(axes[i, 1], metric, "Testing")
fig.suptitle("GAT-GRU: micro precision and recall across three trials (pooled, measured)")
fig.tight_layout(rect=(0, 0.04, 1, 1)); legend_below(fig); save_fig(fig, "R10_precision_recall_train_test"); plt.show()

fig, axes = plt.subplots(4, 2, figsize=(15, 18))
for i, metric in enumerate(("Micro_Precision", "Micro_Recall", "Micro_F1", "Accuracy")):
    trial_bars(axes[i, 0], metric, "Training"); trial_bars(axes[i, 1], metric, "Testing")
fig.suptitle("GAT-GRU: precision, recall, F1-score and accuracy across three trials (pooled)")
fig.tight_layout(rect=(0, 0.02, 1, 0.99)); legend_below(fig, y=0.0); save_fig(fig, "R11_all_metrics_train_test"); plt.show()
print("Note: for single-label multiclass classification micro precision = micro recall = micro F1 = accuracy, "
      "so these panels are identical by definition.")

In [ ]:
c = RV[RV.Dataset == "Combined"]
fig, ax = plt.subplots(figsize=(8, 5))
xt = [f"{t.replace('-', ' ')}\n(seed {SEEDS[t]})" for t in TRIAL_NAMES]
for split, mk in (("Training", "o"), ("Testing", "s")):
    v = [100 * c[(c.Split == split) & (c.Trial == t)]["Accuracy"].iloc[0] for t in TRIAL_NAMES]
    ax.plot(xt, v, marker=mk, label=split, color=("#555555" if split == "Training" else DS_COLORS["Combined"]))
    for xi, vi in zip(xt, v):
        ax.annotate(f"{vi:.2f}", (xi, vi), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=9)
ax.set_ylabel("Accuracy (%)"); ax.set_xlabel("Trial"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("GAT-GRU, Combined: training vs testing accuracy (measured)")
save_fig(fig, "R12_combined_train_vs_test"); plt.show()

te = RV[RV.Split == "Testing"]
fig, ax = plt.subplots(figsize=(9, 5))
for ds in DATASETS:
    g = te[te.Dataset == ds]
    ax.errorbar(HZ, [100 * g[f"Acc_{h}"].mean() for h in HZ], yerr=[100 * g[f"Acc_{h}"].std(ddof=1) for h in HZ],
                marker="o", capsize=4, label=ds, color=DS_COLORS[ds])
ax.set_xlabel("Forecast horizon"); ax.set_ylabel("Test accuracy (%)"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("GAT-GRU: test accuracy per forecast horizon (mean ± SD of 3 trials)")
save_fig(fig, "R13_test_accuracy_per_horizon"); plt.show()

hz_tab = te.groupby("Dataset", sort=False)[[f"Acc_{h}" for h in HZ]].agg(["mean", lambda s: s.std(ddof=1)])
hz_tab.columns = [f"{a}_{'mean' if b == 'mean' else 'SD'}" for a, b in hz_tab.columns]
show_table((100 * hz_tab).round(3), "GAT-GRU test accuracy (%) per horizon, mean and SD (ddof=1)")
save_df(hz_tab.reset_index(), "reviewer", "R13_test_accuracy_per_horizon.csv")
log(f"Reviewer-5 GAT-GRU results saved in {RV_DIR}")

In [ ]:
te = RV[RV.Split == "Testing"]
fig, ax = plt.subplots(figsize=(10, 5.2))
metrics = ["MSE", "RMSE"]; x = np.arange(len(metrics)); w = 0.8 / len(DATASETS)
rows_mr = []
for i, ds in enumerate(DATASETS):
    g = te[te.Dataset == ds]
    means = [g[m].mean() for m in metrics]; sds = [g[m].std(ddof=1) for m in metrics]
    pos = x - 0.4 + w / 2 + i * w
    ax.bar(pos, means, w, yerr=sds, capsize=4, color=DS_COLORS[ds], edgecolor="black", linewidth=0.5, label=ds)
    for j, m in enumerate(metrics):
        ax.scatter([pos[j]] * len(g), g[m], color="black", s=12, zorder=3)
        ax.text(pos[j], means[j] + sds[j], f"{means[j]:.4f}", ha="center", va="bottom", fontsize=8, rotation=90)
    rows_mr.append(dict(Dataset=ds, **{f"{m}_trial{k+1}": v for m in metrics for k, v in enumerate(g[m].to_numpy())},
                        **{f"{m}_mean": g[m].mean() for m in metrics}, **{f"{m}_SD": g[m].std(ddof=1) for m in metrics}))
ax.set_xticks(x); ax.set_xticklabels(["MSE (stage units²)", "RMSE (stage units)"])
ax.set_ylabel("Error on integer stage labels 0–3"); ax.set_xlabel("Metric"); ax.grid(axis="y", alpha=0.3)
ax.set_title("GAT-GRU test MSE and RMSE, silo-wise and combined\n(bars: mean of 3 trials; error bars: SD, ddof=1; dots: individual trials)")
ax.legend(title="Dataset")
fig.tight_layout(); save_fig(fig, "R14_MSE_RMSE_comparison"); plt.show()
MR = pd.DataFrame(rows_mr)
show_table(MR.round(6), "GAT-GRU test MSE / RMSE per dataset (three trials, mean, SD)")
save_df(MR, "reviewer", "R14_MSE_RMSE_comparison.csv")

In [ ]:
HIST = {}
for ds in DATASETS:
    for trial, seed in SEEDS.items():
        p = os.path.join(DIRS["trials"], ds, GG, f"{trial}_seed{seed}", f"history_{MAX_EPOCHS}_epochs.csv")
        if not os.path.exists(p):
            raise FileNotFoundError(f"{p} not found — run Cell 15 with STRICT_EPOCHS=True first.")
        HIST[(ds, trial)] = pd.read_csv(p)
n_ep = len(next(iter(HIST.values())))
log(f"Loss histories loaded: {len(HIST)} runs x {n_ep} epochs")

fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharex=True)
for ax, ds in zip(axes.ravel(), DATASETS):
    for t, trial in enumerate(TRIAL_NAMES):
        h = HIST[(ds, trial)]
        ax.plot(h.epoch, h.train_loss, color=DS_COLORS[ds], alpha=TRIAL_ALPHA[t], lw=1.2,
                label=f"{trial.replace('-', ' ')} training")
        ax.plot(h.epoch, h.test_loss, color=DS_COLORS[ds], alpha=TRIAL_ALPHA[t], lw=1.2, ls="--",
                label=f"{trial.replace('-', ' ')} testing")
    ax.set_title(ds); ax.set_ylabel("Cross-entropy loss"); ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)
for ax in axes[-1]:
    ax.set_xlabel("Epoch")
fig.suptitle(f"GAT-GRU training and testing loss over {n_ep} epochs (testing loss logged for monitoring only)")
fig.tight_layout(); save_fig(fig, "R15_loss_curves_per_dataset"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, col, title in zip(axes, ("train_loss", "test_loss"), ("Training loss", "Testing loss (monitor only)")):
    for ds in DATASETS:
        L = np.stack([HIST[(ds, t)][col].to_numpy() for t in TRIAL_NAMES])
        mu, sd = L.mean(0), L.std(0, ddof=1); ep = HIST[(ds, TRIAL_NAMES[0])].epoch
        ax.plot(ep, mu, color=DS_COLORS[ds], lw=1.4, label=ds)
        ax.fill_between(ep, mu - sd, mu + sd, color=DS_COLORS[ds], alpha=0.15)
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.grid(alpha=0.3); ax.legend(title="Dataset")
axes[0].set_ylabel("Cross-entropy loss (mean of 3 trials ± SD)")
fig.suptitle(f"GAT-GRU loss curves, silo-wise and combined ({n_ep} epochs)")
fig.tight_layout(); save_fig(fig, "R16_loss_curves_all_datasets"); plt.show()

last = pd.DataFrame([dict(Dataset=ds, Trial=t, Epochs=len(HIST[(ds, t)]),
                          Final_train_loss=HIST[(ds, t)].train_loss.iloc[-1], Final_test_loss=HIST[(ds, t)].test_loss.iloc[-1],
                          Min_train_loss=HIST[(ds, t)].train_loss.min(), Mean_seconds_per_epoch=HIST[(ds, t)].epoch_time_s.mean())
                     for ds in DATASETS for t in TRIAL_NAMES])
show_table(last.round(5), "Loss at the final epoch (per dataset and trial)")
save_df(last, "reviewer", "R16_final_epoch_losses.csv")

In [ ]:
EXT_FIRST_DAY, EXT_LAST_DAY = INPUT_DAYS + 1, 50
EXT_H = EXT_LAST_DAY - INPUT_DAYS
EXT_DAYS = list(range(EXT_FIRST_DAY, EXT_LAST_DAY + 1))
EXT_DAY_NAMES = [f"Day{d}" for d in EXT_DAYS]
assert EXT_H >= HORIZONS

DIRS["extended"] = os.path.join(RUN_DIR, "15_extended_day16_50")
os.makedirs(DIRS["extended"], exist_ok=True)
EXT_TRIAL_DIR = os.path.join(DIRS["extended"], "trials")
os.makedirs(EXT_TRIAL_DIR, exist_ok=True)

FIRST_TEST_TARGET_ROW = {}
for silo in SILOS:
    te = SELECTION[silo][SELECTION[silo].split == "test"]
    FIRST_TEST_TARGET_ROW[silo] = int(te.start_row.min()) + INPUT_DAYS

def ext_targets(wins, role):
    out = np.full((len(wins), EXT_H, N_SENSORS), MISSING_LABEL, dtype=np.int64)
    for k, (s, st) in enumerate(wins):
        A = SILO_ARR[s]; D = len(A["dates"])
        for h in range(EXT_H):
            r = st + INPUT_DAYS + h
            if r >= D:
                break
            if A["day"][r] - A["day"][st] != INPUT_DAYS + h:
                break
            if role == "train" and r >= FIRST_TEST_TARGET_ROW[s]:
                break
            out[k, h] = A["Y"][r]
    return out

EXT = {}
day_rows = []
for ds in DATASETS:
    d = get_arrays(ds, "final")
    Ytr = ext_targets(d["w_tr"], "train"); Yte = ext_targets(d["w_ev"], "test")

    assert np.array_equal(Ytr[:, :HORIZONS], d["Y_tr"]), f"{ds}: training Day16-18 targets differ from main experiment"
    assert np.array_equal(Yte[:, :HORIZONS], d["Y_ev"]), f"{ds}: testing Day16-18 targets differ from main experiment"

    for wins, Y in ((d["w_tr"], Ytr), (d["w_ev"], Yte)):
        for k, (s, st) in enumerate(wins):
            A = SILO_ARR[s]
            for h in np.where((Y[k] >= 0).any(1))[0]:
                exp = label_temperature(A["Xraw"][st + INPUT_DAYS + h, :N_SENSORS])
                v = Y[k, h] >= 0
                assert np.array_equal(exp[v], Y[k, h][v]), f"{ds}: label/temperature mismatch"
    EXT[ds] = dict(X_tr=d["X_tr"], Y_tr=Ytr, X_te=d["X_ev"], Y_te=Yte, w_tr=d["w_tr"], w_te=d["w_ev"])
    np.savez_compressed(versioned_path(os.path.join(DIRS["extended"], f"{ds}_tensors_day16_50.npz")),
                        X_train=d["X_tr"], Y_train=Ytr, X_test=d["X_ev"], Y_test=Yte,
                        day=np.array(EXT_DAYS),
                        train_windows=np.array([f"{s}:{r}" for s, r in d["w_tr"]]),
                        test_windows=np.array([f"{s}:{r}" for s, r in d["w_ev"]]))
    for h, dn in enumerate(EXT_DAY_NAMES):
        row = dict(Dataset=ds, Day=dn)
        for split, Y in (("train", Ytr), ("test", Yte)):
            v = Y[:, h]
            row[f"{split}_windows_with_labels"] = int((v >= 0).any(1).sum())
            row[f"{split}_labelled_cells"] = int((v >= 0).sum())
            cnt = np.bincount(v[v >= 0], minlength=N_CLASSES)
            for c in range(N_CLASSES):
                row[f"{split}_stage{c}"] = int(cnt[c])
        day_rows.append(row)

shape_rows = []
for ds in DATASETS:
    e = EXT[ds]; ntr, nte = len(e["w_tr"]), len(e["w_te"])
    shape_rows.append(dict(Dataset=ds, X_train=str(e["X_tr"].shape), Y_train=str(e["Y_tr"].shape),
                           X_test=str(e["X_te"].shape), Y_test=str(e["Y_te"].shape),
                           Train_logits=str((ntr, EXT_H, N_SENSORS, N_CLASSES)),
                           Test_logits=str((nte, EXT_H, N_SENSORS, N_CLASSES)),
                           Test_pred_labels=str((nte, EXT_H, N_SENSORS)),
                           One_day_slice_test=str((nte, N_SENSORS))))
    exp_n = (3 * N_TRAIN_PER_SILO, 3 * N_TEST_PER_SILO) if ds == "Combined" else (N_TRAIN_PER_SILO, N_TEST_PER_SILO)
    assert (ntr, nte) == exp_n, f"{ds}: sample counts changed {ntr}/{nte}"
    assert e["X_tr"].shape == (ntr, INPUT_DAYS, N_NODES, 1) and e["Y_te"].shape == (nte, EXT_H, N_SENSORS)
EXT_SHAPES = pd.DataFrame(shape_rows)
show_table(EXT_SHAPES, "EXTENDED — input / output tensor structure (Day 16–50)")
save_df(EXT_SHAPES, "extended", "E1_tensor_structure_day16_50.csv")

EXT_DAY_TABLE = pd.DataFrame(day_rows)
save_df(EXT_DAY_TABLE, "extended", "E1_daywise_label_availability.csv")
show_table(EXT_DAY_TABLE[EXT_DAY_TABLE.Dataset == "Combined"][["Day", "train_windows_with_labels", "train_labelled_cells",
                                                                 "test_windows_with_labels", "test_labelled_cells"]],
           "Combined — labelled windows / cells per forecast day (train: leakage-masked; test: until data end)")
empty = EXT_DAY_TABLE[(EXT_DAY_TABLE.test_windows_with_labels == 0) | (EXT_DAY_TABLE.train_windows_with_labels == 0)]
if len(empty):
    log(f"Some days have no labelled windows and will show as n/a: {empty[['Dataset', 'Day']].values.tolist()}", "WARN")
save_json({"days": EXT_DAY_NAMES, "first_test_target_row": FIRST_TEST_TARGET_ROW,
           "masking": ["after last recorded date", "training target on/after first testing-target date",
                       "imputed positions / missing temperatures (as in main experiment)"],
           "inputs_and_split": f"identical to main experiment (same {3 * SAMPLES_PER_SILO} windows, {3 * N_TRAIN_PER_SILO}/{3 * N_TEST_PER_SILO}, {GAP_WINDOWS}-window gap)"},
          "extended", "E1_extension_protocol.json")
log(f"E1 done: Day {EXT_FIRST_DAY}-{EXT_LAST_DAY} targets built for {DATASETS}")

In [ ]:
def build_model(arch, hp):
    assert arch == "GAT_GRU", f"This notebook builds GAT-GRU only (got {arch})."
    return GATGRUStageModel(EDGE_INDEX_T, hidden=hp["hidden"], heads=hp.get("heads", 1),
                            gru_layers=hp.get("gru_layers", 1), dropout=hp["dropout"],
                            horizons=hp.get("horizons", HORIZONS))

if "BEST_HP" not in globals() or not BEST_HP:
    with open(os.path.join(DIRS["tuning"], "best_hyperparameters.json")) as f:
        BEST_HP = json.load(f)
EXT_HP = dict(BEST_HP["GAT_GRU"], horizons=EXT_H)
log(f"Extended model hyperparameters: {EXT_HP}")

m = build_model("GAT_GRU", EXT_HP).to(DEVICE).eval()
with torch.no_grad():
    xb = to_dev(EXT["Combined"]["X_te"][:2], torch.float32)
    lo = m(xb)
assert lo.shape == (2, EXT_H, N_SENSORS, N_CLASSES), lo.shape
print(f"Forward test: input {tuple(xb.shape)} -> logits {tuple(lo.shape)} -> predicted stages {tuple(lo.argmax(-1).shape)}")
print(f"Parameters: {count_params(m):,} (main Day16-18 model: {count_params(build_model('GAT_GRU', BEST_HP['GAT_GRU'])):,})")
del m

EXT_SMOKE_EPOCHS = 2
e = EXT["Combined"]
_m, _h, _, _c = fit_model("GAT_GRU", EXT_HP, 42, to_dev(e["X_tr"], torch.float32), to_dev(e["Y_tr"], torch.long),
                          fixed_epochs=EXT_SMOKE_EPOCHS,
                          monitor=(to_dev(e["X_te"], torch.float32), to_dev(e["Y_te"], torch.long)), tag="[ext smoke]")
assert np.isfinite(_h[["train_loss", "test_loss"]].to_numpy()).all()
del _m
if DEVICE.type == "cuda": torch.cuda.empty_cache()
EXT_SMOKE_PASSED = True
log("E2 smoke test passed (Day 16-50 forward, masked loss, backprop, evaluation).")

In [ ]:
assert globals().get("EXT_SMOKE_PASSED"), "Run Cell E2 first."

def daywise_metrics(pred, true):
    rows = []
    for h, dn in enumerate(EXT_DAY_NAMES):
        t, p = true[:, h].ravel(), pred[:, h].ravel()
        mk = t >= 0
        n = int(mk.sum())
        row = dict(Day=dn, Day_number=EXT_DAYS[h], n_windows=int((true[:, h] >= 0).any(1).sum()), n_labels=n)
        if n == 0:
            row.update({k: np.nan for k in ("Accuracy", "Precision", "Recall", "F1_score", "MSE", "RMSE")})
        else:
            acc = accuracy_score(t[mk], p[mk])
            pr, rc, f1, _ = precision_recall_fscore_support(t[mk], p[mk], labels=list(range(N_CLASSES)),
                                                            average="micro", zero_division=0)
            mse = float(np.mean((t[mk] - p[mk].astype(float)) ** 2))
            row.update(Accuracy=acc, Precision=pr, Recall=rc, F1_score=f1, MSE=mse, RMSE=math.sqrt(mse))
        rows.append(row)
    return pd.DataFrame(rows)

def run_ext_trial(ds, trial_name, seed):
    rdir = os.path.join(EXT_TRIAL_DIR, ds, f"{trial_name}_seed{seed}")
    done = os.path.join(rdir, "metrics.json")
    if os.path.exists(done):
        log(f"[ext {ds}/{trial_name}] already complete — loaded (resume)")
        return (pd.read_csv(os.path.join(rdir, "daywise_train.csv")), pd.read_csv(os.path.join(rdir, "daywise_test.csv")))
    if os.path.isdir(rdir):
        shutil.move(rdir, rdir + f"_incomplete_{datetime.datetime.now():%Y%m%d_%H%M%S}")
    os.makedirs(rdir)
    e = EXT[ds]
    Xtr, Ytr = to_dev(e["X_tr"], torch.float32), to_dev(e["Y_tr"], torch.long)
    Xte, Yte = to_dev(e["X_te"], torch.float32), to_dev(e["Y_te"], torch.long)
    t0 = time.time()
    model, hist, _, crit = fit_model("GAT_GRU", EXT_HP, seed, Xtr, Ytr, fixed_epochs=MAX_EPOCHS,
                                     monitor=(Xte, Yte), tag=f"[ext {ds}/{trial_name}]")
    assert len(hist) == MAX_EPOCHS
    hist.to_csv(os.path.join(rdir, f"history_{MAX_EPOCHS}_epochs.csv"), index=False)
    ev_te = evaluate(model, Xte, Yte, crit); ev_tr = evaluate(model, Xtr, Ytr, crit)
    dw_te = daywise_metrics(ev_te["preds"], e["Y_te"]); dw_tr = daywise_metrics(ev_tr["preds"], e["Y_tr"])
    for df_, split in ((dw_tr, "Training"), (dw_te, "Testing")):
        df_.insert(0, "Split", split); df_.insert(0, "Seed", seed); df_.insert(0, "Trial", trial_name); df_.insert(0, "Dataset", ds)
    dw_tr.to_csv(os.path.join(rdir, "daywise_train.csv"), index=False)
    dw_te.to_csv(os.path.join(rdir, "daywise_test.csv"), index=False)
    np.savez_compressed(os.path.join(rdir, "test_predictions_day16_50.npz"), pred_stage=ev_te["preds"],
                        true_stage=e["Y_te"].astype(np.int8), probabilities=ev_te["probs"], day=np.array(EXT_DAYS),
                        silo=np.array([s for s, _ in e["w_te"]]), start_row=np.array([r for _, r in e["w_te"]]))
    torch.save({"state_dict": model.state_dict(), "hp": EXT_HP, "seed": seed, "dataset": ds, "days": EXT_DAY_NAMES},
               os.path.join(rdir, "model.pt"))
    res = dict(Dataset=ds, Trial=trial_name, Seed=seed, Epochs=MAX_EPOCHS, Train_seconds=time.time() - t0,
               Pooled_test=ev_te["metrics"], Pooled_train=ev_tr["metrics"], hp=EXT_HP)
    with open(done, "w") as f:
        json.dump(res, f, indent=2, cls=NpEncoder)
    log(f"[ext {ds}/{trial_name}] pooled Day16-50 TEST acc={ev_te['metrics']['Accuracy']:.4f} "
        f"RMSE={ev_te['metrics']['RMSE']:.4f} ({(time.time() - t0) / 60:.1f} min)")
    del model
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return dw_tr, dw_te

parts = []
for ds in DATASETS:
    for trial_name, seed in SEEDS.items():
        parts += list(run_ext_trial(ds, trial_name, seed))
EXT_DAYWISE = pd.concat(parts, ignore_index=True)
save_df(EXT_DAYWISE, "extended", "E3_daywise_metrics_all_trials.csv")
log(f"E3 done: {len(EXT_DAYWISE)} day-wise rows saved.")

In [ ]:
import os, re, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt

try:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
except ImportError:
    pass

OUTPUT_ROOT = "/content/drive/MyDrive/Colab Notebooks/732 sampleGAT-GRU"
RUN_NAME = None
SPLIT_TO_PLOT = "Testing"

def _latest(path):
    base, ext = os.path.splitext(path)
    c = [p for p in [path] + glob.glob(f"{base}__v*{ext}") if os.path.exists(p)]
    return max(c, key=lambda p: int(re.search(r"__v(\d+)", p).group(1)) if "__v" in p else 1) if c else None

if RUN_NAME is None:
    runs = [d for d in glob.glob(os.path.join(OUTPUT_ROOT, "run_*"))
            if _latest(os.path.join(d, "15_extended_day16_50", "E3_daywise_metrics_all_trials.csv"))]
    assert runs, f"No run with extended Day16-50 results in {OUTPUT_ROOT} (run Cell E3 first)."
    RUN_DIR_E = max(runs, key=os.path.getmtime)
else:
    RUN_DIR_E = os.path.join(OUTPUT_ROOT, RUN_NAME)
EXT_DIR = os.path.join(RUN_DIR_E, "15_extended_day16_50")
FIG_DIR_E = os.path.join(EXT_DIR, "figures"); os.makedirs(FIG_DIR_E, exist_ok=True)
src = _latest(os.path.join(EXT_DIR, "E3_daywise_metrics_all_trials.csv"))
DW = pd.read_csv(src)
print("Reading:", src)

DATASETS_E = ["Silo1", "Silo2", "Silo3", "Combined"]
DS_COLORS_E = {"Silo1": "#1f77b4", "Silo2": "#ff7f0e", "Silo3": "#2ca02c", "Combined": "#d62728"}
METRICS_E = {"Accuracy": "Accuracy (%)", "Precision": "Precision (%)", "Recall": "Recall (%)",
             "F1_score": "F1-score (%)", "MSE": "MSE (stage units²)", "RMSE": "RMSE (stage units)"}
PCT_E = {"Accuracy", "Precision", "Recall", "F1_score"}
DAY_ORDER = sorted(DW.Day.unique(), key=lambda s: int(s[3:]))

def save_fig_e(fig, name):
    for ext in ("png", "pdf"):
        p = os.path.join(FIG_DIR_E, f"{name}.{ext}"); k = 2
        while os.path.exists(p):
            p = os.path.join(FIG_DIR_E, f"{name}__v{k}.{ext}"); k += 1
        fig.savefig(p, dpi=300, bbox_inches="tight")
    print("saved", name)

sub = DW[DW.Split == SPLIT_TO_PLOT].copy()
for m in PCT_E:
    sub[m] = 100 * sub[m]
agg = sub.groupby(["Dataset", "Day"], sort=False).agg(
    **{f"{m}_mean": (m, "mean") for m in METRICS_E}, **{f"{m}_SD": (m, lambda s: s.std(ddof=1)) for m in METRICS_E},
    n_windows=("n_windows", "first"), n_labels=("n_labels", "first")).reset_index()
agg["Day_number"] = agg.Day.str[3:].astype(int)
agg = agg.sort_values(["Dataset", "Day_number"])
out_csv = os.path.join(EXT_DIR, f"E4_daywise_mean_sd_{SPLIT_TO_PLOT.lower()}.csv")
agg.to_csv(out_csv, index=False); print("saved", os.path.basename(out_csv))

for m, ylab in METRICS_E.items():
    fig, axes = plt.subplots(len(DATASETS_E), 1, figsize=(16, 3.2 * len(DATASETS_E)), sharex=True)
    for ax, ds in zip(axes, DATASETS_E):
        g = agg[agg.Dataset == ds].set_index("Day").loc[DAY_ORDER]
        x = np.arange(len(DAY_ORDER))
        ax.bar(x, g[f"{m}_mean"], yerr=g[f"{m}_SD"], capsize=2, color=DS_COLORS_E[ds], edgecolor="black", linewidth=0.4)
        vals = g[f"{m}_mean"].dropna()
        if m in PCT_E and len(vals):
            ax.set_ylim(max(0, vals.min() - 3), min(100.5, vals.max() + 2))
        for xi, (v, n) in enumerate(zip(g[f"{m}_mean"], g["n_windows"])):
            if np.isnan(v):
                ax.text(xi, ax.get_ylim()[0], "n/a", ha="center", va="bottom", fontsize=7, rotation=90)
        ax.set_ylabel(ylab); ax.set_title(ds, loc="left", fontsize=10); ax.grid(axis="y", alpha=0.3)
    axes[-1].set_xticks(np.arange(len(DAY_ORDER))); axes[-1].set_xticklabels(DAY_ORDER, rotation=90)
    axes[-1].set_xlabel("Forecast day")
    nm = METRICS_E[m].split(" (")[0]
    fig.suptitle(f"GAT-GRU day-wise {nm}, {SPLIT_TO_PLOT.lower()} (Day 16–50; bars: mean of 3 trials, error bars: SD)")
    fig.tight_layout(); save_fig_e(fig, f"E4_daywise_{m}_{SPLIT_TO_PLOT.lower()}"); plt.show()

for m, ylab in METRICS_E.items():
    fig, ax = plt.subplots(figsize=(20, 5.5))
    x = np.arange(len(DAY_ORDER)); w = 0.8 / len(DATASETS_E)
    allv = []
    for i, ds in enumerate(DATASETS_E):
        g = agg[agg.Dataset == ds].set_index("Day").loc[DAY_ORDER]
        ax.bar(x - 0.4 + w / 2 + i * w, g[f"{m}_mean"], w, yerr=g[f"{m}_SD"], capsize=1.5,
               color=DS_COLORS_E[ds], edgecolor="black", linewidth=0.3, label=ds)
        allv += list(g[f"{m}_mean"].dropna())
    if m in PCT_E and allv:
        ax.set_ylim(max(0, min(allv) - 3), min(100.5, max(allv) + 2))
    ax.set_xticks(x); ax.set_xticklabels(DAY_ORDER, rotation=90); ax.set_xlabel("Forecast day")
    ax.set_ylabel(ylab); ax.grid(axis="y", alpha=0.3); ax.legend(title="Dataset", ncol=4)
    nm = METRICS_E[m].split(" (")[0]
    ax.set_title(f"GAT-GRU day-wise {nm} comparison, silo-wise and combined ({SPLIT_TO_PLOT.lower()}, mean ± SD of 3 trials)")
    fig.tight_layout(); save_fig_e(fig, f"E4_daywise_{m}_all_datasets_{SPLIT_TO_PLOT.lower()}"); plt.show()

fig, ax = plt.subplots(figsize=(16, 4.5))
x = np.arange(len(DAY_ORDER)); w = 0.8 / len(DATASETS_E)
for i, ds in enumerate(DATASETS_E):
    g = agg[agg.Dataset == ds].set_index("Day").loc[DAY_ORDER]
    ax.bar(x - 0.4 + w / 2 + i * w, g["n_windows"], w, color=DS_COLORS_E[ds], edgecolor="black", linewidth=0.3, label=ds)
ax.set_xticks(x); ax.set_xticklabels(DAY_ORDER, rotation=90); ax.set_xlabel("Forecast day")
ax.set_ylabel("Evaluated windows (samples)"); ax.grid(axis="y", alpha=0.3); ax.legend(title="Dataset", ncol=4)
ax.set_title(f"Number of {SPLIT_TO_PLOT.lower()} windows with a valid label on each forecast day")
fig.tight_layout(); save_fig_e(fig, f"E4_daywise_evaluated_windows_{SPLIT_TO_PLOT.lower()}"); plt.show()

print("Note: for single-label multiclass classification, precision, recall and F1-score (micro-averaged) equal accuracy "
      "on every day; MSE/RMSE use the integer stage labels 0–3.")

In [ ]:
from torch_geometric.nn import GATConv, GCNConv, SAGEConv, GraphConv, GINConv

assert STRICT_EPOCHS and MAX_EPOCHS == 1000, "Comparison uses the same protocol: STRICT_EPOCHS=True, MAX_EPOCHS=1000."
assert "EXT" in globals() and "EXT_H" in globals(), "Run Cell E1 first (Day 16–50 targets)."

COMP_SPEC = OrderedDict([
    ("GraphConv", dict(spatial="graphconv", temporal="mlp", desc="2x GraphConv (mean aggr.) per day -> concat 15 days -> MLP")),
    ("SAGE",      dict(spatial="sage",      temporal="mlp", desc="2x SAGEConv (mean aggr.) per day -> concat 15 days -> MLP")),
    ("GCN",       dict(spatial="gcn",       temporal="mlp", desc="2x GCNConv (sym. norm., self-loops) per day -> concat 15 days -> MLP")),
    ("GIN",       dict(spatial="gin",       temporal="mlp", desc="2x GINConv (2-layer MLP, learnable eps) per day -> concat 15 days -> MLP")),
    ("GAT",       dict(spatial="gat",       temporal="mlp", desc="2x GATConv (multi-head) per day -> concat 15 days -> MLP")),
    ("GRU",       dict(spatial="none",      temporal="gru", desc="per-node Linear (no graph) -> per-node GRU over 15 days")),
    ("GAT_GRU",   dict(spatial="gat",       temporal="gru", desc=ARCH_DESCRIPTION["GAT_GRU"])),

    ("GraphConv_GRU", dict(spatial="graphconv", temporal="gru", desc="2x GraphConv per day -> per-node GRU")),
    ("SAGE_GRU",      dict(spatial="sage",      temporal="gru", desc="2x SAGEConv per day -> per-node GRU")),
    ("GCN_GRU",       dict(spatial="gcn",       temporal="gru", desc="2x GCNConv per day -> per-node GRU")),
    ("GIN_GRU",       dict(spatial="gin",       temporal="gru", desc="2x GINConv per day -> per-node GRU")),
])
COMP_MODELS = ["GraphConv", "SAGE", "GCN", "GIN", "GAT", "GRU", "GAT_GRU"]
COMP_NEW = [m for m in COMP_MODELS if m != "GAT_GRU"]
COMP_TRAIN_ARCHS = list(COMP_NEW)
COMP_SMOKE_EPOCHS = 2
assert all(m in COMP_SPEC for m in COMP_MODELS) and "GAT_GRU" in COMP_MODELS

DIRS["comparison"] = os.path.join(RUN_DIR, "16_comparison")
os.makedirs(DIRS["comparison"], exist_ok=True)
COMP_EXT_TRIAL_DIR = os.path.join(DIRS["extended"], "comparison_trials")
os.makedirs(COMP_EXT_TRIAL_DIR, exist_ok=True)

def latest_version(path):
    base, ext = os.path.splitext(path)
    found = [(1, path)] if os.path.exists(path) else []
    folder = os.path.dirname(path)
    for f in (os.listdir(folder) if os.path.isdir(folder) else []):
        mt = re.fullmatch(re.escape(os.path.basename(base)) + r"__v(\d+)" + re.escape(ext), f)
        if mt:
            found.append((int(mt.group(1)), os.path.join(folder, f)))
    return max(found)[1] if found else None

class CompStageModel(nn.Module):
    def __init__(self, arch, edge_index, hidden=64, heads=4, gru_layers=1, dropout=0.2,
                 node_emb_dim=NODE_EMB_DIM, num_spatial_layers=NUM_SPATIAL_LAYERS,
                 num_nodes=N_NODES, n_targets=N_SENSORS, in_feats=1,
                 horizons=HORIZONS, n_classes=N_CLASSES, input_days=INPUT_DAYS):
        super().__init__()
        spec = COMP_SPEC[arch]
        self.arch, self.spatial_kind, self.temporal_kind = arch, spec["spatial"], spec["temporal"]
        if self.spatial_kind == "gat":
            assert hidden % heads == 0, f"hidden ({hidden}) must be divisible by heads ({heads})"
        self.hidden, self.horizons, self.n_classes = hidden, horizons, n_classes
        self.num_nodes, self.n_targets, self.input_days = num_nodes, n_targets, input_days
        self.register_buffer("edge_index", edge_index.long(), persistent=False)
        self._ei_cache = {}
        self.node_emb = nn.Embedding(num_nodes, node_emb_dim) if node_emb_dim > 0 else None
        d_in = in_feats + node_emb_dim
        self.spatial, self.norms = nn.ModuleList(), nn.ModuleList()
        for l in range(num_spatial_layers):
            di = d_in if l == 0 else hidden
            self.spatial.append(self._make_layer(di, hidden, heads, dropout))
            self.norms.append(nn.LayerNorm(hidden))
        self.drop = nn.Dropout(dropout)
        if self.temporal_kind == "gru":
            self.gru = nn.GRU(hidden, hidden, num_layers=gru_layers, batch_first=True,
                              dropout=dropout if gru_layers > 1 else 0.0)
        elif self.temporal_kind == "mlp":
            self.temporal_mlp = nn.Sequential(nn.Linear(input_days * hidden, hidden), nn.ReLU(), nn.Dropout(dropout))
        else:
            raise ValueError(f"unknown temporal module {self.temporal_kind}")
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
                                  nn.Linear(hidden, horizons * n_classes))

    def _make_layer(self, di, hidden, heads, dropout):
        k = self.spatial_kind
        if k == "graphconv": return GraphConv(di, hidden, aggr="mean")
        if k == "sage":      return SAGEConv(di, hidden, aggr="mean")
        if k == "gcn":       return GCNConv(di, hidden, add_self_loops=True, normalize=True)
        if k == "gin":       return GINConv(nn.Sequential(nn.Linear(di, hidden), nn.ReLU(), nn.Linear(hidden, hidden)),
                                            train_eps=True)
        if k == "gat":       return GATConv(di, hidden // heads, heads=heads, concat=True, dropout=dropout, add_self_loops=True)
        if k == "none":      return nn.Linear(di, hidden)
        raise ValueError(f"unknown spatial layer {k}")

    def batched_edge_index(self, num_graphs):
        key = (num_graphs, self.edge_index.device)
        if key not in self._ei_cache:
            E = self.edge_index.shape[1]
            off = torch.arange(num_graphs, device=self.edge_index.device).repeat_interleave(E) * self.num_nodes
            self._ei_cache[key] = self.edge_index.repeat(1, num_graphs) + off
        return self._ei_cache[key]

    def forward(self, x, trace=None):
        B, T, N, Fin = x.shape
        if self.temporal_kind == "mlp":
            assert T == self.input_days, f"MLP temporal module expects {self.input_days} days, got {T}"
        if trace is not None: trace["01 input (B,T,N,F)"] = tuple(x.shape)
        h = x
        if self.node_emb is not None:
            h = torch.cat([h, self.node_emb.weight[None, None].expand(B, T, N, -1)], dim=-1)
        h = h.reshape(B * T * N, -1)
        ei = None if self.spatial_kind == "none" else self.batched_edge_index(B * T)
        for l, (layer, norm) in enumerate(zip(self.spatial, self.norms)):
            out = layer(h) if ei is None else layer(h, ei)
            z = self.drop(norm(F.elu(out)))
            h = z + h if z.shape == h.shape else z
            if trace is not None: trace[f"05.{l+1} spatial layer {l+1} [{self.spatial_kind}] (B*T*N,H)"] = tuple(h.shape)
        h = h.reshape(B, T, N, self.hidden)
        seq = h.permute(0, 2, 1, 3).reshape(B * N, T, self.hidden)
        if self.temporal_kind == "gru":
            _, h_n = self.gru(seq)
            z = h_n[-1]
        else:
            z = self.temporal_mlp(seq.reshape(B * N, T * self.hidden))
        if trace is not None: trace[f"10 temporal [{self.temporal_kind}] (B*N,H)"] = tuple(z.shape)
        logits = self.head(z).reshape(B, N, self.horizons, self.n_classes)
        logits = logits[:, :self.n_targets].permute(0, 2, 1, 3).contiguous()
        if trace is not None: trace["12 logits (B,horizon,sensor,class)"] = tuple(logits.shape)
        return logits

def build_model(arch, hp):
    if arch == "GAT_GRU":
        return GATGRUStageModel(EDGE_INDEX_T, hidden=hp["hidden"], heads=hp.get("heads", 1),
                                gru_layers=hp.get("gru_layers", 1), dropout=hp["dropout"],
                                horizons=hp.get("horizons", HORIZONS))
    if arch not in COMP_SPEC:
        raise ValueError(f"unknown architecture {arch}")
    return CompStageModel(arch, EDGE_INDEX_T, hidden=hp["hidden"], heads=hp.get("heads", 1),
                          gru_layers=hp.get("gru_layers", 1), dropout=hp["dropout"],
                          horizons=hp.get("horizons", HORIZONS))

def suggest_hp(trial, arch):
    hp = {k: trial.suggest_categorical(k, SEARCH_SPACE[k]) for k in ("lr", "batch_size", "hidden", "dropout", "weight_decay")}
    hp["heads"] = trial.suggest_categorical("heads", SEARCH_SPACE["heads"]) if COMP_SPEC[arch]["spatial"] == "gat" else 1
    hp["gru_layers"] = trial.suggest_categorical("gru_layers", SEARCH_SPACE["gru_layers"]) if COMP_SPEC[arch]["temporal"] == "gru" else 1
    return hp

_gg_hp = []
for ds in DATASETS:
    for trial_name, seed in SEEDS.items():
        p = os.path.join(DIRS["trials"], ds, "GAT_GRU", f"{trial_name}_seed{seed}", "metrics.json")
        if not os.path.exists(p):
            raise FileNotFoundError(f"Saved GAT-GRU result missing: {p}\nSet RUN_NAME in Cell 2 to the run that holds "
                                    "the GAT-GRU results (Cell 15). The comparison never retrains GAT-GRU.")
        with open(p) as f:
            _gg_hp.append(json.loads(json.load(f)["hp"]))
assert all(h == _gg_hp[0] for h in _gg_hp), "GAT-GRU runs used different hyperparameters."
for ds in DATASETS:
    for trial_name, seed in SEEDS.items():
        p = os.path.join(EXT_TRIAL_DIR, ds, f"{trial_name}_seed{seed}", "metrics.json")
        if not os.path.exists(p):
            raise FileNotFoundError(f"Saved GAT-GRU Day 16–50 result missing: {p}. Run Cell E3 first.")
BEST_HP = {"GAT_GRU": _gg_hp[0]}
_hp_file = latest_version(os.path.join(DIRS["comparison"], "comparison_best_hyperparameters.json"))
if _hp_file:
    with open(_hp_file) as f:
        BEST_HP.update({k: v for k, v in json.load(f).items() if k != "GAT_GRU"})
log(f"Comparison models: {COMP_MODELS}; GAT-GRU hyperparameters (from saved runs): {BEST_HP['GAT_GRU']}")
log(f"Tuned baselines already available: {[k for k in BEST_HP if k != 'GAT_GRU']}")
save_json({"models": COMP_MODELS, "spec": {m: COMP_SPEC[m] for m in COMP_MODELS},
           "n_test": {**{s: N_TEST_PER_SILO for s in SILOS}, "Combined": 3 * N_TEST_PER_SILO},
           "n_train": {**{s: N_TRAIN_PER_SILO for s in SILOS}, "Combined": 3 * N_TRAIN_PER_SILO},
           "gap_windows": GAP_WINDOWS, "protocol":
           f"same {3 * SAMPLES_PER_SILO} windows ({N_TRAIN_PER_SILO}/{N_TEST_PER_SILO} per silo, "
           f"{3 * N_TRAIN_PER_SILO}/{3 * N_TEST_PER_SILO} combined, {GAP_WINDOWS}-window train/test gap), fresh threshold labels, 268-node graph, training-only "
           "normalisation, Optuna TPE 20 trials on Combined inner-train/validation, seeds 42/123/2026, exactly 1000 epochs, "
           "Day 16–18 and Day 16–50 outputs; GAT-GRU results reused from saved runs"},
          "comparison", "comparison_config.json")

In [ ]:
assert "CompStageModel" in globals(), "Run Cell C1 first."
ut_rows = []
for arch in COMP_MODELS:
    for H in (HORIZONS, EXT_H):
        set_seed(0)
        hp = dict(SMOKE_HP, horizons=H)
        x = torch.randn(2, INPUT_DAYS, N_NODES, 1, device=DEVICE)
        y = torch.randint(0, N_CLASSES, (2, H, N_SENSORS), device=DEVICE)
        y[0, 0, :5] = MISSING_LABEL
        model = build_model(arch, hp).to(DEVICE)
        trace = OrderedDict()
        logits = model(x, trace=trace)
        assert logits.shape == (2, H, N_SENSORS, N_CLASSES), (arch, logits.shape)
        probs = logits.softmax(-1)
        assert torch.allclose(probs.sum(-1), torch.ones_like(probs.sum(-1)), atol=1e-5)
        loss = nn.CrossEntropyLoss(ignore_index=MISSING_LABEL)(logits.reshape(-1, N_CLASSES), y.reshape(-1))
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        before = [p.detach().clone() for p in model.parameters()]
        loss.backward()
        no_grad = [n for n, p in model.named_parameters() if p.requires_grad and (p.grad is None or not torch.isfinite(p.grad).all())]
        assert not no_grad, f"{arch}: parameters without finite gradient: {no_grad}"
        opt.step()
        changed = sum(not torch.equal(b, p) for b, p in zip(before, model.parameters()))
        assert changed > 0
        model.eval()
        with torch.no_grad():
            o1 = model(x); x2 = x.clone(); x2[:, :, 0, 0] += 5.0; o2 = model(x2)
        propagates = not torch.allclose(o1[:, :, 1:], o2[:, :, 1:], atol=1e-6)
        expected = COMP_SPEC[arch]["spatial"] != "none"
        assert propagates == expected, f"{arch}: message passing={propagates}, expected {expected}"
        ut_rows.append(dict(Model=arch, Horizons=H, Output=str(tuple(logits.shape)), Loss=round(float(loss), 4),
                            Params_SMOKE_HP=count_params(model), Graph_message_passing=propagates,
                            Spatial=COMP_SPEC[arch]["spatial"], Temporal=COMP_SPEC[arch]["temporal"]))
        if H == HORIZONS:
            print(f"--- {arch}: {COMP_SPEC[arch]['desc']}")
            for k, v in trace.items():
                print(f"   {k:<55s} {v}")
        del model, opt
UT = pd.DataFrame(ut_rows)
show_table(UT, "Comparison models — forward / backward / message-passing unit tests (synthetic tensors)")
save_df(UT, "comparison", "C2_unit_tests.csv")

dT = get_arrays("Combined", "tune")
Xtr, Ytr = to_dev(dT["X_tr"], torch.float32), to_dev(dT["Y_tr"], torch.long)
Xva, Yva = to_dev(dT["X_ev"], torch.float32), to_dev(dT["Y_ev"], torch.long)
eC = EXT["Combined"]
smoke = []
for arch in COMP_NEW:
    _m, _h, _, _ = fit_model(arch, SMOKE_HP, 42, Xtr, Ytr, Xva, Yva, fixed_epochs=COMP_SMOKE_EPOCHS, tag=f"[comp smoke {arch} d16-18]")
    assert np.isfinite(_h[["train_loss", "val_loss"]].to_numpy()).all()
    del _m
    _m, _h2, _, _ = fit_model(arch, dict(SMOKE_HP, horizons=EXT_H), 42, to_dev(eC["X_tr"], torch.float32),
                              to_dev(eC["Y_tr"], torch.long), fixed_epochs=COMP_SMOKE_EPOCHS,
                              monitor=(to_dev(eC["X_te"], torch.float32), to_dev(eC["Y_te"], torch.long)),
                              tag=f"[comp smoke {arch} d16-50]")
    assert np.isfinite(_h2[["train_loss", "test_loss"]].to_numpy()).all()
    smoke.append(dict(Model=arch, d16_18_last_train_loss=_h.train_loss.iloc[-1], d16_18_last_val_loss=_h.val_loss.iloc[-1],
                      d16_50_last_train_loss=_h2.train_loss.iloc[-1], sec_per_epoch_d16_18=_h.epoch_time_s.mean()))
    del _m
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
del Xtr, Ytr, Xva, Yva
show_table(pd.DataFrame(smoke), "Comparison smoke test (implementation check only)")
save_df(pd.DataFrame(smoke), "comparison", "C2_smoke_test.csv")
COMP_SMOKE_PASSED = True
log("C2 passed: all comparison models build, train and evaluate for Day 16–18 and Day 16–50.")

In [ ]:
assert globals().get("COMP_SMOKE_PASSED"), "Run Cell C2 first."
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def comp_tune_architecture(arch):
    d = get_arrays(TUNE_DATASET, "tune")
    Xtr, Ytr = to_dev(d["X_tr"], torch.float32), to_dev(d["Y_tr"], torch.long)
    Xva, Yva = to_dev(d["X_ev"], torch.float32), to_dev(d["Y_ev"], torch.long)
    drive_db = os.path.join(DIRS["tuning"], f"optuna_{arch}.db")
    local_db = f"/tmp/optuna_{RUN_NAME}_{arch}.db"
    if os.path.exists(drive_db) and not os.path.exists(local_db):
        shutil.copy(drive_db, local_db)
    study = optuna.create_study(study_name=arch, storage=f"sqlite:///{local_db}", load_if_exists=True,
                                direction="minimize", sampler=optuna.samplers.TPESampler(seed=TUNING_SEED),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10))
    def objective(trial):
        hp = suggest_hp(trial, arch)
        t0 = time.time()
        try:
            model, hist, best, _ = fit_model(arch, hp, TUNING_SEED, Xtr, Ytr, Xva, Yva, max_epochs=TUNE_MAX_EPOCHS,
                                             patience=TUNE_PATIENCE, optuna_trial=trial, tag=f"[tune {arch} #{trial.number}]")
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); log(f"[tune {arch} #{trial.number}] CUDA OOM -> pruned", "WARN")
            raise optuna.TrialPruned()
        trial.set_user_attr("best_epoch", best["epoch"]); trial.set_user_attr("best_val_accuracy", best["acc"])
        trial.set_user_attr("params", count_params(model)); trial.set_user_attr("seconds", time.time() - t0)
        hist.to_csv(versioned_path(os.path.join(DIRS["tuning"], f"{arch}_trial{trial.number:03d}_history.csv")), index=False)
        del model
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
        return best["loss"]
    def backup(study, trial):
        shutil.copy(local_db, drive_db)
    remaining = N_TUNING_TRIALS - len([t for t in study.trials if t.state.is_finished()])
    if remaining > 0:
        study.optimize(objective, n_trials=remaining, callbacks=[backup])
    shutil.copy(local_db, drive_db)
    tdf = study.trials_dataframe()
    best_hp = suggest_hp(optuna.trial.FixedTrial(study.best_params), arch)
    return study, tdf, best_hp, (remaining > 0)

COMP_TUNE_ARCHS = list(COMP_TRAIN_ARCHS)
for arch in COMP_TUNE_ARCHS:
    log(f"Tuning {arch}: {N_TUNING_TRIALS} trials, <= {TUNE_MAX_EPOCHS} epochs, dataset={TUNE_DATASET}")
    study, tdf, best_hp, ran = comp_tune_architecture(arch)
    if ran or latest_version(os.path.join(DIRS["tuning"], f"{arch}_tuning_trials.csv")) is None:
        save_df(tdf, "tuning", f"{arch}_tuning_trials.csv")
    BEST_HP[arch] = best_hp
    log(f"[{arch}] best validation loss {study.best_value:.5f} with {best_hp}")
save_json({k: v for k, v in BEST_HP.items() if k != "GAT_GRU"}, "comparison", "comparison_best_hyperparameters.json")
save_json({"GAT_GRU": BEST_HP["GAT_GRU"], "note": "read from the saved GAT-GRU trial results; not re-tuned"},
          "comparison", "gatgru_hyperparameters_used.json")
show_table(pd.DataFrame([dict(Model=k, **v) for k, v in BEST_HP.items() if k in COMP_MODELS]),
           "Selected hyperparameters (baselines: tuned here; GAT-GRU: from its saved runs)")

In [ ]:
assert "BEST_HP" in globals() and all(a in BEST_HP for a in COMP_TRAIN_ARCHS), \
    f"Missing tuned hyperparameters for {[a for a in COMP_TRAIN_ARCHS if a not in BEST_HP]} - run Cell C3."

def load_ckpt(p):
    try:
        return torch.load(p, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(p, map_location=DEVICE)

rep_rows = []
for ds in DATASETS:
    dF = get_arrays(ds, "final")
    Xte, Yte = to_dev(dF["X_ev"], torch.float32), to_dev(dF["Y_ev"], torch.long)
    for trial_name, seed in SEEDS.items():
        rdir = os.path.join(DIRS["trials"], ds, "GAT_GRU", f"{trial_name}_seed{seed}")
        with open(os.path.join(rdir, "metrics.json")) as f:
            saved = json.load(f)
        ck = load_ckpt(os.path.join(rdir, "model.pt"))
        mdl = build_model("GAT_GRU", ck["hp"]).to(DEVICE); mdl.load_state_dict(ck["state_dict"])
        ev = evaluate(mdl, Xte, Yte, nn.CrossEntropyLoss(ignore_index=MISSING_LABEL))
        sp = np.load(os.path.join(rdir, "test_predictions.npz"))
        assert np.array_equal(sp["true_stage"].astype(np.int64), dF["Y_ev"]), f"{ds}: test targets changed!"
        rep_rows.append(dict(Setting="Day16-18", Dataset=ds, Trial=trial_name, Saved_accuracy=saved["Accuracy"],
                             Recomputed_accuracy=ev["metrics"]["Accuracy"],
                             Abs_diff=abs(saved["Accuracy"] - ev["metrics"]["Accuracy"]),
                             Prediction_agreement=float((sp["pred_stage"] == ev["preds"]).mean())))
        del mdl
for ds in DATASETS:
    e = EXT[ds]
    Xte, Yte = to_dev(e["X_te"], torch.float32), to_dev(e["Y_te"], torch.long)
    for trial_name, seed in SEEDS.items():
        rdir = os.path.join(EXT_TRIAL_DIR, ds, f"{trial_name}_seed{seed}")
        with open(os.path.join(rdir, "metrics.json")) as f:
            saved = json.load(f)
        ck = load_ckpt(os.path.join(rdir, "model.pt"))
        if {k: v for k, v in ck["hp"].items() if k != "horizons"} != BEST_HP["GAT_GRU"]:
            log(f"[{ds}/{trial_name}] Day16-50 GAT-GRU hyperparameters differ from Day16-18: {ck['hp']}", "WARN")
        mdl = build_model("GAT_GRU", ck["hp"]).to(DEVICE); mdl.load_state_dict(ck["state_dict"])
        ev = evaluate(mdl, Xte, Yte, nn.CrossEntropyLoss(ignore_index=MISSING_LABEL))
        sp = np.load(os.path.join(rdir, "test_predictions_day16_50.npz"))
        assert np.array_equal(sp["true_stage"].astype(np.int64), e["Y_te"]), f"{ds}: Day16-50 test targets changed!"
        rep_rows.append(dict(Setting="Day16-50", Dataset=ds, Trial=trial_name, Saved_accuracy=saved["Pooled_test"]["Accuracy"],
                             Recomputed_accuracy=ev["metrics"]["Accuracy"],
                             Abs_diff=abs(saved["Pooled_test"]["Accuracy"] - ev["metrics"]["Accuracy"]),
                             Prediction_agreement=float((sp["pred_stage"] == ev["preds"]).mean())))
        del mdl
if DEVICE.type == "cuda": torch.cuda.empty_cache()
REPRO = pd.DataFrame(rep_rows)
show_table(REPRO.round(6), "GAT-GRU reproducibility: saved vs recomputed test accuracy (saved values are the ones reported)")
save_df(REPRO, "comparison", "C4_gatgru_reproducibility_check.csv")
if REPRO.Abs_diff.max() > 1e-4:
    log(f"GAT-GRU recomputed accuracy differs from saved by up to {REPRO.Abs_diff.max():.2e} (GPU float "
        "non-determinism or a changed environment). The SAVED GAT-GRU results are used unchanged.", "WARN")
else:
    log("GAT-GRU reproducibility check PASSED: saved results reproduced from checkpoints (|diff| <= 1e-4).")

def comp_run_trial(ds, arch, trial_name, seed):
    assert arch != "GAT_GRU", "GAT-GRU is never retrained here."
    rdir = os.path.join(DIRS["trials"], ds, arch, f"{trial_name}_seed{seed}")
    done = os.path.join(rdir, "metrics.json")
    if os.path.exists(done):
        log(f"[{ds}/{arch}/{trial_name}] already complete — loaded (resume)")
        with open(done) as f:
            return json.load(f)
    if os.path.isdir(rdir):
        shutil.move(rdir, rdir + f"_incomplete_{datetime.datetime.now():%Y%m%d_%H%M%S}")
    os.makedirs(rdir)
    hp = BEST_HP[arch]
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t_start = time.time()
    dF = get_arrays(ds, "final")
    Xte, Yte = to_dev(dF["X_ev"], torch.float32), to_dev(dF["Y_ev"], torch.long)
    model, hist, _, crit = fit_model(arch, hp, seed, to_dev(dF["X_tr"], torch.float32), to_dev(dF["Y_tr"], torch.long),
                                     fixed_epochs=MAX_EPOCHS, monitor=(Xte, Yte), tag=f"[{ds}/{arch}/{trial_name}]")
    hist.to_csv(os.path.join(rdir, f"history_{MAX_EPOCHS}_epochs.csv"), index=False)
    assert len(hist) == MAX_EPOCHS
    train_seconds = time.time() - t_start
    t_inf = time.time()
    ev = evaluate(model, Xte, Yte, crit)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    inf_seconds = time.time() - t_inf
    ev_tr = evaluate(model, to_dev(dF["X_tr"], torch.float32), to_dev(dF["Y_tr"], torch.long), crit)
    def horizon_acc(pred, true):
        return {f"Day{INPUT_DAYS + 1 + h}": float((pred[:, h][true[:, h] >= 0] == true[:, h][true[:, h] >= 0]).mean())
                for h in range(HORIZONS)}
    hz_te, hz_tr = horizon_acc(ev["preds"], dF["Y_ev"]), horizon_acc(ev_tr["preds"], dF["Y_tr"])
    torch.save({"state_dict": model.state_dict(), "arch": arch, "hp": hp, "seed": seed, "dataset": ds,
                "norm_stats": dF["stats"], "epochs_final": MAX_EPOCHS, "edge_index": EDGE_INDEX_T},
               os.path.join(rdir, "model.pt"))
    np.savez_compressed(os.path.join(rdir, "test_predictions.npz"), pred_stage=ev["preds"], true_stage=dF["Y_ev"].astype(np.int8),
                        probabilities=ev["probs"], silo=np.array([s for s, _ in dF["w_ev"]]),
                        start_row=np.array([r for _, r in dF["w_ev"]]))
    res = OrderedDict(Dataset=ds, Architecture=arch, Trial=trial_name, Seed=seed, **ev["metrics"],
                      Best_val_epoch=np.nan, Best_val_loss=np.nan, Epochs_final=MAX_EPOCHS,
                      Train_seconds=train_seconds, Inference_seconds=inf_seconds,
                      Peak_GPU_MB=(torch.cuda.max_memory_allocated() / 2**20) if DEVICE.type == "cuda" else np.nan,
                      Params=count_params(model), Device=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu",
                      Final_fit_mode=f"strict_{MAX_EPOCHS}_epochs", hp=json.dumps(hp))
    res.update({f"Train_{k}": ev_tr["metrics"][k] for k in METRIC_COLS})
    res.update({f"Test_Acc_{k}": v for k, v in hz_te.items()})
    res.update({f"Train_Acc_{k}": v for k, v in hz_tr.items()})
    with open(os.path.join(rdir, "config.json"), "w") as f:
        json.dump(dict(hp=hp, seed=seed, arch=arch, dataset=ds, gap_policy=GAP_POLICY, loss_weighting=LOSS_WEIGHTING), f, indent=2)
    with open(done, "w") as f:
        json.dump(res, f, indent=2, cls=NpEncoder)
    log(f"[{ds}/{arch}/{trial_name}] TEST acc={res['Accuracy']:.4f} RMSE={res['RMSE']:.4f} ({train_seconds/60:.1f} min)")
    del model
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return res

COMP_RESULTS_3 = []
for arch in COMP_TRAIN_ARCHS:
    for ds in DATASETS:
        for trial_name, seed in SEEDS.items():
            COMP_RESULTS_3.append(comp_run_trial(ds, arch, trial_name, seed))
save_df(pd.DataFrame(COMP_RESULTS_3), "comparison", "C4_baseline_trial_metrics_day16_18.csv")

pc = []
for arch in COMP_MODELS:
    if arch not in BEST_HP:
        continue
    pc.append(dict(Model=arch, Spatial=COMP_SPEC[arch]["spatial"], Temporal=COMP_SPEC[arch]["temporal"],
                   Description=COMP_SPEC[arch]["desc"],
                   Params_day16_18=count_params(build_model(arch, BEST_HP[arch])),
                   Params_day16_50=count_params(build_model(arch, dict(BEST_HP[arch], horizons=EXT_H))),
                   **{f"hp_{k}": v for k, v in BEST_HP[arch].items()}))
save_df(pd.DataFrame(pc), "comparison", "C4_param_counts.csv")
log("C4 done (Day 16–18 baselines).")

In [ ]:
assert all(a in BEST_HP for a in COMP_TRAIN_ARCHS), "Run Cell C3 first."

def comp_daywise_metrics(pred, true):
    rows = []
    for h, dn in enumerate(EXT_DAY_NAMES):
        t, p = true[:, h].ravel(), pred[:, h].ravel()
        mk = t >= 0
        n = int(mk.sum())
        row = dict(Day=dn, Day_number=EXT_DAYS[h], n_windows=int((true[:, h] >= 0).any(1).sum()), n_labels=n)
        if n == 0:
            row.update({k: np.nan for k in ("Accuracy", "Precision", "Recall", "F1_score", "MSE", "RMSE")})
        else:
            acc = accuracy_score(t[mk], p[mk])
            pr, rc, f1, _ = precision_recall_fscore_support(t[mk], p[mk], labels=list(range(N_CLASSES)),
                                                            average="micro", zero_division=0)
            mse = float(np.mean((t[mk] - p[mk].astype(float)) ** 2))
            row.update(Accuracy=acc, Precision=pr, Recall=rc, F1_score=f1, MSE=mse, RMSE=math.sqrt(mse))
        rows.append(row)
    return pd.DataFrame(rows)

def comp_run_ext_trial(ds, arch, trial_name, seed):
    assert arch != "GAT_GRU", "GAT-GRU Day 16–50 results are reused from Cell E3."
    rdir = os.path.join(COMP_EXT_TRIAL_DIR, arch, ds, f"{trial_name}_seed{seed}")
    done = os.path.join(rdir, "metrics.json")
    if os.path.exists(done):
        log(f"[ext {arch}/{ds}/{trial_name}] already complete — loaded (resume)")
        return pd.read_csv(os.path.join(rdir, "daywise_train.csv")), pd.read_csv(os.path.join(rdir, "daywise_test.csv"))
    if os.path.isdir(rdir):
        shutil.move(rdir, rdir + f"_incomplete_{datetime.datetime.now():%Y%m%d_%H%M%S}")
    os.makedirs(rdir)
    hp = dict(BEST_HP[arch], horizons=EXT_H)
    e = EXT[ds]
    Xtr, Ytr = to_dev(e["X_tr"], torch.float32), to_dev(e["Y_tr"], torch.long)
    Xte, Yte = to_dev(e["X_te"], torch.float32), to_dev(e["Y_te"], torch.long)
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t0 = time.time()
    model, hist, _, crit = fit_model(arch, hp, seed, Xtr, Ytr, fixed_epochs=MAX_EPOCHS,
                                     monitor=(Xte, Yte), tag=f"[ext {arch}/{ds}/{trial_name}]")
    assert len(hist) == MAX_EPOCHS
    train_seconds = time.time() - t0
    hist.to_csv(os.path.join(rdir, f"history_{MAX_EPOCHS}_epochs.csv"), index=False)
    t_inf = time.time()
    ev_te = evaluate(model, Xte, Yte, crit)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    inf_seconds = time.time() - t_inf
    ev_tr = evaluate(model, Xtr, Ytr, crit)
    dw_te = comp_daywise_metrics(ev_te["preds"], e["Y_te"]); dw_tr = comp_daywise_metrics(ev_tr["preds"], e["Y_tr"])
    for df_, split in ((dw_tr, "Training"), (dw_te, "Testing")):
        df_.insert(0, "Split", split); df_.insert(0, "Seed", seed); df_.insert(0, "Trial", trial_name); df_.insert(0, "Dataset", ds)
        df_.insert(0, "Model", arch)
    dw_tr.to_csv(os.path.join(rdir, "daywise_train.csv"), index=False)
    dw_te.to_csv(os.path.join(rdir, "daywise_test.csv"), index=False)
    np.savez_compressed(os.path.join(rdir, "test_predictions_day16_50.npz"), pred_stage=ev_te["preds"],
                        true_stage=e["Y_te"].astype(np.int8), probabilities=ev_te["probs"], day=np.array(EXT_DAYS),
                        silo=np.array([s for s, _ in e["w_te"]]), start_row=np.array([r for _, r in e["w_te"]]))
    torch.save({"state_dict": model.state_dict(), "arch": arch, "hp": hp, "seed": seed, "dataset": ds, "days": EXT_DAY_NAMES},
               os.path.join(rdir, "model.pt"))
    res = dict(Dataset=ds, Model=arch, Trial=trial_name, Seed=seed, Epochs=MAX_EPOCHS, Train_seconds=train_seconds,
               Inference_seconds=inf_seconds, Params=count_params(model),
               Peak_GPU_MB=(torch.cuda.max_memory_allocated() / 2**20) if DEVICE.type == "cuda" else np.nan,
               Pooled_test=ev_te["metrics"], Pooled_train=ev_tr["metrics"], hp=hp)
    with open(done, "w") as f:
        json.dump(res, f, indent=2, cls=NpEncoder)
    log(f"[ext {arch}/{ds}/{trial_name}] pooled Day16-50 TEST acc={ev_te['metrics']['Accuracy']:.4f} "
        f"RMSE={ev_te['metrics']['RMSE']:.4f} ({train_seconds / 60:.1f} min)")
    del model
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return dw_tr, dw_te

parts = []
for arch in COMP_TRAIN_ARCHS:
    for ds in DATASETS:
        for trial_name, seed in SEEDS.items():
            parts += list(comp_run_ext_trial(ds, arch, trial_name, seed))
if parts:
    save_df(pd.concat(parts, ignore_index=True), "comparison", "C5_baseline_daywise_metrics_day16_50.csv")
log("C5 done (Day 16–50 baselines). Now run Cell C6 (it can also be run on its own later).")

In [ ]:
import os, re, glob, json, math, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
try:
    from IPython.display import display
except ImportError:
    display = print
try:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
except ImportError:
    pass

OUTPUT_ROOT = "/content/drive/MyDrive/Colab Notebooks/732 sampleGAT-GRU"
RUN_NAME = None
DATASETS = ["Silo1", "Silo2", "Silo3", "Combined"]
SEEDS = {"Trial-1": 42, "Trial-2": 123, "Trial-3": 2026}
MAX_EPOCHS, INPUT_DAYS, HORIZONS, N_CLASSES = 1000, 15, 3, 4
STAGES = ["Stage 0 Safe", "Stage 1 Initial", "Stage 2 Active", "Stage 3 Severe"]
METRICS = ["Accuracy", "Micro_Precision", "Micro_Recall", "Micro_F1", "MSE", "RMSE"]
PCT = {"Accuracy", "Micro_Precision", "Micro_Recall", "Micro_F1", "Macro_F1"}
REF = "GAT_GRU"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

def _latest(path):
    base, ext = os.path.splitext(path)
    c = [p for p in [path] + glob.glob(glob.escape(base) + "__v*" + ext) if os.path.exists(p)]
    return max(c, key=lambda p: int(re.search(r"__v(\d+)", p).group(1)) if "__v" in p else 1) if c else None

if RUN_NAME is None:
    runs = [d for d in glob.glob(os.path.join(OUTPUT_ROOT, "run_*"))
            if _latest(os.path.join(d, "16_comparison", "comparison_config.json"))]
    assert runs, f"No run with comparison results in {OUTPUT_ROOT} (run Cells C1–C5 first)."
    RUN_DIR_C = max(runs, key=os.path.getmtime)
else:
    RUN_DIR_C = os.path.join(OUTPUT_ROOT, RUN_NAME)
CMP_DIR = os.path.join(RUN_DIR_C, "16_comparison")
TAB_DIR, FIG_DIR = os.path.join(CMP_DIR, "tables"), os.path.join(CMP_DIR, "figures")
os.makedirs(TAB_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
EXT_DIR = os.path.join(RUN_DIR_C, "15_extended_day16_50")
with open(_latest(os.path.join(CMP_DIR, "comparison_config.json"))) as f:
    CFG = json.load(f)
MODELS_ALL = CFG["models"]
print("Run:", RUN_DIR_C, "\nModels:", MODELS_ALL)

_pal = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd", "#8c564b", "#7f7f7f", "#e377c2", "#17becf", "#bcbd22", "#aec7e8"]
COL = {m: ("#d62728" if m == REF else _pal[i % len(_pal)]) for i, m in enumerate([m for m in MODELS_ALL if m != REF])}
COL[REF] = "#d62728"
LBL = {m: m.replace("_", "-") for m in MODELS_ALL}
LBL.update({"SAGE": "GraphSAGE", "GraphConv": "GraphConv"})

def _unique(path):
    base, ext = os.path.splitext(path); p, k = path, 2
    while os.path.exists(p):
        p = f"{base}__v{k}{ext}"; k += 1
    return p

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(_unique(os.path.join(FIG_DIR, f"{name}.{ext}")), dpi=300, bbox_inches="tight")
    print("saved figure", name)

ALL_TABLES = {}
def save_tab(df, name, title=None, index=False):
    df.to_csv(_unique(os.path.join(TAB_DIR, f"{name}.csv")), index=index)
    ALL_TABLES[name] = df.reset_index() if index else df
    if title:
        print("\n" + "=" * 110 + f"\n{title}\n" + "=" * 110); display(df)

def dir3(m, ds, t, s):
    return os.path.join(RUN_DIR_C, "08_trials", ds, m, f"{t}_seed{s}")
def dir35(m, ds, t, s):
    return (os.path.join(EXT_DIR, "trials", ds, f"{t}_seed{s}") if m == REF
            else os.path.join(EXT_DIR, "comparison_trials", m, ds, f"{t}_seed{s}"))

def extra_metrics(pred, true):
    t, p = true.ravel().astype(int), pred.ravel().astype(int)
    mk = t >= 0; t, p = t[mk], p[mk]
    present = sorted(np.unique(t).tolist())
    _, _, f1m, _ = precision_recall_fscore_support(t, p, labels=present, average="macro", zero_division=0)
    _, rc, _, sup = precision_recall_fscore_support(t, p, labels=list(range(N_CLASSES)), average=None, zero_division=0)
    out = {"Macro_F1": float(f1m)}
    for k in range(N_CLASSES):
        out[f"Recall_stage{k}"] = float(rc[k]) if sup[k] > 0 else np.nan
    return out

def pooled(pred, true):
    mk = true >= 0
    if not mk.any():
        return np.nan, np.nan
    acc = float((pred[mk] == true[mk]).mean())
    return acc, float(np.sqrt(np.mean((true[mk].astype(float) - pred[mk].astype(float)) ** 2)))

R3, R35, DWL, missing = [], [], [], []
for m in MODELS_ALL:
    for ds in DATASETS:
        for t, s in SEEDS.items():
            d3 = dir3(m, ds, t, s)
            if os.path.exists(os.path.join(d3, "metrics.json")):
                with open(os.path.join(d3, "metrics.json")) as f:
                    r = json.load(f)
                row = dict(Model=m, Dataset=ds, Trial=t, Seed=s, **{k: r[k] for k in METRICS},
                           **{f"Train_{k}": r.get(f"Train_{k}", np.nan) for k in METRICS},
                           Params=r.get("Params", np.nan), Train_seconds=r.get("Train_seconds", np.nan),
                           Inference_seconds=r.get("Inference_seconds", np.nan), Peak_GPU_MB=r.get("Peak_GPU_MB", np.nan))
                npz = os.path.join(d3, "test_predictions.npz")
                if os.path.exists(npz):
                    z = np.load(npz); pr, tr = z["pred_stage"].astype(int), z["true_stage"].astype(int)
                    for h in range(HORIZONS):
                        row[f"Acc_Day{INPUT_DAYS + 1 + h}"] = pooled(pr[:, h], tr[:, h])[0]
                    row.update(extra_metrics(pr, tr))
                R3.append(row)
            else:
                missing.append(("Day16-18", m, ds, t))
            d35 = dir35(m, ds, t, s)
            if os.path.exists(os.path.join(d35, "metrics.json")):
                with open(os.path.join(d35, "metrics.json")) as f:
                    r = json.load(f)
                row = dict(Model=m, Dataset=ds, Trial=t, Seed=s, **{k: r["Pooled_test"][k] for k in METRICS},
                           **{f"Train_{k}": r["Pooled_train"][k] for k in METRICS},
                           Params=r.get("Params", np.nan), Train_seconds=r.get("Train_seconds", np.nan),
                           Inference_seconds=r.get("Inference_seconds", np.nan), Peak_GPU_MB=r.get("Peak_GPU_MB", np.nan))
                npz = os.path.join(d35, "test_predictions_day16_50.npz")
                if os.path.exists(npz):
                    z = np.load(npz); pr, tr = z["pred_stage"].astype(int), z["true_stage"].astype(int)
                    row["Acc_Day16_18"], row["RMSE_Day16_18"] = pooled(pr[:, :HORIZONS], tr[:, :HORIZONS])
                    row["Acc_Day19_50"], row["RMSE_Day19_50"] = pooled(pr[:, HORIZONS:], tr[:, HORIZONS:])
                    row.update(extra_metrics(pr, tr))
                R35.append(row)
                for split, fn in (("Testing", "daywise_test.csv"), ("Training", "daywise_train.csv")):
                    dw = pd.read_csv(os.path.join(d35, fn))
                    dw["Model"] = m; dw["Split"] = split
                    DWL.append(dw)
            else:
                missing.append(("Day16-50", m, ds, t))
R3, R35 = pd.DataFrame(R3), pd.DataFrame(R35)
DW = pd.concat(DWL, ignore_index=True) if DWL else pd.DataFrame()
n_need = len(DATASETS) * len(SEEDS)
def complete(df):
    return [m for m in MODELS_ALL if len(df) and (df.Model == m).sum() == n_need]
M3, M35 = complete(R3), complete(R35)
if missing:
    miss = pd.DataFrame(missing, columns=["Setting", "Model", "Dataset", "Trial"])
    print(f"WARNING: {len(miss)} runs not found; incomplete models are left out of that setting:")
    display(miss.groupby(["Setting", "Model"]).size().rename("missing_runs").reset_index())
assert REF in M3 and REF in M35, "GAT-GRU results not found (Cells 15 and E3)."
R3, R35 = R3[R3.Model.isin(M3)].copy(), R35[R35.Model.isin(M35)].copy()
DW = DW[DW.Model.isin(M35)].copy()
MB = [m for m in MODELS_ALL if m in M3 and m in M35]
print("Day 16–18 models:", M3, "\nDay 16–50 models:", M35)

pc_path = _latest(os.path.join(CMP_DIR, "C4_param_counts.csv"))
PC = pd.read_csv(pc_path) if pc_path else pd.DataFrame(columns=["Model"])
if len(PC):
    p35 = PC.set_index("Model")["Params_day16_50"]
    R35["Params"] = R35["Params"].fillna(R35.Model.map(p35))
for df_ in (R3, R35):
    df_["Model"] = pd.Categorical(df_["Model"], MODELS_ALL, ordered=True)
    df_["Dataset"] = pd.Categorical(df_["Dataset"], DATASETS, ordered=True)
save_tab(R3.sort_values(["Model", "Dataset", "Trial"]), "CT00a_all_trials_day16_18")
save_tab(R35.sort_values(["Model", "Dataset", "Trial"]), "CT00b_all_trials_day16_50")
save_tab(DW, "CT00c_daywise_all_trials_day16_50")

def agg(df, cols, keys=("Dataset", "Model")):
    g = df.groupby(list(keys), observed=True, sort=True)[cols]
    return g.mean().add_suffix("_mean").join(g.std(ddof=1).add_suffix("_SD")).reset_index()

def fmt_table(A, cols):
    out = A[["Dataset", "Model"]].copy()
    for c in cols:
        k = 100 if (c in PCT or c.startswith("Acc_") or c.startswith("Recall_") or c.startswith("Train_") and c[6:] in PCT) else 1
        out[c + (" (%)" if k == 100 else "")] = [f"{k * a:.2f} ± {k * b:.2f}" if k == 100 else f"{a:.4f} ± {b:.4f}"
                                                 for a, b in zip(A[c + "_mean"], A[c + "_SD"])]
    return out

tune_rows = []
for m in MODELS_ALL:
    p = _latest(os.path.join(RUN_DIR_C, "06_tuning", f"{m}_tuning_trials.csv"))
    row = dict(Model=m)
    if p:
        t = pd.read_csv(p); cpl = t[t.state == "COMPLETE"]
        row.update(Trials_completed=len(cpl), Trials_pruned=int((t.state == "PRUNED").sum()))
        if len(cpl):
            b = cpl.loc[cpl.value.idxmin()]
            row.update(Best_val_loss=b.value, Best_epoch=b.get("user_attrs_best_epoch", np.nan),
                       Best_val_accuracy=b.get("user_attrs_best_val_accuracy", np.nan))
    tune_rows.append(row)
CT02 = pd.DataFrame(tune_rows)
CT01 = PC.copy() if len(PC) else pd.DataFrame({"Model": MODELS_ALL})
CT01 = CT01.merge(CT02, on="Model", how="left")
save_tab(CT01, "CT01_architecture_params_tuning", "CT01 — Architectures, parameters, selected hyperparameters and tuning")

M_COLS = METRICS + [f"Acc_Day{INPUT_DAYS + 1 + h}" for h in range(HORIZONS)] + ["Macro_F1"] + [f"Recall_stage{k}" for k in range(N_CLASSES)]
A3 = agg(R3, [c for c in M_COLS if c in R3] + [f"Train_{k}" for k in METRICS] + ["Train_seconds", "Inference_seconds", "Params", "Peak_GPU_MB"])
M35_COLS = METRICS + ["Acc_Day16_18", "RMSE_Day16_18", "Acc_Day19_50", "RMSE_Day19_50", "Macro_F1"] + [f"Recall_stage{k}" for k in range(N_CLASSES)]
A35 = agg(R35, [c for c in M35_COLS if c in R35] + [f"Train_{k}" for k in METRICS] + ["Train_seconds", "Inference_seconds", "Params", "Peak_GPU_MB"])
save_tab(A3, "CT03n_day16_18_numeric_mean_sd"); save_tab(A35, "CT06n_day16_50_numeric_mean_sd")

save_tab(fmt_table(A3, METRICS), "CT03_day16_18_test_mean_sd", "CT03 — Day 16–18 TEST metrics, mean ± SD (ddof=1), 3 trials")
save_tab(fmt_table(A3, [f"Train_{k}" for k in METRICS]), "CT04_day16_18_train_mean_sd", "CT04 — Day 16–18 TRAINING metrics, mean ± SD")
save_tab(fmt_table(A3, [f"Acc_Day{INPUT_DAYS + 1 + h}" for h in range(HORIZONS)]), "CT05_day16_18_per_horizon_test",
         "CT05 — Day 16–18 test accuracy per forecast day")
save_tab(fmt_table(A35, METRICS), "CT06_day16_50_pooled_test_mean_sd", "CT06 — Day 16–50 pooled TEST metrics, mean ± SD")
save_tab(fmt_table(A35, [f"Train_{k}" for k in METRICS]), "CT06b_day16_50_pooled_train_mean_sd", "CT06b — Day 16–50 pooled TRAINING metrics")

DWT = DW[DW.Split == "Testing"].copy()
DWT["Day_number"] = DWT.Day.str[3:].astype(int)
DAYS = sorted(DWT.Day_number.unique())
DWA = DWT.groupby(["Dataset", "Model", "Day_number"])[["Accuracy", "RMSE", "n_windows", "n_labels"]].agg(["mean", lambda s: s.std(ddof=1)])
DWA.columns = [f"{a}_{'mean' if b == 'mean' else 'SD'}" for a, b in DWA.columns]
DWA = DWA.reset_index()
save_tab(DWA, "CT07n_daywise_test_mean_sd_long")
for met, k in (("Accuracy", 100), ("RMSE", 1)):
    w = DWA.pivot_table(index=["Dataset", "Model"], columns="Day_number", values=f"{met}_mean", observed=True) * k
    w.columns = [f"Day{c}" for c in w.columns]
    w = w.reindex(pd.MultiIndex.from_product([DATASETS, M35], names=["Dataset", "Model"])).round(3)
    save_tab(w, f"CT07_{'a' if met == 'Accuracy' else 'b'}_daywise_{met}_test", f"CT07 — day-wise test {met}"
             + (" (%)" if k == 100 else "") + " (mean of 3 trials)", index=True)
nw = DWA[DWA.Model == REF].pivot_table(index="Dataset", columns="Day_number", values="n_windows_mean", observed=True)
nw.columns = [f"Day{c}" for c in nw.columns]
save_tab(nw.reindex(DATASETS), "CT07c_evaluated_test_windows_per_day", "Evaluated test windows per forecast day", index=True)

c8 = []
for ds in DATASETS:
    for m in MB:
        a = R3[(R3.Dataset == ds) & (R3.Model == m)]; b = R35[(R35.Dataset == ds) & (R35.Model == m)]
        c8.append(dict(Dataset=ds, Model=m,
                       Acc_3day_model_Day16_18=100 * a.Accuracy.mean(), SD1=100 * a.Accuracy.std(ddof=1),
                       Acc_35day_model_Day16_18=100 * b.Acc_Day16_18.mean(), SD2=100 * b.Acc_Day16_18.std(ddof=1),
                       Acc_35day_model_Day19_50=100 * b.Acc_Day19_50.mean(), SD3=100 * b.Acc_Day19_50.std(ddof=1),
                       Acc_35day_model_Day16_50=100 * b.Accuracy.mean(), SD4=100 * b.Accuracy.std(ddof=1),
                       RMSE_3day_model_Day16_18=a.RMSE.mean(), RMSE_35day_model_Day16_18=b.RMSE_Day16_18.mean(),
                       RMSE_35day_model_Day16_50=b.RMSE.mean()))
CT08 = pd.DataFrame(c8)
CT08["Delta_acc_Day16_18_(35day-3day)"] = CT08.Acc_35day_model_Day16_18 - CT08.Acc_3day_model_Day16_18
CT08 = CT08.rename(columns={"SD1": "SD_3day_Day16_18", "SD2": "SD_35day_Day16_18", "SD3": "SD_35day_Day19_50", "SD4": "SD_35day_Day16_50"})
save_tab(CT08.round(4), "CT08_3day_vs_35day_models", "CT08 — 3-day (Day 16–18) model vs 35-day (Day 16–50) model, accuracy %")

def ranks(A, setting, models):
    rows = []
    for ds in DATASETS:
        g = A[(A.Dataset == ds) & (A.Model.isin(models))].copy()
        g["Rank_accuracy"] = g.Accuracy_mean.rank(ascending=False, method="min")
        g["Rank_RMSE"] = g.RMSE_mean.rank(ascending=True, method="min")
        for _, r in g.iterrows():
            rows.append(dict(Setting=setting, Dataset=ds, Model=r.Model, Rank_accuracy=r.Rank_accuracy, Rank_RMSE=r.Rank_RMSE))
    return rows
RK = pd.DataFrame(ranks(A3, "Day16-18", M3) + ranks(A35, "Day16-50", M35))
RK["Model"] = RK.Model.astype(str)
CT09 = RK.pivot_table(index=["Setting", "Dataset"], columns="Model", values="Rank_accuracy")
CT09 = CT09[[m for m in MODELS_ALL if m in CT09.columns]]
avg = RK.groupby(["Setting", "Model"]).Rank_accuracy.mean().unstack()[CT09.columns]
avg.index = pd.MultiIndex.from_tuples([(s, "Mean rank") for s in avg.index], names=CT09.index.names)
CT09 = pd.concat([CT09, avg])
CT09 = CT09.reindex(pd.MultiIndex.from_tuples([(s_, d_) for s_ in ("Day16-18", "Day16-50") for d_ in DATASETS + ["Mean rank"]],
                                              names=CT09.index.names))
save_tab(CT09, "CT09_rank_by_accuracy", "CT09 — Rank by mean test accuracy (1 = best)", index=True)
save_tab(RK, "CT09b_ranks_long")

try:
    from scipy.stats import ttest_rel
except ImportError:
    ttest_rel = None
c10 = []
for setting, R, models in (("Day16-18", R3, M3), ("Day16-50", R35, M35)):
    for ds in DATASETS:
        ref = R[(R.Model == REF) & (R.Dataset == ds)].set_index("Trial").loc[list(SEEDS)]
        for m in models:
            if m == REF:
                continue
            g = R[(R.Model == m) & (R.Dataset == ds)].set_index("Trial").loc[list(SEEDS)]
            d = (ref.Accuracy - g.Accuracy).to_numpy(float)
            p = np.nan
            if ttest_rel is not None and np.std(d) > 0:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore"); p = float(ttest_rel(ref.Accuracy, g.Accuracy).pvalue)
            c10.append(dict(Setting=setting, Dataset=ds, Model=m,
                            Model_acc_mean=100 * g.Accuracy.mean(), GAT_GRU_acc_mean=100 * ref.Accuracy.mean(),
                            Delta_acc_pp_GATGRU_minus_model=100 * d.mean(),
                            Delta_RMSE_model_minus_GATGRU=g.RMSE.mean() - ref.RMSE.mean(),
                            GAT_GRU_better_in_trials=f"{int((d > 0).sum())}/{len(d)}", Paired_t_p_value=p))
CT10 = pd.DataFrame(c10)
save_tab(CT10.round(5), "CT10_difference_vs_GAT_GRU",
         "CT10 — GAT-GRU minus each model (positive = GAT-GRU better); paired t-test over 3 seeds (low power, n=3)")

if "n_test" in CFG:
    n_te = CFG["n_test"]
else:
    _sel = sorted(glob.glob(os.path.join(RUN_DIR_C, "03_samples", "selected_samples_*.csv")), key=os.path.getmtime)
    assert _sel, "Sample counts not found: rerun Cell C1 or Cell 8 for this run."
    _s = pd.read_csv(_sel[-1]); _t = _s[_s.split == "test"]
    n_te = {**_t.groupby("silo").size().astype(int).to_dict(), "Combined": int(len(_t))}
c11 = []
for setting, A in (("Day16-18", A3), ("Day16-50", A35)):
    for _, r in A.iterrows():
        c11.append(dict(Setting=setting, Dataset=r.Dataset, Model=r.Model, Params=r.Params_mean,
                        Train_minutes_mean=r.Train_seconds_mean / 60, Train_minutes_SD=r.Train_seconds_SD / 60,
                        Inference_ms_per_sample=1000 * r.Inference_seconds_mean / n_te[r.Dataset], Peak_GPU_MB=r.Peak_GPU_MB_mean))
CT11 = pd.DataFrame(c11)
save_tab(CT11.round(3), "CT11_computational_cost", "CT11 — Parameters, training time (1000 epochs) and inference time")

CT12 = pd.concat([fmt_table(A3, ["Macro_F1"] + [f"Recall_stage{k}" for k in range(N_CLASSES)]).assign(Setting="Day16-18"),
                  fmt_table(A35, ["Macro_F1"] + [f"Recall_stage{k}" for k in range(N_CLASSES)]).assign(Setting="Day16-50")])
save_tab(CT12, "CT12_supplementary_macroF1_stage_recall",
         "CT12 — SUPPLEMENTARY (from saved predictions): macro-F1 and per-stage recall on test (nan = stage absent)")

HIST = {}
for setting, models, dfun in (("Day16-18", M3, dir3), ("Day16-50", M35, dir35)):
    for m in models:
        for ds in DATASETS:
            for t, s in SEEDS.items():
                p = os.path.join(dfun(m, ds, t, s), f"history_{MAX_EPOCHS}_epochs.csv")
                if os.path.exists(p):
                    HIST[(setting, m, ds, t)] = pd.read_csv(p)
CT13 = pd.DataFrame([dict(Setting=k[0], Model=k[1], Dataset=k[2], Trial=k[3], Epochs=len(h),
                          Final_train_loss=h.train_loss.iloc[-1], Final_test_loss=h.test_loss.iloc[-1],
                          Min_test_loss=h.test_loss.min(), Epoch_of_min_test_loss=int(h.epoch[h.test_loss.idxmin()]))
                     for k, h in HIST.items()])
if len(CT13):
    CT13s = CT13.groupby(["Setting", "Dataset", "Model"], sort=False)[["Final_train_loss", "Final_test_loss"]].mean().reset_index()
    save_tab(CT13s.round(5), "CT13_final_epoch_loss", "CT13 — Loss after 1000 epochs (mean of 3 trials)")

def zoom(ax, vals, pct):
    v = np.asarray([x for x in vals if np.isfinite(x)])
    if pct and len(v):
        ax.set_ylim(max(0, v.min() - 1.0), min(101, v.max() + 1.0))

def grouped(ax, A, metric, models, title):
    pct = metric in PCT; k = 100 if pct else 1
    x = np.arange(len(DATASETS)); w = 0.84 / len(models); vals = []
    for i, m in enumerate(models):
        g = A[A.Model == m].set_index("Dataset").reindex(DATASETS)
        mu, sd = k * g[f"{metric}_mean"].to_numpy(float), k * g[f"{metric}_SD"].to_numpy(float)
        ax.bar(x - 0.42 + w / 2 + i * w, mu, w, yerr=sd, capsize=2, color=COL[m], edgecolor="black",
               linewidth=1.2 if m == REF else 0.3, label=LBL[m])
        vals += list(mu - sd) + list(mu + sd)
    zoom(ax, vals, pct)
    ax.set_xticks(x); ax.set_xticklabels(DATASETS); ax.set_ylabel(metric.replace("_", " ") + (" (%)" if pct else " (stage units)"))
    ax.set_title(title, fontsize=9); ax.grid(axis="y", alpha=0.3)

def legend_models(fig, models, y=-0.01):
    from matplotlib.patches import Patch
    fig.legend(handles=[Patch(facecolor=COL[m], edgecolor="black", label=LBL[m]) for m in models],
               loc="lower center", ncol=len(models), bbox_to_anchor=(0.5, y), fontsize=9)

for A, models, tag, nm in ((A3, M3, "Day 16–18 (3-day forecast)", "C_Fig01_day16_18_test_metrics"),
                           (A35, M35, "Day 16–50 (35-day forecast, pooled)", "C_Fig02_day16_50_test_metrics")):
    fig, axes = plt.subplots(2, 2, figsize=(16, 9))
    for ax, met in zip(axes.ravel(), ("Accuracy", "Micro_F1", "MSE", "RMSE")):
        grouped(ax, A, met, models, f"{met.replace('_', ' ')} — test")
    fig.suptitle(f"Model comparison, {tag}: mean ± SD of 3 trials (accuracy axes zoomed)")
    fig.tight_layout(rect=(0, 0.05, 1, 1)); legend_models(fig, models); save_fig(fig, nm); plt.show()

fig, axes = plt.subplots(2, len(DATASETS), figsize=(20, 8.5), squeeze=False)
for i, (A, models, tag) in enumerate(((A3, M3, "Day 16–18"), (A35, M35, "Day 16–50"))):
    for j, ds in enumerate(DATASETS):
        ax = axes[i, j]; g = A[A.Dataset == ds].set_index("Model").reindex(models); x = np.arange(len(models)); vals = []
        for o, (col, lab, hatch) in enumerate((("Train_Accuracy", "Training", "//"), ("Accuracy", "Testing", ""))):
            mu, sd = 100 * g[f"{col}_mean"].to_numpy(float), 100 * g[f"{col}_SD"].to_numpy(float)
            ax.bar(x + (o - 0.5) * 0.4, mu, 0.4, yerr=sd, capsize=2, color=[COL[m] for m in models],
                   hatch=hatch, edgecolor="black", linewidth=0.3, alpha=0.6 if o == 0 else 1.0, label=lab)
            vals += list(mu)
        zoom(ax, vals, True); ax.set_xticks(x); ax.set_xticklabels([LBL[m] for m in models], rotation=45, ha="right")
        ax.set_title(f"{ds} — {tag}", fontsize=9); ax.grid(axis="y", alpha=0.3)
        if j == 0: ax.set_ylabel("Accuracy (%)")
axes[0, 0].legend(fontsize=8)
fig.suptitle("Training (hatched) vs testing accuracy, all models"); fig.tight_layout(); save_fig(fig, "C_Fig03_train_vs_test_accuracy"); plt.show()

fig, axes = plt.subplots(1, len(DATASETS), figsize=(20, 4.5), sharey=False)
HZN = [f"Acc_Day{INPUT_DAYS + 1 + h}" for h in range(HORIZONS)]
for ax, ds in zip(axes, DATASETS):
    vals = []
    for m in M3:
        g = A3[(A3.Dataset == ds) & (A3.Model == m)]
        mu = [100 * g[f"{h}_mean"].iloc[0] for h in HZN]; sd = [100 * g[f"{h}_SD"].iloc[0] for h in HZN]
        ax.errorbar([h[4:] for h in HZN], mu, yerr=sd, marker="o", capsize=3, color=COL[m], lw=2.2 if m == REF else 1.2, label=LBL[m])
        vals += mu
    zoom(ax, vals, True); ax.set_title(ds); ax.set_xlabel("Forecast day"); ax.grid(alpha=0.3)
axes[0].set_ylabel("Test accuracy (%)"); axes[-1].legend(fontsize=7)
fig.suptitle("Day 16–18 models: test accuracy per forecast day (mean ± SD)"); fig.tight_layout()
save_fig(fig, "C_Fig04_day16_18_per_horizon_accuracy"); plt.show()

for met, nm in (("Accuracy", "C_Fig05_daywise_accuracy_day16_50"), ("RMSE", "C_Fig06_daywise_RMSE_day16_50")):
    k = 100 if met == "Accuracy" else 1
    fig, axes = plt.subplots(2, 2, figsize=(18, 9), sharex=True)
    for ax, ds in zip(axes.ravel(), DATASETS):
        vals = []
        for m in M35:
            g = DWA[(DWA.Dataset == ds) & (DWA.Model == m)].sort_values("Day_number")
            mu, sd = k * g[f"{met}_mean"].to_numpy(float), k * g[f"{met}_SD"].to_numpy(float)
            ax.plot(g.Day_number, mu, color=COL[m], lw=2.2 if m == REF else 1.1, marker="o", ms=2.5, label=LBL[m])
            ax.fill_between(g.Day_number, mu - sd, mu + sd, color=COL[m], alpha=0.12)
            vals += list(mu)
        if met == "Accuracy": zoom(ax, vals, True)
        ax.set_title(ds); ax.grid(alpha=0.3); ax.set_ylabel(f"Test {met}" + (" (%)" if k == 100 else " (stage units)"))
    for ax in axes[-1]: ax.set_xlabel("Forecast day")
    axes[0, 0].legend(fontsize=8, ncol=2)
    fig.suptitle(f"Day-wise test {met}, Day 16–50 (lines: mean of 3 trials; band: ± SD)")
    fig.tight_layout(); save_fig(fig, nm); plt.show()

fig, axes = plt.subplots(len(DATASETS), 1, figsize=(18, 2.2 + 0.55 * len(M35) * len(DATASETS)), squeeze=False)
for ax, ds in zip(axes[:, 0], DATASETS):
    w = DWA[DWA.Dataset == ds].pivot_table(index="Model", columns="Day_number", values="Accuracy_mean", observed=True).reindex(M35) * 100
    im = ax.imshow(w.to_numpy(float), aspect="auto", cmap="viridis")
    ax.set_yticks(range(len(M35))); ax.set_yticklabels([LBL[m] for m in M35])
    ax.set_xticks(range(len(w.columns))); ax.set_xticklabels([f"D{c}" for c in w.columns], fontsize=7, rotation=90)
    ax.set_title(f"{ds}: day-wise test accuracy (%)", fontsize=9); fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01)
fig.tight_layout(); save_fig(fig, "C_Fig07_daywise_accuracy_heatmap"); plt.show()

fig, axes = plt.subplots(1, len(DATASETS), figsize=(22, 5), sharey=False)
SER = [("Acc_3day_model_Day16_18", "SD_3day_Day16_18", "3-day model, Day16–18", ""),
       ("Acc_35day_model_Day16_18", "SD_35day_Day16_18", "35-day model, Day16–18", "//"),
       ("Acc_35day_model_Day16_50", "SD_35day_Day16_50", "35-day model, Day16–50", "..")]
for ax, ds in zip(axes, DATASETS):
    g = CT08[CT08.Dataset == ds].set_index("Model").reindex(MB); x = np.arange(len(MB)); vals = []
    for o, (c, sdc, lab, hatch) in enumerate(SER):
        ax.bar(x + (o - 1) * 0.27, g[c], 0.27, yerr=g[sdc], capsize=2, color=[COL[m] for m in MB],
               hatch=hatch, edgecolor="black", linewidth=0.3, alpha=[1.0, 0.75, 0.5][o], label=lab)
        vals += list(g[c])
    zoom(ax, vals, True); ax.set_xticks(x); ax.set_xticklabels([LBL[m] for m in MB], rotation=45, ha="right")
    ax.set_title(ds); ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("Test accuracy (%)"); axes[0].legend(fontsize=7)
fig.suptitle("3-day vs 35-day models: accuracy on Day 16–18 and over all 35 days (mean ± SD)"); fig.tight_layout()
save_fig(fig, "C_Fig08_3day_vs_35day"); plt.show()

fig, ax = plt.subplots(figsize=(1.3 * CT09.shape[1] + 3, 0.45 * CT09.shape[0] + 1.5))
im = ax.imshow(CT09.to_numpy(float), cmap="RdYlGn_r", aspect="auto")
for (i, j), v in np.ndenumerate(CT09.to_numpy(float)):
    if np.isfinite(v):
        ax.text(j, i, f"{v:.1f}" if CT09.index[i][1] == "Mean rank" else f"{v:.0f}", ha="center", va="center", fontsize=8,
                color="white" if (v <= 1.6 or v >= len(CT09.columns) - 0.6) else "black")
ax.set_xticks(range(CT09.shape[1])); ax.set_xticklabels([LBL[c] for c in CT09.columns], rotation=30, ha="right")
ax.set_yticks(range(CT09.shape[0])); ax.set_yticklabels([f"{a} | {b}" for a, b in CT09.index])
ax.set_title("Rank by mean test accuracy (1 = best)"); fig.colorbar(im, ax=ax, fraction=0.03)
fig.tight_layout(); save_fig(fig, "C_Fig09_rank_heatmap"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(18, 5), sharey=True)
for ax, setting in zip(axes, ("Day16-18", "Day16-50")):
    g = CT10[CT10.Setting == setting]; others = [m for m in MODELS_ALL if m in set(g.Model)]
    x = np.arange(len(DATASETS)); w = 0.84 / max(len(others), 1)
    for i, m in enumerate(others):
        v = g[g.Model == m].set_index("Dataset").reindex(DATASETS)["Delta_acc_pp_GATGRU_minus_model"]
        ax.bar(x - 0.42 + w / 2 + i * w, v, w, color=COL[m], edgecolor="black", linewidth=0.3, label=LBL[m])
    ax.axhline(0, color="black", lw=0.8); ax.set_xticks(x); ax.set_xticklabels(DATASETS)
    ax.set_title(f"{setting}: GAT-GRU accuracy minus model accuracy"); ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("Δ accuracy (percentage points; > 0 = GAT-GRU better)"); axes[1].legend(fontsize=8)
fig.tight_layout(); save_fig(fig, "C_Fig10_delta_vs_GAT_GRU"); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
cc = CT11[CT11.Dataset == "Combined"]
for ax, col, ylab in zip(axes, ("Params", "Train_minutes_mean", "Inference_ms_per_sample"),
                         ("Trainable parameters", "Training time, 1000 epochs (min)", "Inference (ms / sample)")):
    x = np.arange(len(MB))
    for o, setting in enumerate(("Day16-18", "Day16-50")):
        g = cc[cc.Setting == setting].set_index("Model").reindex(MB)
        ax.bar(x + (o - 0.5) * 0.4, g[col], 0.4, color=[COL[m] for m in MB], hatch=["", "//"][o],
               edgecolor="black", linewidth=0.3, alpha=[1.0, 0.6][o], label=setting)
    ax.set_xticks(x); ax.set_xticklabels([LBL[m] for m in MB], rotation=45, ha="right"); ax.set_ylabel(ylab); ax.grid(axis="y", alpha=0.3)
axes[0].legend(fontsize=8)
fig.suptitle("Computational cost, Combined dataset (hatched = Day 16–50 model)"); fig.tight_layout()
save_fig(fig, "C_Fig11_computational_cost"); plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
for m in MODELS_ALL:
    p = _latest(os.path.join(RUN_DIR_C, "06_tuning", f"{m}_tuning_trials.csv"))
    if p:
        t = pd.read_csv(p); t = t[t.state == "COMPLETE"].sort_values("number")
        if len(t):
            ax.plot(t.number, t.value, "o", color=COL[m], alpha=0.3, ms=4)
            ax.plot(t.number, t.value.cummin(), "-", color=COL[m], lw=2.2 if m == REF else 1.3, label=LBL[m])
ax.set_xlabel("Optuna trial"); ax.set_ylabel("Validation cross-entropy loss"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
ax.set_title("Hyperparameter tuning (same budget for every model): points = trials, lines = running minimum")
fig.tight_layout(); save_fig(fig, "C_Fig12_tuning_progress"); plt.show()

if HIST:
    fig, axes = plt.subplots(2, 2, figsize=(16, 9), sharex=True)
    for i, (setting, models) in enumerate((("Day16-18", M3), ("Day16-50", M35))):
        for j, col in enumerate(("train_loss", "test_loss")):
            ax = axes[i, j]
            for m in models:
                hs = [HIST[(setting, m, "Combined", t)][col].to_numpy() for t in SEEDS if (setting, m, "Combined", t) in HIST]
                if not hs: continue
                L = np.stack(hs); ep = np.arange(1, L.shape[1] + 1)
                ax.plot(ep, L.mean(0), color=COL[m], lw=2.0 if m == REF else 1.0, label=LBL[m])
            ax.set_title(f"Combined, {setting}: {'training' if j == 0 else 'testing (monitor only)'} loss")
            ax.set_yscale("log"); ax.grid(alpha=0.3)
            if j == 0: ax.set_ylabel("Cross-entropy (mean of 3 trials, log)")
    for ax in axes[-1]: ax.set_xlabel("Epoch")
    axes[0, 0].legend(fontsize=8, ncol=2); fig.tight_layout(); save_fig(fig, "C_Fig13_loss_curves_combined"); plt.show()

if "Macro_F1_mean" in A3 and "Macro_F1_mean" in A35:
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    grouped(axes[0], A3, "Macro_F1", M3, "Day 16–18 test macro-F1 (supplementary)")
    grouped(axes[1], A35, "Macro_F1", M35, "Day 16–50 test macro-F1 (supplementary)")
    fig.tight_layout(rect=(0, 0.07, 1, 1)); legend_models(fig, MB); save_fig(fig, "C_Fig14_macroF1_supplementary"); plt.show()

fig, axes = plt.subplots(2, 1, figsize=(18, 9))
for ax, (A, models, setting) in zip(axes, ((A3, M3, "Day16-18"), (A35, M35, "Day16-50"))):
    g = A[A.Dataset == "Combined"].set_index("Model").reindex(models); x = np.arange(N_CLASSES); w = 0.84 / len(models)
    for i, m in enumerate(models):
        ax.bar(x - 0.42 + w / 2 + i * w, [100 * g.loc[m, f"Recall_stage{k}_mean"] for k in range(N_CLASSES)], w,
               color=COL[m], edgecolor="black", linewidth=0.3, label=LBL[m])
    ax.set_xticks(x); ax.set_xticklabels(STAGES); ax.set_ylabel("Test recall (%)"); ax.grid(axis="y", alpha=0.3)
    ax.set_title(f"Combined, {setting}: per-stage test recall (supplementary; missing bar = stage absent in test)")
axes[0].legend(fontsize=8, ncol=len(MB)); fig.tight_layout(); save_fig(fig, "C_Fig15_stage_recall_supplementary"); plt.show()

try:
    xl = _unique(os.path.join(TAB_DIR, "Comparison_all_tables.xlsx"))
    with pd.ExcelWriter(xl) as wr:
        for name, df in ALL_TABLES.items():
            if not name.startswith("CT00"):
                df.to_excel(wr, sheet_name=name[:31], index=False)
    print("saved", os.path.basename(xl))
except Exception as e:
    print("Excel workbook not written:", e)
print("Notes: micro precision = micro recall = micro F1 = accuracy for single-label multiclass outputs; "
      "MSE/RMSE use integer stage labels 0–3; macro-F1 and per-stage recall are supplementary and computed from "
      f"the saved predictions. All tables: {TAB_DIR}\nAll figures: {FIG_DIR}")

In [ ]:
import sklearn, torch_geometric
env = OrderedDict(python=sys.version, platform=platform.platform(), torch=torch.__version__,
                  cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(), torch_geometric=torch_geometric.__version__,
                  optuna=optuna.__version__, numpy=np.__version__, pandas=pd.__version__, sklearn=sklearn.__version__,
                  matplotlib=matplotlib.__version__, device=str(DEVICE),
                  gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
                  determinism_note="cuDNN deterministic mode is on; PyG scatter/aggregation on GPU can still be "
                                   "non-deterministic at floating-point level, so reruns may differ slightly.",
                  finished=str(datetime.datetime.now()))
save_json(env, "environment", "environment.json")
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
with open(versioned_path(os.path.join(DIRS["environment"], "pip_freeze.txt")), "w") as f:
    f.write(freeze)
manifest = [dict(path=os.path.relpath(os.path.join(r, f), RUN_DIR), bytes=os.path.getsize(os.path.join(r, f)))
            for r, _, fs in os.walk(RUN_DIR) for f in fs]
save_df(pd.DataFrame(manifest), "logs", "output_manifest.csv")
log(f"ALL DONE. Outputs in {RUN_DIR}")